# Book Calculations

Every computed figure quoted in *Nobody in Charge*, organised by chapter, with an
assertion against the value printed in the book.

**This notebook is a regression test, not just a record.** If a cell fails, either the
model changed or the book is wrong, and one of them needs fixing. Run it after any
change to `aa_group_model.py` and before calling any chapter done.

Requires `numpy` and `aa_group_model.py` in the same directory.

| Section | Chapter | What it verifies |
|---|---|---|
| 1 | Ch 1, 2, 4 | Inflow channels and the three decline scenarios |
| 2 | Ch 3, 8, 9 | Influence weights and consensus error |
| 3 | Ch 5, 6 | Protective traditions and the tradition comparison |
| 4 | Ch 7 | The five-member worked example |
| 5 | Ch 9 | The asymmetry coefficients |
| 6 | - | Full regression sweep |

In [ ]:
import numpy as np, importlib.util, sys, os
_candidates = ('aa_group_model.py', 'model/aa_group_model.py')
_p = next((p for p in _candidates if os.path.exists(p)), None)
if _p is None: raise FileNotFoundError('run from the repository root or model directory')
spec = importlib.util.spec_from_file_location('m', _p)
m = importlib.util.module_from_spec(spec); sys.modules['m'] = m; spec.loader.exec_module(m)

FAILURES = []
def check(label, got, want, tol=0.005):
    ok = abs(got - want) <= tol
    if not ok: FAILURES.append(f"{label}: got {got:.4f}, book says {want}")
    print(f"  {'OK ' if ok else 'FAIL'} {label:<46} {got:>8.3f}  (book: {want})")
    return ok

print("loaded model:", _p)

loaded model: aa_group_model.py


---
## 1. Inflow channels and decline scenarios
### Chapters 1, 2 and 4

The model has two ways a newcomer arrives: **attraction**, proportional to members'
current twelfth-step practice, and **referral**, an exogenous stream independent of
member activity.

> lambda = lambda_exog + lambda_0 * (sum of x12 across members) * T11

Chapter 1 argues the Washingtonians ran on attraction alone, because in 1840 there was
no treatment system, no courts assigning attendance, and no medical consensus. Chapter 4
argues AA acquired the second channel between the 1939 book and the 1941 *Saturday
Evening Post* article. Chapter 2 uses the same runs for the fade.

In [ ]:
H = 1560   # 30 years in weeks

# --- The decline table, recomputed at 400 seeds with Monte Carlo error -----------------
# This table was originally computed from 10 seeds. The cross-seed SD of final membership
# is about 15 against a mean near 42, so a 10-seed mean carries SE ~5 and a 10-seed
# survival fraction is quantised to tenths. Both are far coarser than three significant
# figures. `scenarios_hiseed.py` reruns all five at 400 seeds; results are read from disk.
import json as _json, os as _os
_R = '../research' if _os.path.isdir('../research') else 'research'
HS = list(_json.load(open(f'{_R}/scenarios_hiseed.json')).values())

def wilson(k, n, z=1.96):
    p = k/n; den = 1 + z*z/n
    c = (p + z*z/(2*n))/den
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/den
    return max(0.0, c-h), min(1.0, c+h)

def hs(scen):
    r = [v for v in HS if v['scen'] == scen]
    N = np.array([v['N'] for v in r], float)          # over ALL runs, deaths as zero
    k = sum(v['alive'] for v in r); n = len(r)
    q = np.array([v['est_mean'] for v in r if v['alive']], float)
    return dict(n=n, surv=k/n, ci=wilson(k, n), N=N.mean(), N_se=N.std(ddof=1)/np.sqrt(n),
                q=q.mean() if len(q) else 0.0,
                q_se=q.std(ddof=1)/np.sqrt(len(q)) if len(q) > 1 else 0.0, n_alive=len(q))

print("Decline scenarios, 30-year horizon, 400 seeds, with Monte Carlo error\n")
print(f"{'scenario':<26}{'survival':>9}{'95% CI':>17}{'mean N':>9}{'+/-':>7}{'quality':>9}")
R = {}
for lab, s in [("Both channels intact", 'full'), ("Attraction lost", 'attraction'),
               ("Referrals lost", 'referral'), ("Both lost", 'both'),
               ("Gatekeeping culture", 'gatekeeping')]:
    v = R[s] = hs(s)
    print(f"{lab:<26}{v['surv']:>9.3f}   [{v['ci'][0]:.3f}, {v['ci'][1]:.3f}]"
          f"{v['N']:>9.1f}{1.96*v['N_se']:>7.1f}{v['q']:>9.3f}")

check("full adherence, survival (400 seeds)",   R['full']['surv'],        0.995, tol=0.003)
check("full adherence, mean N (400 seeds)",     R['full']['N'],           41.7,  tol=0.3)
check("attraction lost, survival",              R['attraction']['surv'],  0.998, tol=0.003)
check("attraction lost, mean N",                R['attraction']['N'],     13.5,  tol=0.3)
check("referrals lost, survival",               R['referral']['surv'],    0.360, tol=0.01)
check("referrals lost, mean N",                 R['referral']['N'],       9.9,   tol=0.3)
check("both channels lost, survival",           R['both']['surv'],        0.000, tol=0.001)
check("gatekeeping, survival",                  R['gatekeeping']['surv'], 0.940, tol=0.01)
check("gatekeeping, mean N",                    R['gatekeeping']['N'],    27.5,  tol=0.3)

print("\nWHAT CHANGED FROM THE 10-SEED TABLE, and it is not rounding:\n")
print(f"{'quantity':<34}{'10 seeds':>10}{'400 seeds':>11}{'in CI?':>8}")
for lab, s, old_s, old_N in [("full adherence survival",  'full',        1.00, 45.0),
                             ("full adherence mean N",    'full',        None, 45.0),
                             ("referral-starved survival",'referral',    0.20, 2.9),
                             ("referral-starved mean N",  'referral',    None, 2.9),
                             ("gatekeeping survival",     'gatekeeping', 0.90, 30.7),
                             ("gatekeeping mean N",       'gatekeeping', None, 30.7)]:
    v = R[s]
    if old_s is not None:
        inci = v['ci'][0] <= old_s <= v['ci'][1]
        print(f"{lab:<34}{old_s:>10.2f}{v['surv']:>11.3f}{'yes' if inci else 'NO':>8}")
    else:
        lo, hi = v['N'] - 1.96*v['N_se'], v['N'] + 1.96*v['N_se']
        print(f"{lab:<34}{old_N:>10.1f}{v['N']:>11.1f}{'yes' if lo <= old_N <= hi else 'NO':>8}")
print("\n  -> The referral-starved case was WRONG, not merely imprecise. The book said it")
print("     survives one run in five at an average size of 2.9. Over 400 seeds it survives")
print("     36 per cent of the time at an average size of 9.9. Both book figures lie far")
print("     outside the 95 per cent interval. Corrected in Chapters 1, 2, 4 and the preface.")
print("     Full-adherence membership was also 8 per cent high, and gatekeeping 12 per cent.")

print("\nQuality among survivors, 400 seeds (mean practice of established members):")
for lab, s in [("full", 'full'), ("attraction lost", 'attraction'),
               ("referrals lost", 'referral'), ("gatekeeping", 'gatekeeping')]:
    v = R[s]
    print(f"  {lab:<16}{v['q']:.3f} +/- {1.96*v['q_se']:.3f}   (n = {v['n_alive']} surviving runs)")
check("quality, full adherence",   R['full']['q'],        0.354, tol=0.006)
check("quality, referrals lost",   R['referral']['q'],    0.372, tol=0.011)
check("quality, gatekeeping",      R['gatekeeping']['q'], 0.400, tol=0.008)
print("\n  -> quality among survivors is at least as high in every declining case as in the")
print("     healthy one, which is a SELECTION effect and not a finding about resilience:")
print("     the runs that survive starvation are the ones that were doing well anyway.")
print("     Chapters 2 and 21 must state it that way.")


Decline scenarios, 30-year horizon, 400 seeds, with Monte Carlo error

scenario                   survival           95% CI   mean N    +/-  quality
Both channels intact          0.995   [0.982, 0.999]     41.7    1.5    0.354
Attraction lost               0.998   [0.986, 1.000]     13.5    0.4    0.312
Referrals lost                0.360   [0.314, 0.408]      9.9    1.6    0.372
Both lost                     0.000   [0.000, 0.010]      0.0    0.0    0.000
Gatekeeping culture           0.940   [0.912, 0.959]     27.5    1.7    0.400
  OK  full adherence, survival (400 seeds)              0.995  (book: 0.995)
  OK  full adherence, mean N (400 seeds)               41.727  (book: 41.7)
  OK  attraction lost, survival                         0.998  (book: 0.998)
  OK  attraction lost, mean N                          13.523  (book: 13.5)
  OK  referrals lost, survival                          0.360  (book: 0.36)
  OK  referrals lost, mean N                            9.900  (book: 9.9)
  OK

In [ ]:
# --- The two membership thresholds, at 400 seeds, and they are NOT the same quantity
# This cell previously read `range(SEEDS)` with SEEDS defined nowhere. It raised NameError,
# produced no output, and no checker caught it: the status check looks for STORED FAILURES
# and a cell with an empty output is not one. Found on 2 August 2026 by executing the
# notebook end to end for the first time, rather than by inspecting it.
#
# The model carries two thresholds on a member's mean practice and the book had been using
# the word "core" for both:
#     act_thr = 0.1   ESTABLISHED. What simulate() returns as n_est. About 37 of 42.
#     exp_thr = 0.5   EXPERIENCED. What appendix A6 and Chapter 4 mean by the calibration
#                     target. About 8.
# Read from research/core_thresholds.json, 400 seeds, model/core_thresholds.py.
_CT = _json.load(open(f'{_R}/core_thresholds.json'))
_ctm, _ctr = _CT['meta'], _CT['result']
print(f"Full adherence, {_ctm['nseed']} seeds, thirty-year horizon\n")
print(f"  survival                              {_ctr['survival']:.3f}")
print(f"  members, all runs                     {_ctr['N'][0]:.1f} +/- {_ctr['N'][1]:.1f}")
print(f"  members, surviving runs               {_ctr['N_alive'][0]:.1f} +/- {_ctr['N_alive'][1]:.1f}")
print(f"  ESTABLISHED, practice > {_ctm['act_thr']}          {_ctr['established'][0]:.1f} +/- {_ctr['established'][1]:.1f}")
print(f"  EXPERIENCED, practice > {_ctm['exp_thr']}          {_ctr['experienced'][0]:.1f} +/- {_ctr['experienced'][1]:.1f}")
check("act_thr, the established threshold", _ctm['act_thr'], 0.1, tol=1e-9)
check("exp_thr, the experienced threshold", _ctm['exp_thr'], 0.5, tol=1e-9)
check("full adherence survival", _ctr['survival'], 0.995, tol=0.003)
check("full adherence mean N, all runs", _ctr['N'][0], 41.7, tol=0.06)
check("established members at full adherence", _ctr['established'][0], 37.2, tol=0.06)
check("experienced core at full adherence", _ctr['experienced'][0], 7.7, tol=0.06)
check("experienced core half-width", _ctr['experienced'][1], 0.5, tol=0.06)
print(f"""
  -> THE CALIBRATION STATEMENT WAS WRONG AND THIS CELL IS WHY IT WAS NOT CAUGHT.
     Appendix A6 and Chapter 4 say the model is tuned so that a fully adherent group holds a
     steady state near 45 members with 'an established core near 9'. At 400 seeds the
     experienced core is {_ctr['experienced'][0]:.1f} with a 95 per cent half-width of {_ctr['experienced'][1]:.1f}, so 9 lies
     outside the interval. The membership half of the same statement was already corrected to
     41.7 against a target of 45. Both halves are now stated as measured.
     The word 'core' has also been separated: the Part Five tables print the ESTABLISHED
     count and the calibration target is the EXPERIENCED count, and they differ by a factor
     of nearly five.""")


Full adherence, 400 seeds, thirty-year horizon

  survival                              0.995
  members, all runs                     41.7 +/- 1.5
  members, surviving runs               41.9 +/- 1.5
  ESTABLISHED, practice > 0.1          37.2 +/- 1.4
  EXPERIENCED, practice > 0.5          7.7 +/- 0.5
  OK  act_thr, the established threshold                0.100  (book: 0.1)
  OK  exp_thr, the experienced threshold                0.500  (book: 0.5)
  OK  full adherence survival                           0.995  (book: 0.995)
  OK  full adherence mean N, all runs                  41.727  (book: 41.7)
  OK  established members at full adherence            37.201  (book: 37.2)
  OK  experienced core at full adherence                7.719  (book: 7.7)
  OK  experienced core half-width                       0.546  (book: 0.5)

  -> THE CALIBRATION STATEMENT WAS WRONG AND THIS CELL IS WHY IT WAS NOT CAUGHT.
     Appendix A6 and Chapter 4 say the model is tuned so that a fully adherent group h

---
## 2. Influence weights and consensus error
### Chapters 3, 8 and 9

A room of N members with row-stochastic trust matrix A. Influence vector s is the
normalised left dominant eigenvector. Consensus error is exact, not simulated:

> expected |consensus - truth| = sigma * (length of s) * sqrt(2/pi)

Under flat weighting the length of s is 1/sqrt(N), so the error is exactly
sigma * sqrt(2/pi) / sqrt(N). One person alone gives 0.798.

Four regimes: **flat**, **dominant** (one member holds 35%), **caucus** (three hold 50%
between them), **closed core** (five receive 45% and attend only to each other).

In [ ]:
C = np.sqrt(2/np.pi)

def influence(A):
    w, v = np.linalg.eig(A.T)
    s = np.abs(np.real(v[:, np.argmax(np.real(w))]))
    return s/s.sum()

def err(A):  return C*np.linalg.norm(influence(A))

def flat(N):      return np.full((N, N), 1.0/N)
def dominant(N, share=0.35):
    A = np.full((N, N), (1-share)/(N-1)); A[:, 0] = share
    return A/A.sum(1, keepdims=True)
def caucus(N, k=3, share=0.50):
    A = np.full((N, N), (1-share)/(N-k)); A[:, :k] = share/k
    return A/A.sum(1, keepdims=True)
def closed_core(N, k=5, share=0.45):
    A = np.full((N, N), (1-share)/(N-k)); A[:, :k] = share/k
    A[:k, :] = 0.0; A[:k, :k] = 1.0/k          # the core attends only to itself
    return A/A.sum(1, keepdims=True)
def rotating(N, pool=12, share=0.35):
    acc = np.zeros((N, N))
    for t in range(pool):
        A = np.full((N, N), (1-share)/(N-1)); A[:, t % N] = share
        acc += A/A.sum(1, keepdims=True)
    return acc/pool

print(f"{'N':>5}{'flat':>9}{'dominant':>10}{'caucus':>9}{'core':>9}   MAX INFLUENCE")
for N in [10, 50, 250, 500]:
    print(f"{N:>5}" + "".join(f"{influence(f(N)).max():>9.3f}" for f in (flat, dominant, caucus, closed_core)))
print(f"\n{'N':>5}{'flat':>9}{'dominant':>10}{'caucus':>9}{'core':>9}   CONSENSUS ERROR")
for N in [10, 50, 250, 500]:
    print(f"{N:>5}" + "".join(f"{err(f(N)):>9.3f}" for f in (flat, dominant, caucus, closed_core)))
print(f"\none person alone: {C:.3f}")

    N     flat  dominant   caucus     core   MAX INFLUENCE
   10    0.100    0.350    0.167    0.200
   50    0.020    0.350    0.167    0.200
  250    0.004    0.350    0.167    0.200
  500    0.002    0.350    0.167    0.200

    N     flat  dominant   caucus     core   CONSENSUS ERROR
   10    0.252    0.328    0.275    0.357
   50    0.113    0.289    0.238    0.357
  250    0.050    0.281    0.232    0.357
  500    0.036    0.280    0.231    0.357

one person alone: 0.798


In [ ]:
print("Ch 3 and Ch 8 tables: flat vs dominant\n")
for N, wi_f, wi_d, e_f, e_d in [(10, 0.100, 0.350, 0.252, 0.328), (500, 0.002, 0.350, 0.036, 0.280)]:
    check(f"N={N} max influence, flat",     influence(flat(N)).max(),     wi_f)
    check(f"N={N} max influence, dominant", influence(dominant(N)).max(), wi_d)
    check(f"N={N} error, flat",             err(flat(N)),                 e_f)
    check(f"N={N} error, dominant",         err(dominant(N)),             e_d)

print("\nCh 9 tables: caucus and closed core")
check("N=500 max influence, caucus",  influence(caucus(500)).max(),      0.167)
check("N=500 error, caucus",          err(caucus(500)),                  0.231)
check("N=500 max influence, core",    influence(closed_core(500)).max(), 0.200)
check("N=500 error, closed core",     err(closed_core(500)),             0.357)

print("\nCh 9 claim: the closed core holds ALL the influence")
s = influence(closed_core(250))
check("closed core, share held by the five", float(s[:5].sum()), 1.000)
check("closed core, share held by the rest", float(s[5:].sum()), 0.000)
print(f"  -> a group of 250 decides exactly as well as a group of 5: {C/np.sqrt(5):.3f}")

print("\nAnalytic limits confirm the tables")
check("dominant limit = sigma*alpha*sqrt(2/pi)", 0.35*C, 0.279)
check("caucus limit",  float(np.sqrt(3*(0.5/3)**2)*C), 0.230)
check("core limit = sqrt(2/pi)/sqrt(5)", C/np.sqrt(5), 0.357)

Ch 3 and Ch 8 tables: flat vs dominant

  OK  N=10 max influence, flat                          0.100  (book: 0.1)
  OK  N=10 max influence, dominant                      0.350  (book: 0.35)
  OK  N=10 error, flat                                  0.252  (book: 0.252)
  OK  N=10 error, dominant                              0.328  (book: 0.328)
  OK  N=500 max influence, flat                         0.002  (book: 0.002)
  OK  N=500 max influence, dominant                     0.350  (book: 0.35)
  OK  N=500 error, flat                                 0.036  (book: 0.036)
  OK  N=500 error, dominant                             0.280  (book: 0.28)

Ch 9 tables: caucus and closed core
  OK  N=500 max influence, caucus                       0.167  (book: 0.167)
  OK  N=500 error, caucus                               0.231  (book: 0.231)
  OK  N=500 max influence, core                         0.200  (book: 0.2)
  OK  N=500 error, closed core                          0.357  (book: 0.357)

Ch 9 

### Chapter 10: rotation must scale

Rotation is modelled by averaging the trust matrix over a cycle: for each of R terms one
member holds an officeholder's share, then average across terms. The claim is that a
**fixed** pool produces a floor at roughly alpha/R while the flat benchmark keeps falling
as 1/N, so the gap widens without limit.

In [ ]:
print("Pool sweep at N=400, officeholder share 0.35\n")
print(f"{'pool R':>8}{'max infl':>11}{'error':>9}{'alpha/R':>10}")
for R in [3, 6, 12, 25, 50, 100, 400]:
    A = rotating(400, pool=R)
    print(f"{R:>8}{influence(A).max():>11.4f}{err(A):>9.3f}{0.35/R:>10.4f}")

check("N=400 pool 12, max influence", influence(rotating(400, 12)).max(), 0.031)
check("N=400 pool 12, error",          err(rotating(400, 12)),            0.089)
check("N=400 flat, max influence",     influence(flat(400)).max(),        0.0025)
check("N=400 flat, error",             err(flat(400)),                    0.040)

print("\nA fixed pool of 12 floors while the flat benchmark keeps falling:")
print(f"{'N':>6}{'pool 12':>10}{'flat':>10}{'ratio':>8}")
for N in [50, 100, 200, 400, 800]:
    mi = influence(rotating(N, 12)).max()
    print(f"{N:>6}{mi:>10.4f}{1/N:>10.4f}{mi*N:>8.1f}")
check("N=800 pool 12 still floored", influence(rotating(800, 12)).max(), 0.030)

print("\nPool needed to come within 2x the flat benchmark:")
for N in [50, 100, 200, 400, 800]:
    # Bisection rather than a linear scan. Max influence is monotone decreasing in the pool
    # size, so the two are equivalent, and they were verified to return identical R for every
    # N below on 2 August 2026. The linear version took 31 seconds at N = 800 and was most of
    # the notebook's total runtime, which mattered once the notebook started being run end to
    # end rather than cell by cell.
    _lo, _hi = 2, N
    while _lo < _hi:
        _mid = (_lo + _hi) // 2
        if influence(rotating(N, _mid)).max() <= 2.0/N: _hi = _mid
        else: _lo = _mid + 1
    R = _lo
    print(f"  N={N:>4}  pool {R:>3}  ({100*R/N:.0f}% of the group)")
    check(f"N={N} required pool is 26% of group", R/N, 0.26, tol=0.02)
print("\n  -> the requirement is a PROPORTION, not a headcount")

Pool sweep at N=400, officeholder share 0.35

  pool R   max infl    error   alpha/R
       3     0.1178    0.165    0.1167
       6     0.0597    0.119    0.0583
      12     0.0307    0.089    0.0292
      25     0.0156    0.067    0.0140
      50     0.0086    0.054    0.0070
     100     0.0051    0.047    0.0035
     400     0.0025    0.040    0.0009
  OK  N=400 pool 12, max influence                      0.031  (book: 0.031)
  OK  N=400 pool 12, error                              0.089  (book: 0.089)
  OK  N=400 flat, max influence                         0.003  (book: 0.0025)
  OK  N=400 flat, error                                 0.040  (book: 0.04)

A fixed pool of 12 floors while the flat benchmark keeps falling:
     N   pool 12      flat   ratio
    50    0.0413    0.0200     2.1
   100    0.0352    0.0100     3.5
   200    0.0322    0.0050     6.4
   400    0.0307    0.0025    12.3
   800    0.0299    0.0013    23.9
  OK  N=800 pool 12 still floored                       0

---
## 3. Protective traditions and the tradition comparison
### Chapters 5 and 6

Five traditions govern no resource any step consumes: autonomy (4), no endorsement (6),
self-support (7), no hierarchy (9), no outside issues (10). They act only as multipliers
on the enabling traditions they guard, applied once each with no compounding.

The comparison uses **common random numbers**: the identical seed sequence for every
configuration, so differences are not contaminated by differences in the random draws.

In [ ]:
print("Governance matrix: which traditions have zero rows (protective tier)\n")
protective = [j for j in range(12) if m.GOV[j].sum() == 0]
print("  protective (govern no resource):", [f"T{j+1}" for j in protective])
assert protective == [3, 5, 6, 8, 9], "protective tier changed"
print("  -> T4, T6, T7, T9, T10, as derived in Part Four\n")

print("Effective adherence applies each guard once, no compounding:")
T = np.full(12, 0.8)
print("  uniform 0.8 raw ->", np.round(m.effective_adherence(T)[[1, 4]], 3),
      "for T2 and T5 (compounding would drive these far lower)")

Governance matrix: which traditions have zero rows (protective tier)

  protective (govern no resource): ['T4', 'T6', 'T7', 'T9', 'T10']
  -> T4, T6, T7, T9, T10, as derived in Part Four

Effective adherence applies each guard once, no compounding:
  uniform 0.8 raw -> [0.692 0.692] for T2 and T5 (compounding would drive these far lower)


In [ ]:
# --- The twelve-Tradition degradation comparison, at 400 paired replications
# This cell used 30 replications, which violated CLAUDE.md rule 0 (minimum 400) and was the
# last small-sample figure in the project. It was also slow enough that the notebook was
# never run end to end, which is how the broken cell above went unnoticed. Recomputed by
# model/tradition_paired.py and read from disk. THE ORDERING CHANGED; see below.
_TP = _json.load(open(f'{_R}/tradition_paired.json'))
_tpm, _tpr = _TP['meta'], _TP['result']
print(f"Paired comparison, {_tpm['nseed']} replications, common random numbers.")
print(f"Each Tradition degraded alone from {_tpm['ref']} to {_tpm['low']}; horizon "
      f"{_tpm['horizon']} half-weeks, twenty years.\n")
print(f"reference, all traditions at {_tpm['ref']}: N = {_tpr['base_mean']:.1f}")
print(f"cross-seed SD = {_tpr['base_sd']:.1f}  <- this is the noise CRN removes\n")
print(f"{'trad':<6}{'tier':<12}{'members lost':>13}{'95% hw':>8}{'t':>7}  verdict")
_TPROT = (4, 6, 7, 9, 10)
for _r in sorted(_tpr['rows'], key=lambda r: -r['loss']):
    _j = _r['tradition']
    _tier = "PROTECTIVE" if _j in _TPROT else "enabling"
    _v = "significant" if abs(_r['t']) > 2.5 else ("marginal" if abs(_r['t']) > 1.8
                                                   else "not resolved")
    print(f"T{_j:<5}{_tier:<12}{_r['loss']:>13.2f}{1.96*_r['se']:>8.2f}{_r['t']:>7.1f}  {_v}")
_TPD = {r['tradition']: r for r in _tpr['rows']}
check("Tp reference N at 0.85", _tpr['base_mean'], 23.44, tol=0.02)
check("Tp cross-seed SD", _tpr['base_sd'], 11.93, tol=0.02)
for _j, _w in [(11, 7.90), (3, 5.16), (1, 4.47), (2, 3.43), (4, 2.82), (7, 2.82),
               (12, 2.56), (5, 2.27), (8, 1.02), (9, 0.89), (6, 0.71), (10, 0.71)]:
    check(f"Tp loss from degrading T{_j}", _TPD[_j]['loss'], _w, tol=0.02)
for _j, _w in [(11, 12.9), (3, 8.1), (1, 6.7), (2, 5.2), (4, 4.0), (7, 4.0), (12, 4.1)]:
    check(f"Tp t for T{_j}", _TPD[_j]['t'], _w, tol=0.06)
_nsig = sum(1 for r in _tpr['rows'] if abs(r['t']) > 2.5)
check("Tp comparisons resolved at |t|>2.5", float(_nsig), 8.0, tol=0.01)
_top = max(_tpr['rows'], key=lambda r: r['loss'])['tradition']
check("Tp largest loss is T11", float(_top), 11.0, tol=0.01)
_protrank = sorted([r['tradition'] for r in _tpr['rows']],
                   key=lambda j: -_TPD[j]['loss']).index(4) + 1
check("Tp rank of T4 among the twelve", float(_protrank), 5.0, tol=0.01)
print(f"\n{_nsig}/12 resolved at |t| > 2.5.")
print("""
  -> THE 30-REPLICATION VERSION OF THIS CELL GAVE A DIFFERENT AND WRONG ORDERING.
     It reported that only Traditions 4 and 7 cleared significance, at 5.5 members each with
     t = 2.6, with unity tied on the point estimate and six comparisons unresolved. That put
     the two PROTECTIVE Traditions at the top of the table, and Chapter 5 noted the
     coincidence with their derived role while warning it was not strong evidence.

     At 400 replications the top of the table is ENABLING, not protective. Attraction leads
     by a wide margin at 7.90 members and t = 12.9, then the open door at 5.16, unity at 4.47
     and group conscience at 3.43. Traditions 4 and 7 are still significant but rank fifth
     and sixth at 2.82. Eight of twelve are now resolved rather than two.

     Chapter 5's caution was right and its table was wrong. Both have been rewritten.""")

# --- figures Chapter 6 quotes from this same comparison
_ses = [r['se'] for r in _tpr['rows']]
print(f"\npaired SE range {min(_ses):.2f} to {max(_ses):.2f}; "
      f"unpaired cross-seed SD {_tpr['base_sd']:.2f}; "
      f"variance reduction {_tpr['base_sd']/np.mean(_ses):.1f}x")
check("Ch6 smallest paired SE", min(_ses), 0.57, tol=0.006)
check("Ch6 largest paired SE", max(_ses), 0.70, tol=0.006)
check("Ch6 CRN variance reduction factor", _tpr['base_sd']/np.mean(_ses), 18.8, tol=0.06)
check("Ch6 comparisons below |t|=1.8", float(sum(1 for r in _tpr['rows'] if abs(r['t']) < 1.8)), 4.0, tol=0.01)


Paired comparison, 400 replications, common random numbers.
Each Tradition degraded alone from 0.85 to 0.5; horizon 1040 half-weeks, twenty years.

reference, all traditions at 0.85: N = 23.4
cross-seed SD = 11.9  <- this is the noise CRN removes

trad  tier         members lost  95% hw      t  verdict
T11   enabling             7.90    1.20   12.9  significant
T3    enabling             5.16    1.25    8.1  significant
T1    enabling             4.47    1.30    6.7  significant
T2    enabling             3.43    1.30    5.2  significant
T4    PROTECTIVE           2.82    1.37    4.0  significant
T7    PROTECTIVE           2.82    1.37    4.0  significant
T12   enabling             2.56    1.24    4.1  significant
T5    enabling             2.27    1.33    3.4  significant
T8    enabling             1.02    1.20    1.7  not resolved
T9    PROTECTIVE           0.89    1.15    1.5  not resolved
T6    PROTECTIVE           0.71    1.13    1.2  not resolved
T10   PROTECTIVE           0.71  

---
## 4. The five-member worked example
### Chapter 7

Five people deciding how many months of expenses to hold in reserve. Starting views
9, 2, 6, 4, 7. The chapter claims: they converge in three rounds to 5.74; the plain
average is 5.60; and the two most open-minded members finish with the least influence.

In [ ]:
names = ["Ann", "Ben", "Cara", "Dan", "Eve"]
A5 = np.array([
    [.30, .20, .30, .10, .10],
    [.15, .25, .35, .15, .10],
    [.20, .15, .35, .15, .15],
    [.15, .15, .35, .20, .15],
    [.20, .15, .30, .15, .20]])
A5 = A5/A5.sum(1, keepdims=True)
b0 = np.array([9., 2., 6., 4., 7.])

b = b0.copy()
print("round-by-round:")
for r in range(4):
    print(f"  {r}: " + "  ".join(f"{n} {v:5.2f}" for n, v in zip(names, b)))
    b = A5 @ b

s5 = influence(A5)
print()
check("settles at",                    float(b[0]),        5.74)
check("plain average of start views",  float(b0.mean()),   5.60)
check("influence-weighted average",    float(s5 @ b0),     5.74)
print()
print("influence vs open-mindedness:")
for i, n in enumerate(names):
    print(f"  {n:<5} weight placed on others {1-A5[i,i]:.2f}   influence {s5[i]:.3f}")
check("Cara influence (least open-minded)", float(s5[2]), 0.333)
check("Dan influence",                       float(s5[3]), 0.147)
check("Eve influence (most open-minded)",    float(s5[4]), 0.138)
print("\n  -> influence is conferred by others, not earned by listening")

round-by-round:
  0: Ann  9.00  Ben  2.00  Cara  6.00  Dan  4.00  Eve  7.00
  1: Ann  6.00  Ben  5.25  Cara  5.85  Dan  5.60  Eve  5.90
  2: Ann  5.75  Ben  5.69  Cara  5.76  Dan  5.74  Eve  5.76
  3: Ann  5.74  Ben  5.74  Cara  5.75  Dan  5.75  Eve  5.75

  OK  settles at                                        5.744  (book: 5.74)
  OK  plain average of start views                      5.600  (book: 5.6)
  OK  influence-weighted average                        5.744  (book: 5.74)

influence vs open-mindedness:
  Ann   weight placed on others 0.70   influence 0.204
  Ben   weight placed on others 0.75   influence 0.178
  Cara  weight placed on others 0.65   influence 0.333
  Dan   weight placed on others 0.80   influence 0.147
  Eve   weight placed on others 0.80   influence 0.138
  OK  Cara influence (least open-minded)                0.333  (book: 0.333)
  OK  Dan influence                                     0.147  (book: 0.147)
  OK  Eve influence (most open-minded)                  

In [ ]:
# The split room: two factions, no cross-listening, never converges
B = np.array([[.5, .5, 0, 0], [.5, .5, 0, 0], [0, 0, .5, .5], [0, 0, .5, .5]])
b = np.array([10., 8., 2., 0.])
for _ in range(20): b = B @ b
print("split room after 20 rounds:", np.round(b, 3))
check("faction one settles at", float(b[0]), 9.0)
check("faction two settles at", float(b[2]), 1.0)
print("  -> two settled camps, no consensus: the chain is not strongly connected")

split room after 20 rounds: [9. 9. 1. 1.]
  OK  faction one settles at                            9.000  (book: 9.0)
  OK  faction two settles at                            1.000  (book: 1.0)
  -> two settled camps, no consensus: the chain is not strongly connected


---
## 5. The asymmetry
### Chapter 9

Group state is an aggregate of member states, so member-to-group transmission carries
weight 1 by construction. Group-to-member transmission is multiplied by a per-step
coefficient derived in Part Four from what each step consumes from a group.

In [ ]:
steps = ["1 admit", "2 believe", "3 decide", "4 inventory", "5 tell someone", "6 willing",
         "7 ask", "8 list harms", "9 amends", "10 daily", "11 connect", "12 carry it"]
for n, b in zip(steps, m.BETA):
    print(f"  {n:<16}{b:.2f}")
print()
check("max group-dependence (Steps 1 and 12)", float(m.BETA.max()),  1.00)
check("min group-dependence (Step 7)",          float(m.BETA.min()),  0.17)
check("mean group-dependence",                  float(m.BETA.mean()), 0.53)
print(f"\nupward:downward transmission ratio ranges 1 to {1/m.BETA.min():.1f}, averaging {1/m.BETA.mean():.1f}")

  1 admit         1.00
  2 believe       0.71
  3 decide        0.29
  4 inventory     0.33
  5 tell someone  0.71
  6 willing       0.25
  7 ask           0.17
  8 list harms    0.33
  9 amends        0.62
  10 daily        0.62
  11 connect      0.33
  12 carry it     1.00

  OK  max group-dependence (Steps 1 and 12)             1.000  (book: 1.0)
  OK  min group-dependence (Step 7)                     0.167  (book: 0.17)
  OK  mean group-dependence                             0.531  (book: 0.53)

upward:downward transmission ratio ranges 1 to 6.0, averaging 1.9


---
## 7. Sensitivity analysis
### Preface, and `research/PARAMETERS.md`

The model contains **118 numbers chosen by hand** and none fitted to data. This section
tests which of the book's simulation claims survive perturbing them.

Four designs, in increasing order of how defensible they are:

1. **Uniform global.** Every parameter and both matrices jittered by the same percentage.
   Run at 12.5, 25 and 50 per cent to give a degradation curve.
2. **One at a time.** Each scalar varied alone, to find which ones drive the fragile result.
3. **Tiered.** Parameters pinned by a size-and-composition target moved by 25 per cent;
   parameters that are pure judgement moved by 50, with bounds enforced where theory
   requires them (the Hill exponent must exceed 1 for bistability to be possible at all).
   This is more defensible than uniform, because it perturbs each number by how little
   evidence constrains it rather than treating them as equally uncertain.
4. **Structural.** The two matrices replaced with random values, preserving only which
   cells are zero. This asks whether the results depend on my judgements about magnitudes
   or only on the derived sparsity pattern.

These runs take roughly forty minutes. Set `REPRODUCE = True` to re-run them; otherwise the
recorded values are asserted below.

In [ ]:
REPRODUCE = False   # set True to re-run the raw sweeps from scratch (~40 min)

# ---- the sweeps are now DERIVED from the saved raw draws, not typed in by hand ---------
# research/sens3.json  : 30 draws at each of 12.5 / 25 / 50 per cent, plus the OAT sweep
# research/tiered.json : 30 draws with per-parameter credible ranges, and 30 with both
#                        matrices replaced by random numbers keeping only their sparsity
import json as _json, os as _os
_R = '../research' if _os.path.isdir('../research') else 'research'
S3 = _json.load(open(f'{_R}/sens3.json'))
TI = _json.load(open(f'{_R}/tiered.json'))

# Each draw records the fraction of 3 seeds that survived, so a claim can be scored two
# ways. Which one is used matters and was previously inconsistent between rows.
EVERY = lambda v: v == 1.0        # survived in EVERY seed
ANY   = lambda v: v > 0.0         # survived in AT LEAST ONE seed

CLAIMS = {
    'full adherence survives':            lambda r: r['full_s'],
    'attraction lost, group survives':    lambda r: r['att_s'],
}
def frac(rows, get, std): return sum(1 for r in rows if std(get(r)))/len(rows)
def pair(rows, get):      return frac(rows, get, EVERY), frac(rows, get, ANY)

LEV = [(12.5,'g125'), (25,'g250'), (50,'g500')]

print("1. UNIFORM GLOBAL, degradation curve. Survival scored two ways, because the\n"
      "   two standards diverge and earlier drafts mixed them.\n")
print(f"{'claim':<34}{'standard':<14}" + "".join(f"{l:>8}%" for l, _ in LEV))
for name, get in CLAIMS.items():
    for lab, std in (('every seed', EVERY), ('any seed', ANY)):
        vals = [frac(S3[k], get, std) for _, k in LEV]
        print(f"{name:<34}{lab:<14}" + "".join(f"{v*100:>8.0f}%" for v in vals))

ORD = lambda r: 1.0 if r['ref_s'] <= r['att_s'] else 0.0
WOR = lambda r: 1.0 if r['ref_s'] <  r['full_s'] else 0.0
ordv = [frac(S3[k], ORD, EVERY) for _, k in LEV]
worv = [frac(S3[k], WOR, EVERY) for _, k in LEV]
print(f"\n{'referral loss <= attraction loss':<48}" + "".join(f"{v*100:>8.0f}%" for v in ordv))
print(f"{'referral loss strictly worse than full':<48}" + "".join(f"{v*100:>8.0f}%" for v in worv))

check("full adherence survives every seed, 12.5%", frac(S3['g125'], CLAIMS['full adherence survives'], EVERY), 0.97)
check("full adherence survives every seed, 25%",   frac(S3['g250'], CLAIMS['full adherence survives'], EVERY), 0.90)
check("full adherence survives every seed, 50%",   frac(S3['g500'], CLAIMS['full adherence survives'], EVERY), 0.73)
check("attraction lost, every seed, 25%",  frac(S3['g250'], CLAIMS['attraction lost, group survives'], EVERY), 0.97)
check("attraction lost, every seed, 50%",  frac(S3['g500'], CLAIMS['attraction lost, group survives'], EVERY), 0.63)
check("attraction lost, any seed, 50%",    frac(S3['g500'], CLAIMS['attraction lost, group survives'], ANY),   0.83)
for lvl, k, want in [(12.5,'g125',1.00), (25,'g250',1.00), (50,'g500',0.87)]:
    check(f"ordering holds at {lvl}%", frac(S3[k], ORD, EVERY), want)
for lvl, k, want in [(12.5,'g125',0.73), (25,'g250',0.57), (50,'g500',0.50)]:
    check(f"referral-starved death at {lvl}%", frac(S3[k], WOR, EVERY), want)

print("\n   -> the attraction row is the one that changes with the standard: 100/97/63 if\n"
      "      the group must survive in every seed, 100/100/83 if one seed is enough.\n"
      "      PARAMETERS.md now states both. Earlier drafts quoted the second under a\n"
      "      heading that implied the first.")

print("\n2. ONE AT A TIME: referral-starved survival, each scalar varied alone by 25%\n")
OAT_SHOWN = ['delta0', 'p_gate', 'drop_k', 'churn', 'psi', 'k_proof', 'het_sd']
print(f"{'parameter':<12}{'-25%':>8}{'+25%':>8}{'swing':>8}")
for k in OAT_SHOWN:
    lo, hi = S3['oat'][k]['low'], S3['oat'][k]['high']
    print(f"{k:<12}{lo:>8.2f}{hi:>8.2f}{abs(hi-lo):>8.2f}")
for k, wl, wh in [('delta0',1.00,0.00), ('p_gate',1.00,0.00), ('drop_k',0.00,1.00),
                  ('churn',1.00,0.33), ('psi',0.67,0.00), ('k_proof',0.67,0.00), ('het_sd',0.00,0.67)]:
    check(f"OAT {k} at -25%", S3['oat'][k]['low'],  wl)
    check(f"OAT {k} at +25%", S3['oat'][k]['high'], wh)
zero = sorted(k for k, v in S3['oat'].items() if v['low'] == 0.0 and v['high'] == 0.0)
flat = sorted(k for k, v in S3['oat'].items() if v['low'] == v['high'] and k not in zero)
print(f"\n  no effect at either end ({len(zero)}): {zero}")
print(f"  same value at both ends but not zero ({len(flat)}): {flat}")
check("count of no-effect parameters", float(len(zero)), 4.0, tol=0.01)
check("count of flat-but-nonzero parameters", float(len(flat)), 3.0, tol=0.01)
print("  PARAMETERS.md said seven had no effect. Four do. The other three move the")
print("  outcome to 0.33 in BOTH directions, which is not the same thing as inert.")
print("  Three parameters alone take the outcome from certain death to certain survival.")
print("  Referral-starved survival is a knife-edge, not an estimate.")

print("\n3. TIERED (credible ranges) vs 4. STRUCTURAL (matrices randomised)\n")
print(f"{'claim':<40}{'standard':<13}{'tiered':>9}{'struct':>9}")
for name, get in CLAIMS.items():
    for lab, std in (('every seed', EVERY), ('any seed', ANY)):
        print(f"{name:<40}{lab:<13}{frac(TI['tiered'],get,std)*100:>8.0f}%{frac(TI['randmat'],get,std)*100:>8.0f}%")
for nm, key, o, w in [('tiered','tiered',1.00,0.67), ('structural','randmat',1.00,0.90)]:
    check(f"{nm}: ordering holds",   frac(TI[key], ORD, EVERY), o)
    check(f"{nm}: referral-starved", frac(TI[key], WOR, EVERY), w)
check("tiered: attraction lost, every seed",  frac(TI['tiered'],  CLAIMS['attraction lost, group survives'], EVERY), 0.93)
check("struct: full adherence, every seed",   frac(TI['randmat'], CLAIMS['full adherence survives'],         EVERY), 0.93)

print("\n--- verdict ---")
assert frac(S3['g125'], ORD, EVERY) == 1.00
assert frac(TI['tiered'], ORD, EVERY) == 1.00 and frac(TI['randmat'], ORD, EVERY) == 1.00
print("ROBUST     : losing referrals is worse than losing attraction.")
print("             100% under every design, including fully randomised matrices.")
assert frac(S3['g125'], WOR, EVERY) < 0.8
print("NOT ROBUST : the magnitude of referral-starved death. Fails at the gentlest shake.")
print("\nThe structural test is the strongest defence in the book: the ordering survives")
print("replacing both hand-written matrices with random numbers, so it depends on the")
print("DERIVED sparsity pattern, not on my judgement about magnitudes.")
print("\nEverything above is now computed from the saved raw draws, so the preface's")
print("robustness paragraph is reproducible from this repository.")


1. UNIFORM GLOBAL, degradation curve. Survival scored two ways, because the
   two standards diverge and earlier drafts mixed them.

claim                             standard          12.5%      25%      50%
full adherence survives           every seed          97%      90%      73%
full adherence survives           any seed           100%     100%      93%
attraction lost, group survives   every seed         100%      97%      63%
attraction lost, group survives   any seed           100%     100%      83%

referral loss <= attraction loss                     100%     100%      87%
referral loss strictly worse than full                73%      57%      50%
  OK  full adherence survives every seed, 12.5%         0.967  (book: 0.97)
  OK  full adherence survives every seed, 25%           0.900  (book: 0.9)
  OK  full adherence survives every seed, 50%           0.733  (book: 0.73)
  OK  attraction lost, every seed, 25%                  0.967  (book: 0.97)
  OK  attraction lost, every se

In [ ]:
if REPRODUCE:
    import json, subprocess, sys
    # The sweep scripts live alongside this notebook. Each saves incrementally so a
    # timeout does not lose completed levels.
    for script in ('sensitivity_uniform.py', 'sensitivity_tiered.py'):
        print('running', script)
        subprocess.run([sys.executable, script], check=True)
    print('re-run complete; compare against the recorded dictionaries above')
else:
    print('REPRODUCE is False. Recorded values shown above; set True to re-run.')

REPRODUCE is False. Recorded values shown above; set True to re-run.


---
## 7. Can a step be skipped?
### Chapter 13

Each step is a production stage combining own stock, prior step, group input and
maintenance, in the CES form of Cunha, Heckman and Schennach. One parameter, rho, governs
substitutability. The folk claim that a step cannot be skipped is the limiting case as rho
goes to minus infinity, but strict ordering in fact holds for **all rho <= 0**, which is a
much wider hypothesis than the extreme corner.

In [ ]:
np.random.seed(3)
GAM = np.array([.30, .30, .25, .15])     # own stock, prior step, group input, maintenance

def ces(x, g, rho):
    x = np.asarray(x, float); g = np.asarray(g, float)
    if abs(rho) < 1e-8: return float(np.prod(x**g))     # Cobb-Douglas limit
    if rho < -60:       return float(x.min())           # Leontief limit
    with np.errstate(divide='ignore', invalid='ignore'):
        return float((g @ (x**rho))**(1/rho))

print("Output when the prior step is zero (other inputs 0.7, 0.8, 0.6)\n")
print(f"{'rho':>8}{'elasticity':>12}{'output':>10}   reading")
for r, want in [(-100,0.0), (-4,0.0), (-1,0.0), (0.0,0.0), (0.3,0.2167), (0.6,0.3933), (0.9,0.4803)]:
    o = ces([0.7, 0.0, 0.8, 0.6], GAM, r)
    e = '0.00' if r < -60 else f"{1/(1-r):.2f}"
    print(f"{r:>8.1f}{e:>12}{o:>10.4f}   {'cannot skip' if o < 1e-9 else 'CAN skip'}")
    check(f"rho={r} output with prior step zero", o, want, tol=0.002)
print("\n  -> the boundary is at rho = 0, not at minus infinity")

print("\nDynamic complementarity: is group help worth more after the prior work?")
def cross(rho, h=1e-4, b=(.7, .4, .5, .6)):
    f = lambda a, c: ces([b[0], a, c, b[3]], GAM, rho)
    return (f(b[1]+h,b[2]+h) - f(b[1]+h,b[2]-h) - f(b[1]-h,b[2]+h) + f(b[1]-h,b[2]-h))/(4*h*h)
for r, want in [(-4,1.729), (-1,0.523), (0.0,0.199), (0.5,0.084)]:
    check(f"cross-partial at rho={r}", cross(r), want, tol=0.01)
print("  positive throughout, larger the more complementary the technology")

Output when the prior step is zero (other inputs 0.7, 0.8, 0.6)

     rho  elasticity    output   reading
  -100.0        0.00    0.0000   cannot skip
  OK  rho=-100 output with prior step zero              0.000  (book: 0.0)
    -4.0        0.20    0.0000   cannot skip
  OK  rho=-4 output with prior step zero                0.000  (book: 0.0)
    -1.0        0.50    0.0000   cannot skip
  OK  rho=-1 output with prior step zero                0.000  (book: 0.0)
     0.0        1.00    0.0000   cannot skip
  OK  rho=0.0 output with prior step zero               0.000  (book: 0.0)
     0.3        1.43    0.2167   CAN skip
  OK  rho=0.3 output with prior step zero               0.217  (book: 0.2167)
     0.6        2.50    0.3933   CAN skip
  OK  rho=0.6 output with prior step zero               0.393  (book: 0.3933)
     0.9       10.00    0.4803   CAN skip
  OK  rho=0.9 output with prior step zero               0.480  (book: 0.4803)

  -> the boundary is at rho = 0, not at minus infinit

In [ ]:
# --- Ch 13: recovery of rho, replicated 25 times per cell instead of once
# The original reported a SINGLE draw at seed 3, to three significant figures. One
# realisation of an estimator says nothing about the estimator. Raw replications in
# research/ch13_reps.json, generated by ch13_reps.py.
RP = list(_json.load(open(f'{_R}/ch13_reps.json')).values())
BOOK1 = {(-4.0,1):-3.45, (-4.0,3):-3.99, (-1.0,1):-1.12, (-1.0,3):-1.00,
         (0.0,1):-0.10,  (0.0,3):0.02,   (0.5,1):0.35,   (0.5,3):0.56}
print("Recovery of rho: the single draw the book printed, against 25 replications\n")
print(f"{'true rho':>9}{'proxies':>9}{'one draw':>10}{'mean':>8}{'sd':>7}{'bias':>8}{'RMSE':>8}")
ST = {}
for rt in (-4., -1., 0., 0.5):
    for npx in (1, 3):
        e = np.array([v['est'] for v in RP if v['rt'] == rt and v['npx'] == npx], float)
        ST[(rt, npx)] = (e.mean(), e.std(ddof=1))
        print(f"{rt:>9.1f}{npx:>9}{BOOK1[(rt,npx)]:>10.2f}{e.mean():>8.2f}{e.std(ddof=1):>7.2f}"
              f"{e.mean()-rt:>8.2f}{np.sqrt(((e-rt)**2).mean()):>8.2f}")
for (rt, npx), (wm, ws) in {(-4.,1):(-4.21,0.59), (-4.,3):(-4.02,0.23),
                            (-1.,1):(-1.02,0.15), (-1.,3):(-1.00,0.07),
                            (0.,1):(0.06,0.17),   (0.,3):(0.02,0.08),
                            (0.5,1):(0.49,0.17),  (0.5,3):(0.48,0.08)}.items():
    check(f"Ch13 rho={rt} npx={npx}, mean", ST[(rt,npx)][0], wm, tol=0.011)
    check(f"Ch13 rho={rt} npx={npx}, sd",   ST[(rt,npx)][1], ws, tol=0.011)

print("\nSign recovery, which is the whole question:")
for npx in (1, 3):
    e0 = np.array([v['est'] for v in RP if v['rt'] == 0.0 and v['npx'] == npx], float)
    e5 = np.array([v['est'] for v in RP if v['rt'] == 0.5 and v['npx'] == npx], float)
    print(f"  {npx} proxy/proxies: true 0.0 -> {100*np.mean(e0<=0):.0f}% called non-positive;"
          f"  true +0.5 -> {100*np.mean(e5>0):.0f}% called positive")
check("Ch13 sign recovery at true +0.5, one proxy",
      float(np.mean(np.array([v['est'] for v in RP if v['rt']==0.5 and v['npx']==1])>0)), 1.0, tol=0.01)

print("\n  -> THE SINGLE DRAW WAS MISLEADING, and the corrected story is better.")
print("     The estimator is close to unbiased with one proxy or three; the bias never")
print("     exceeds 0.21. What three proxies buy is PRECISION: the standard deviation")
print("     roughly halves, from 0.17 to 0.08 near the boundary and 0.59 to 0.23 far from it.")
print("     Exactly at rho = 0 the sign is a coin flip whatever you do, because an")
print("     approximately unbiased estimator sitting on a boundary lands either side. That")
print("     is a property of the boundary, not a failure of the instrument.")
print("     The defensible design requirement is therefore about RESOLUTION, not sign:")
print("     one proxy cannot separate rho = 0 from rho = +/-0.3; three can separate it")
print("     from about +/-0.15. Neither can resolve anything closer than that.")


Recovery of rho: the single draw the book printed, against 25 replications

 true rho  proxies  one draw    mean     sd    bias    RMSE
     -4.0        1     -3.45   -4.21   0.59   -0.21    0.61
     -4.0        3     -3.99   -4.02   0.23   -0.02    0.23
     -1.0        1     -1.12   -1.02   0.15   -0.02    0.15
     -1.0        3     -1.00   -1.00   0.07   -0.00    0.07
      0.0        1     -0.10    0.06   0.17    0.06    0.17
      0.0        3      0.02    0.02   0.08    0.02    0.08
      0.5        1      0.35    0.49   0.17   -0.01    0.16
      0.5        3      0.56    0.48   0.08   -0.02    0.08
  OK  Ch13 rho=-4.0 npx=1, mean                        -4.207  (book: -4.21)
  OK  Ch13 rho=-4.0 npx=1, sd                           0.588  (book: 0.59)
  OK  Ch13 rho=-4.0 npx=3, mean                        -4.021  (book: -4.02)
  OK  Ch13 rho=-4.0 npx=3, sd                           0.229  (book: 0.23)
  OK  Ch13 rho=-1.0 npx=1, mean                        -1.016  (book: -1.02)
 

---
## 8. The leaky bucket: maintenance, the threshold, and what it does not know
### Chapter 14

Maintenance capacity is the mean of Steps 10, 11 and 12. It gates every step's growth
through a Hill function, weighted by step index (0.05 at Step 1 to 1.00 at Step 12), so
maintenance gates its own accumulation and the loop is **bistable**.

The chapter claims the **shape** and explicitly refuses the **location**. This section
computes both, and the fragility sweep is the reason for the refusal: a 2.79 per cent
rise in `delta0` does not move the threshold, it **destroys the healthy equilibrium**.

The final block is the individual-versus-group result: member heterogeneity spreads
individual critical decay rates over a factor of four, so a population of individually
bistable members responds smoothly at every point.

In [ ]:
# --- Ch 14 individual dynamics: one member, group held at its full-adherence equilibrium
WGT   = np.linspace(0.05, 1.0, 12)          # per-step exposure to the capacity gate
R_EQ  = np.array([1., .476, .824, .875, .824, .243, .508, .508])   # equilibrium resources
GCAP  = 0.245                                # mean own-capacity across living members

def _rhs(X, P, het, d0, peer, gcap):
    hill = lambda M: M**P['hill_n'] / (P['hill_k']**P['hill_n'] + M**P['hill_n'])
    gate = np.ones_like(X); gate[:, 1:] = np.clip(X[:, :-1], 0, 1)**P['p_gate']
    own  = hill(X[:, 9:12].mean(axis=1))
    C    = own + (1 - own)*P['omega']*gcap
    Cm   = 1.0 - WGT[None, :]*(1.0 - C[:, None])
    g    = P['a']*gate*peer*Cm*het[:, None]
    d    = np.repeat(d0.reshape(-1, 1), 12, axis=1)*np.ones_like(X)
    d[:, :11] *= (1 + P['psi']*(1 - X[:, 1:12]))
    return g*(1 - X) - d*X

def settle(X0, P=None, het=None, d0=None, T=800., dt=0.5, R=None, gcap=None):
    """Integrate to the attractor. T=800, dt=0.5 is converged to five decimals (checked
    against T=5000, dt=0.25). R and gcap are passed explicitly, never via globals, so the
    result does not depend on the order cells are run in."""
    P = P or m.DEFAULTS
    X = np.array(X0, float, ndmin=2).copy(); n = len(X)
    het = np.ones(n) if het is None else np.broadcast_to(np.atleast_1d(het), (n,)).astype(float)
    d0  = np.full(n, P['delta0']) if d0 is None else np.broadcast_to(np.atleast_1d(d0), (n,)).astype(float)
    R    = R_EQ if R is None else R
    gcap = GCAP if gcap is None else gcap
    peer = (1 - m.BETA) + m.BETA*(m.Snorm @ R)
    for _ in range(int(T/dt)): X = np.clip(X + dt*_rhs(X, P, het, d0, peer, gcap), 0, 1)
    return X

maint = lambda X: X[:, 9:12].mean(axis=1)

hi = settle(np.full((1, 12), 0.95))[0]
lo = settle(np.full((1, 12), 0.02))[0]
steps = ["1 admit","2 believe","3 decide","4 inventory","5 tell someone","6 willing",
         "7 ask","8 list harms","9 amends","10 daily","11 connect","12 carry it"]
print("Two attractors, identical circumstances, different starting points\n")
print(f"{'':<16}{'healthy':>9}{'collapsed':>11}")
for s, h, l in zip(steps, hi, lo): print(f"  {s:<14}{h:>9.3f}{l:>11.3f}")
print(f"  {'mean of twelve':<14}{hi.mean():>9.4f}{lo.mean():>11.4f}")
print(f"  {'maintenance':<14}{hi[9:12].mean():>9.4f}{lo[9:12].mean():>11.4f}\n")

check("Ch14 healthy maintenance",        float(hi[9:12].mean()), 0.3205, tol=0.002)
check("Ch14 collapsed maintenance",      float(lo[9:12].mean()), 0.0003, tol=0.002)
check("Ch14 healthy mean of twelve",     float(hi.mean()),       0.5003, tol=0.002)
check("Ch14 collapsed mean of twelve",   float(lo.mean()),       0.2698, tol=0.002)
for i, (wh, wl) in {0:(0.766,0.758), 3:(0.559,0.441), 7:(0.459,0.056), 11:(0.250,0.000)}.items():
    check(f"Ch14 Step {i+1} healthy",   float(hi[i]), wh, tol=0.002)
    check(f"Ch14 Step {i+1} collapsed", float(lo[i]), wl, tol=0.002)
print("\n  -> the collapse is back to front: Step 1 is untouched, the back end is gone")


Two attractors, identical circumstances, different starting points

                  healthy  collapsed
  1 admit           0.766      0.758
  2 believe         0.673      0.641
  3 decide          0.659      0.592
  4 inventory       0.559      0.441
  5 tell someone    0.537      0.348
  6 willing         0.517      0.245
  7 ask             0.508      0.147
  8 list harms      0.459      0.056
  9 amends          0.364      0.009
  10 daily          0.365      0.001
  11 connect        0.347      0.000
  12 carry it       0.250      0.000
  mean of twelve   0.5003     0.2698
  maintenance      0.3205     0.0003

  OK  Ch14 healthy maintenance                          0.320  (book: 0.3205)
  OK  Ch14 collapsed maintenance                        0.000  (book: 0.0003)
  OK  Ch14 healthy mean of twelve                       0.500  (book: 0.5003)
  OK  Ch14 collapsed mean of twelve                     0.270  (book: 0.2698)
  OK  Ch14 Step 1 healthy                               0.766  (

In [ ]:
# --- Ch 14 separatrix: uniform scaling of the healthy state
f = np.linspace(0.01, 1.0, 400)
mm = maint(settle(hi[None, :]*f[:, None]))
fc = f[np.argmax(mm > 0.05)]
print(f"critical scale factor {fc:.4f}; maintenance at the boundary {hi[9:12].mean()*fc:.4f}\n")
for g in [0.70, 0.73, 0.75, 0.90]:
    print(f"  start at {g:.2f} of healthy -> settles at maintenance {maint(settle(hi[None, :]*g))[0]:.4f}")
check("Ch14 separatrix scale factor", float(fc),                  0.742,  tol=0.004)
check("Ch14 maintenance at boundary", float(hi[9:12].mean()*fc),  0.2378, tol=0.002)

# --- the fragility that forbids the location claim
print("\nDecay-rate sweep. Above the critical value the HEALTHY state ceases to exist.\n")
pcs = np.array([-20, -5, 0, 2, 5], float)
d0  = m.DEFAULTS['delta0']*(1 + pcs/100)
H = maint(settle(np.tile(np.full(12, 0.95), (len(d0), 1)), d0=d0))
L = maint(settle(np.tile(np.full(12, 0.02), (len(d0), 1)), d0=d0))
print(f"{'delta0':>9}{'change':>9}{'healthy':>10}{'collapsed':>11}  ")
for p, d, h, l in zip(pcs, d0, H, L):
    print(f"{d:>9.4f}{p:>8.0f}%{h:>10.4f}{l:>11.4f}  {'bistable' if h-l > 0.05 else 'SINGLE STATE'}")
for p, want in [(-20, 0.5569), (-5, 0.3987), (0, 0.3205), (2, 0.2718)]:
    check(f"Ch14 healthy state at delta0 {p:+.0f}%",
          float(maint(settle(np.full((1, 12), 0.95), d0=np.array([m.DEFAULTS['delta0']*(1+p/100)])))[0]),
          want, tol=0.003)

grid = np.linspace(m.DEFAULTS['delta0'], m.DEFAULTS['delta0']*1.10, 1200)
Hg   = maint(settle(np.tile(np.full(12, 0.95), (len(grid), 1)), d0=grid))
dcrit = grid[np.argmax(Hg < 0.05)]
pct   = 100*(dcrit/m.DEFAULTS['delta0'] - 1)
print(f"\n  healthy state annihilated at delta0 = {dcrit:.5f}, {pct:.2f}% above baseline")
check("Ch14 critical delta0", float(dcrit), 0.0617, tol=0.0002)
check("Ch14 critical delta0, per cent above baseline", float(pct), 2.79, tol=0.05)
print("  -> the chapter claims the SHAPE and refuses the LOCATION. This is why.")


critical scale factor 0.7420; maintenance at the boundary 0.2378

  start at 0.70 of healthy -> settles at maintenance 0.0003
  start at 0.73 of healthy -> settles at maintenance 0.0003
  start at 0.75 of healthy -> settles at maintenance 0.3205
  start at 0.90 of healthy -> settles at maintenance 0.3205
  OK  Ch14 separatrix scale factor                      0.742  (book: 0.742)
  OK  Ch14 maintenance at boundary                      0.238  (book: 0.2378)

Decay-rate sweep. Above the critical value the HEALTHY state ceases to exist.

   delta0   change   healthy  collapsed  
   0.0480     -20%    0.5569     0.0074  bistable
   0.0570      -5%    0.3987     0.0007  bistable
   0.0600       0%    0.3205     0.0003  bistable
   0.0612       2%    0.2718     0.0002  bistable
   0.0630       5%    0.0001     0.0001  SINGLE STATE
  OK  Ch14 healthy state at delta0 -20%                 0.557  (book: 0.5569)
  OK  Ch14 healthy state at delta0 -5%                  0.399  (book: 0.3987)
  OK  C

In [ ]:
# --- Ch 14 hysteresis: withdraw the group entirely for D weeks, then restore it fully
print("Total withdrawal of group support, then five years back in the room\n")
print(f"{'weeks away':>11}{'maintenance at end':>21}{'after five years back':>23}")
rows = {}
for w in [8., 11., 12., 26.]:
    x = settle(hi[None, :], T=w, R=np.zeros(8), gcap=0.0)
    end, back = maint(x)[0], maint(settle(x, T=520.))[0]
    rows[w] = (end, back)
    print(f"{w:>11.0f}{end:>21.4f}{back:>23.4f}")
for w, (we, wb) in {8.:(0.2397,0.3205), 11.:(0.2132,0.3204), 12.:(0.2047,0.0003), 26.:(0.1060,0.0003)}.items():
    check(f"Ch14 absence {w:.0f}w, maintenance at end", float(rows[w][0]), we, tol=0.002)
    check(f"Ch14 absence {w:.0f}w, after restoration",  float(rows[w][1]), wb, tol=0.002)

a, b = 0., 60.
for _ in range(30):
    mid = (a + b)/2
    x = settle(hi[None, :], T=mid, R=np.zeros(8), gcap=0.0)
    if maint(settle(x, T=520.))[0] > 0.05: a = mid
    else: b = mid
print(f"\n  critical absence {(a+b)/2:.2f} weeks")
check("Ch14 critical absence, weeks", float((a+b)/2), 11.50, tol=0.05)
print("  -> hysteresis: restoring the conditions does not restore the state.")
print("     The ASYMMETRY is the claim. The eleven weeks is not.")


Total withdrawal of group support, then five years back in the room

 weeks away   maintenance at end  after five years back
          8               0.2397                 0.3205
         11               0.2132                 0.3204
         12               0.2047                 0.0003
         26               0.1060                 0.0003
  OK  Ch14 absence 8w, maintenance at end               0.240  (book: 0.2397)
  OK  Ch14 absence 8w, after restoration                0.320  (book: 0.3205)
  OK  Ch14 absence 11w, maintenance at end              0.213  (book: 0.2132)
  OK  Ch14 absence 11w, after restoration               0.320  (book: 0.3204)
  OK  Ch14 absence 12w, maintenance at end              0.205  (book: 0.2047)
  OK  Ch14 absence 12w, after restoration               0.000  (book: 0.0003)
  OK  Ch14 absence 26w, maintenance at end              0.106  (book: 0.106)
  OK  Ch14 absence 26w, after restoration               0.000  (book: 0.0003)

  critical absence 11.50 we

In [ ]:
# --- Ch 14 individual bistability vs graded group response
rng = np.random.default_rng(0)
NPOP = 2000
het = np.exp(rng.normal(0, m.DEFAULTS['het_sd'], NPOP))
lo_, hi_ = np.full(NPOP, 0.002), np.full(NPOP, 0.60)
X95 = np.tile(np.full(12, 0.95), (NPOP, 1))
for _ in range(26):                                  # each member's own critical decay rate
    mid = (lo_ + hi_)/2
    ok  = maint(settle(X95, het=het, d0=mid)) > 0.05
    lo_ = np.where(ok, mid, lo_); hi_ = np.where(ok, hi_, mid)
crit = (lo_ + hi_)/2
crit[maint(settle(X95, het=het, d0=np.full(NPOP, 0.002))) <= 0.05] = 0.0
p10, p50, p90 = np.percentile(crit, [10, 50, 90])
D0 = m.DEFAULTS['delta0']
print(f"Critical decay rate across {NPOP} members (capability lognormal, sd {m.DEFAULTS['het_sd']})\n")
print(f"  p10 {p10:.4f} ({p10/D0:.2f}x)   median {p50:.4f} ({p50/D0:.2f}x)   p90 {p90:.4f} ({p90/D0:.2f}x)")
check("Ch14 critical delta0, p10",    float(p10), 0.0307, tol=0.0015)
check("Ch14 critical delta0, median", float(p50), 0.0605, tol=0.0015)
check("Ch14 critical delta0, p90",    float(p90), 0.1236, tol=0.0030)

print(f"\n{'delta0':>9}{'change':>9}{'fraction with a healthy state':>32}")
FRAC = {0: 0.507, 10: 0.448, 20: 0.383, 50: 0.238, 100: 0.111}
for pc, want in FRAC.items():
    d = D0*(1 + pc/100); got = float(np.mean(crit > d))
    print(f"{d:>9.4f}{pc:>8.0f}%{got:>32.3f}")
    check(f"Ch14 fraction healthy at {pc:+.0f}%", got, want, tol=0.02)
print("\n  -> every member has a hard edge; the edges are in different places, so the")
print("     population curve is smooth everywhere. Each member has a cliff, the room a slope.")


Critical decay rate across 2000 members (capability lognormal, sd 0.55)

  p10 0.0307 (0.51x)   median 0.0605 (1.01x)   p90 0.1236 (2.06x)
  OK  Ch14 critical delta0, p10                         0.031  (book: 0.0307)
  OK  Ch14 critical delta0, median                      0.061  (book: 0.0605)
  OK  Ch14 critical delta0, p90                         0.124  (book: 0.1236)

   delta0   change   fraction with a healthy state
   0.0600       0%                           0.507
  OK  Ch14 fraction healthy at +0%                      0.507  (book: 0.507)
   0.0660      10%                           0.448
  OK  Ch14 fraction healthy at +10%                     0.448  (book: 0.448)
   0.0720      20%                           0.383
  OK  Ch14 fraction healthy at +20%                     0.383  (book: 0.383)
   0.0900      50%                           0.238
  OK  Ch14 fraction healthy at +50%                     0.238  (book: 0.238)
   0.1200     100%                           0.111
  OK  Ch14 f

In [ ]:
# --- Ch 14: the same sweep in the full group simulation, at 400 seeds
# Originally 10 seeds. Six of the eight maintenance figures then lay outside the interval
# the larger sample gives, and the 10-seed curve was not even monotone. Raw runs in
# research/ch14_sweep.json, generated by ch14_sweep.py.
CW = _json.load(open(f'{_R}/ch14_sweep.json')).values()
CW = list(CW)
print("Full group simulation, 30 years, 400 seeds, full Tradition adherence\n")
print(f"{'change':>8}{'survival':>10}{'95% CI':>18}{'maintenance':>13}{'+/-':>8}")
WANT = {-30: (1.000, 0.3299), -20: (1.000, 0.2563), -10: (1.000, 0.1883),
          0: (0.995, 0.1158),   5: (1.000, 0.0881),  10: (0.995, 0.0651),
         30: (0.970, 0.0220),  50: (0.927, 0.0096)}
prev = None
for pc, (ws, wm) in WANT.items():
    r = [v for v in CW if v['pc'] == pc]; n = len(r); k = sum(v['alive'] for v in r)
    mt = np.array([v['maint'] for v in r if v['alive']], float)
    lo, hi = wilson(k, n); se = mt.std(ddof=1)/np.sqrt(len(mt))
    print(f"{pc:>7}%{k/n:>10.3f}   [{lo:.3f}, {hi:.3f}]{mt.mean():>13.4f}{1.96*se:>8.4f}")
    check(f"Ch14 group survival at {pc:+.0f}%",    k/n,       ws, tol=0.006)
    check(f"Ch14 group maintenance at {pc:+.0f}%", mt.mean(), wm, tol=0.004)
    if prev is not None: assert mt.mean() < prev + 1e-9, f"maintenance not monotone at {pc}%"
    prev = mt.mean()
print("\n  -> survival stays above 0.92 across the whole range and maintenance declines")
print("     smoothly and MONOTONICALLY from 0.330 to 0.010. At 10 seeds the curve had two")
print("     reversals that were pure sampling noise; at 400 it is clean.")
print("     The group has no threshold even though every member in it does.")


Full group simulation, 30 years, 400 seeds, full Tradition adherence

  change  survival            95% CI  maintenance     +/-
    -30%     1.000   [0.990, 1.000]       0.3299  0.0061
  OK  Ch14 group survival at -30%                       1.000  (book: 1.0)
  OK  Ch14 group maintenance at -30%                    0.330  (book: 0.3299)
    -20%     1.000   [0.990, 1.000]       0.2563  0.0059
  OK  Ch14 group survival at -20%                       1.000  (book: 1.0)
  OK  Ch14 group maintenance at -20%                    0.256  (book: 0.2563)
    -10%     1.000   [0.990, 1.000]       0.1883  0.0063
  OK  Ch14 group survival at -10%                       1.000  (book: 1.0)
  OK  Ch14 group maintenance at -10%                    0.188  (book: 0.1883)
      0%     0.995   [0.982, 0.999]       0.1158  0.0062
  OK  Ch14 group survival at +0%                        0.995  (book: 0.995)
  OK  Ch14 group maintenance at +0%                     0.116  (book: 0.1158)
      5%     1.000   [0.990, 1

---
## 9. Targeted sensitivity over all 118 parameters
### Preface, `research/PARAMETERS.md`, and Chapters 1, 4, 6 and 14

The earlier one-at-a-time sweep covered **21 of the 118** hand-chosen numbers: the
continuous scalars, minus room capacity, scored on a single outcome. Ninety-seven
parameters had never been varied alone, including every step speed and every cell of both
matrices.

`sensitivity_oat_full.py` varies each of the 118 alone by plus and minus 25 per cent and
scores three scenarios (full adherence, attraction lost, referrals lost) on four outcomes.
That is 236 perturbations. Raw results in `research/oat_full.json`.

Three results matter and one of them is algebra rather than simulation.

In [ ]:
# --- Ch: the full one-at-a-time sweep, read from the saved raw results
import json as _json, os as _os
_R = '../research' if _os.path.isdir('../research') else 'research'
OAT = _json.load(open(f'{_R}/oat_full.json'))
BASE = OAT['baseline']
PARAM = {k: v for k, v in OAT.items() if k != 'baseline'}
KINDS = ('scalar', 'a', 'S', 'GOV')

print(f"{len(PARAM)} parameters, 2 directions each = {2*len(PARAM)} perturbations\n")
print(f"{'kind':<8}{'count':>7}   what they are")
for k, what in zip(KINDS, ['continuous scalars, cap now included',
                           'step speeds, one per step',
                           'non-zero cells of the step-consumption matrix',
                           'non-zero cells of the tradition-governance matrix']):
    print(f"{k:<8}{sum(1 for p in PARAM if p.startswith(k+':')):>7}   {what}")
check("parameters in the full sweep", float(len(PARAM)), 118.0, tol=0.5)
check("scalars swept (cap included)", float(sum(1 for p in PARAM if p.startswith('scalar:'))), 22.0, tol=0.5)

print(f"\nBaseline, 3 seeds, 30 years:")
for s, v in BASE.items():
    print(f"  {s:<11}survival {v['surv']:.2f}   N {v['N']:6.1f}   practice {v['practice']:.3f}   maintenance {v['maint']:.4f}")
check("baseline full-adherence N",           BASE['full']['N'],        47.3,   tol=0.2)
check("baseline full-adherence maintenance", BASE['full']['maint'],     0.1458, tol=0.002)
check("baseline referral-starved survival",  BASE['referral']['surv'],  0.00,  tol=0.01)


118 parameters, 2 directions each = 236 perturbations

kind      count   what they are
scalar       22   continuous scalars, cap now included
a            12   step speeds, one per step
S            49   non-zero cells of the step-consumption matrix
GOV          35   non-zero cells of the tradition-governance matrix
  OK  parameters in the full sweep                    118.000  (book: 118.0)
  OK  scalars swept (cap included)                     22.000  (book: 22.0)

Baseline, 3 seeds, 30 years:
  full       survival 1.00   N   47.3   practice 0.342   maintenance 0.1458
  attraction survival 1.00   N   15.0   practice 0.232   maintenance 0.0527
  referral   survival 0.00   N    0.0   practice 0.000   maintenance 0.0000
  OK  baseline full-adherence N                        47.333  (book: 47.3)
  OK  baseline full-adherence maintenance               0.146  (book: 0.1458)
  OK  baseline referral-starved survival                0.000  (book: 0.0)


In [ ]:
# --- result 1: the ordering claim survives every single-parameter perturbation
viol = [(p, t) for p, r in PARAM.items() for t in ('low', 'high')
        if r[t]['referral']['surv'] > r[t]['attraction']['surv']]
print(f"Ordering claim (losing referrals is at least as bad as losing attraction)")
print(f"  violations out of {2*len(PARAM)} perturbations: {len(viol)}")
check("ordering violations in the full OAT", float(len(viol)), 0.0, tol=0.01)
print("  -> holds under every one of the 118 parameters moved alone, in both directions.")
print("     Previously this was known to hold under GLOBAL perturbation (100/100/87 per")
print("     cent). It now also holds under targeted perturbation, which is a different")
print("     and stronger test: no single number in the model carries it.\n")

# --- result 2: full-adherence persistence has exactly one weak point
bad = [(p, t, r[t]['full']['surv']) for p, r in PARAM.items() for t in ('low', 'high')
       if r[t]['full']['surv'] < 1.0]
print("Full-adherence persistence, scored on every seed")
print(f"  perturbations where a fully adherent group failed in any seed: {len(bad)}")
for p, t, s in bad: print(f"     {p} at {'-' if t=='low' else '+'}25%  ->  survival {s:.2f}")
check("perturbations breaking full adherence", float(len(bad)), 1.0, tol=0.01)
print("  -> 235 of 236 leave it untouched. The sole exception is member heterogeneity,")
print("     which PARAMETERS.md already names as the least defensible number in the model.\n")

# --- result 3: how many parameters can rescue a referral-starved group
full = sorted(p for p, r in PARAM.items()
              if max(r['low']['referral']['surv'], r['high']['referral']['surv']) == 1.0)
swing1 = sorted(p for p, r in PARAM.items()
                if abs(r['high']['referral']['surv'] - r['low']['referral']['surv']) == 1.0)
part = [p for p, r in PARAM.items()
        if 0 < max(r['low']['referral']['surv'], r['high']['referral']['surv']) < 1.0]
print("Referral-starved survival, baseline 0.00 (the group dies in all three seeds)")
print(f"  full 0-to-1 swing from one parameter alone: {len(swing1)} -> {swing1}")
print(f"  reach certain survival at one end:          {len(full)} -> {full}")
print(f"  move it off zero at all:                    {len(full)+len(part)} of 118")
print(f"  no effect whatever:                         {118-len(full)-len(part)} of 118")
check("parameters with a full 0-to-1 swing", float(len(swing1)), 3.0, tol=0.01)
check("parameters reaching certain survival", float(len(full)), 4.0, tol=0.01)
check("parameters moving it off zero", float(len(full)+len(part)), 53.0, tol=0.01)
print("  -> the preface says THREE parameters flip this outcome. That is right on the")
print("     strictest reading and understates the problem: 53 of 118 move it off the")
print("     floor. Referral-starved survival is not a quantity this model estimates.")


Ordering claim (losing referrals is at least as bad as losing attraction)
  violations out of 236 perturbations: 0
  OK  ordering violations in the full OAT               0.000  (book: 0.0)
  -> holds under every one of the 118 parameters moved alone, in both directions.
     Previously this was known to hold under GLOBAL perturbation (100/100/87 per
     cent). It now also holds under targeted perturbation, which is a different
     and stronger test: no single number in the model carries it.

Full-adherence persistence, scored on every seed
  perturbations where a fully adherent group failed in any seed: 1
     scalar:het_sd at -25%  ->  survival 0.67
  OK  perturbations breaking full adherence             1.000  (book: 1.0)
  -> 235 of 236 leave it untouched. The sole exception is member heterogeneity,
     which PARAMETERS.md already names as the least defensible number in the model.

Referral-starved survival, baseline 0.00 (the group dies in all three seeds)
  full 0-to-1 swing f

In [ ]:
# --- result 4: the governance matrix cannot affect a fully adherent group. This is
#     algebra, not a simulation finding, and it is worth stating as such.
Te = m.effective_adherence(m.FULL)
q  = m.GOVW.T @ Te
print("At full adherence, effective adherence of every tradition is", np.unique(np.round(Te, 12)))
print("Governance quality q for the eight resources is", np.unique(np.round(q, 12)))
check("q is identically 1 at full adherence", float(q.min()), 1.0, tol=1e-9)
check("q is identically 1 at full adherence (max)", float(q.max()), 1.0, tol=1e-9)
print("\n  GOVW is column-normalised, so q = GOVW.T @ 1 = 1 for every resource whatever")
print("  the GOV entries are. The 35 governance numbers are exactly cancelled.\n")

bn = BASE['full']['N']
print(f"{'kind':<8}{'n':>4}{'max swing in N':>16}{'median':>9}{'exactly zero':>14}")
for k in KINDS:
    sw = [abs(r['high']['full']['N'] - r['low']['full']['N'])/bn
          for p, r in PARAM.items() if p.startswith(k+':')]
    print(f"{k:<8}{len(sw):>4}{max(sw):>16.3f}{float(np.median(sw)):>9.3f}"
          f"{sum(1 for s in sw if s == 0):>14}")
govzero = sum(1 for p, r in PARAM.items() if p.startswith('GOV:')
              and r['high']['full']['N'] == r['low']['full']['N'])
check("governance cells with exactly zero effect", float(govzero), 35.0, tol=0.01)
print("\n  -> all 35 governance cells are exactly inert here, and under the attraction")
print("     scenario, where the cancellation does not apply, a 25 per cent change in one")
print("     cell moves the column-normalised quality by about one per cent and moves no")
print("     outcome at all. The preface says the model rests on the matrices' STRUCTURE")
print("     rather than their magnitudes. For the governance matrix that is now provable.")


At full adherence, effective adherence of every tradition is [1.]
Governance quality q for the eight resources is [1.]
  OK  q is identically 1 at full adherence              1.000  (book: 1.0)
  OK  q is identically 1 at full adherence (max)        1.000  (book: 1.0)

  GOVW is column-normalised, so q = GOVW.T @ 1 = 1 for every resource whatever
  the GOV entries are. The 35 governance numbers are exactly cancelled.

kind       n  max swing in N   median  exactly zero
scalar    22           1.021    0.218             1
a         12           0.697    0.194             0
S         49           0.585    0.148             1
GOV       35           0.000    0.000            35
  OK  governance cells with exactly zero effect        35.000  (book: 35.0)

  -> all 35 governance cells are exactly inert here, and under the attraction
     scenario, where the cancellation does not apply, a 25 per cent change in one
     cell moves the column-normalised quality by about one per cent and moves no


In [ ]:
# --- result 5: what the sweep says about Chapter 14's maintenance result
bm = BASE['full']['maint']
mv = sorted(((p, abs(r['high']['full']['maint'] - r['low']['full']['maint'])/bm)
             for p, r in PARAM.items()), key=lambda x: -x[1])
print(f"Group maintenance at baseline {bm:.4f}. Largest movers, as multiples of baseline:\n")
print(f"{'parameter':<16}{'-25%':>9}{'+25%':>9}{'swing/base':>12}")
for p, s in mv[:6]:
    print(f"{p:<16}{PARAM[p]['low']['full']['maint']:>9.4f}"
          f"{PARAM[p]['high']['full']['maint']:>9.4f}{s:>12.2f}")
check("p_gate swing in maintenance",  mv[0][1], 2.98, tol=0.02)
check("delta0 swing in maintenance",  next(s for p, s in mv if p == 'scalar:delta0'), 1.83, tol=0.02)
check("het_sd swing in maintenance",  next(s for p, s in mv if p == 'scalar:het_sd'), 1.69, tol=0.02)
zero = [p for p, r in PARAM.items()
        if r['low']['full']['maint'] == 0.0 or r['high']['full']['maint'] == 0.0]
check("params driving group maintenance to zero", float(len(zero)), 0.0, tol=0.01)
print(f"\n  parameters that drive GROUP maintenance to exactly zero: {len(zero)}")
print("  -> Chapter 14's individual threshold is knife-edge in the decay rate: 2.79 per")
print("     cent kills the healthy state for a typical member. At group level nothing")
print("     goes to zero under any single parameter moved by a quarter. That is the")
print("     same individual-versus-group distinction, arrived at a second way.")
print("\n  The step-ordering exponent p_gate is the strongest single influence on")
print("  maintenance, swinging it by three times its baseline. Chapter 13 makes the sign")
print("  of the related substitution parameter its whole subject; this says the magnitude")
print("  matters at least as much, and nobody has measured either.")


Group maintenance at baseline 0.1458. Largest movers, as multiples of baseline:

parameter            -25%     +25%  swing/base
scalar:p_gate      0.4386   0.0045        2.98
scalar:delta0      0.3305   0.0645        1.83
scalar:het_sd      0.0032   0.2493        1.69
a:9                0.0712   0.2121        0.97
S:0,6              0.1799   0.0538        0.86
scalar:hill_n      0.1814   0.0647        0.80
  OK  p_gate swing in maintenance                       2.978  (book: 2.98)
  OK  delta0 swing in maintenance                       1.825  (book: 1.83)
  OK  het_sd swing in maintenance                       1.688  (book: 1.69)
  OK  params driving group maintenance to zero          0.000  (book: 0.0)

  parameters that drive GROUP maintenance to exactly zero: 0
  -> Chapter 14's individual threshold is knife-edge in the decay rate: 2.79 per
     cent kills the healthy state for a typical member. At group level nothing
     goes to zero under any single parameter moved by a quarter. 

---
## 10. The apparatus itself
### Chapter 12

Chapter 12 owns the description of the model: the twelve dials, the eight group resources,
the depreciation structure and the derived group-dependence coefficients. Most of its
figures are asserted in sections 5 and 9 already; this section adds the ones that are only
stated there, and re-asserts the parameter inventory in one place because the preface, the
plans and PARAMETERS.md all quote it and it has drifted twice.

In [ ]:
# --- Ch 12: the apparatus, its inventory, and the derived coefficients
P = m.DEFAULTS
half = np.log(2)/P['delta0']
print(f"depreciation delta0 = {P['delta0']} per week  ->  unattended half-life {half:.1f} weeks")
print(f"backward complementarity psi = {P['psi']}")
print(f"ordering exponent p_gate     = {P['p_gate']}")
print(f"top step speeds a: min {P['a'].min():.2f} (Step {int(P['a'].argmin())+1}), "
      f"max {P['a'].max():.2f} (Step {int(P['a'].argmax())+1})")
check("Ch12 unattended half-life, weeks", float(half), 11.6,  tol=0.06)
check("Ch12 delta0",                      float(P['delta0']), 0.06, tol=1e-9)
check("Ch12 psi",                         float(P['psi']),    0.20, tol=1e-9)
check("Ch12 p_gate",                      float(P['p_gate']), 1.5,  tol=1e-9)
check("Ch12 slowest step speed",          float(P['a'].min()), 0.15, tol=1e-9)
check("Ch12 fastest step speed",          float(P['a'].max()), 0.30, tol=1e-9)

print(f"\neight group resources: {m.RES}")
check("Ch12 number of group resources", float(len(m.RES)), 8.0, tol=0.01)

print("\nderived group-dependence, beta = row sum of S, normalised by the largest")
steps = ["1 admit","2 believe","3 decide","4 inventory","5 tell someone","6 willing",
         "7 ask","8 list harms","9 amends","10 daily","11 connect","12 carry it"]
for s, b in zip(steps, m.BETA): print(f"   {s:<16}{b:.2f}")
check("Ch12 beta max (Steps 1 and 12)", float(m.BETA.max()),  1.00, tol=0.005)
check("Ch12 beta min (Step 7)",         float(m.BETA.min()),  0.17, tol=0.005)
check("Ch12 beta mean",                 float(m.BETA.mean()), 0.53, tol=0.005)
check("Ch12 ratio of extremes",         float(m.BETA.max()/m.BETA.min()), 6.0, tol=0.05)
top = [i for i in range(12) if m.BETA[i] > m.BETA.max() - 1e-9]
assert top == [0, 11], f"the maximally group-dependent steps are no longer 1 and 12: {top}"
assert int(m.BETA.argmin()) == 6, "least social step is no longer Step 7"
print("\n  -> the two steps needing a group most are the first and the last; the least")
print("     is Step 7. That falls out of the resource assignment, it was not chosen.")

print("\nparameter inventory")
nsc = sum(1 for v in P.values() if np.isscalar(v))
inv = dict(scalars=nsc, step_speeds=len(P['a']),
           S_nonzero=int((m.S != 0).sum()), GOV_nonzero=int((m.GOV != 0).sum()))
inv['chosen'] = sum(inv.values())
inv['cells']  = nsc + len(P['a']) + m.S.size + m.GOV.size
for k, v in inv.items(): print(f"   {k:<14}{v:>5}")
check("Ch12 continuous scalars",   float(inv['scalars']),     22.0, tol=0.01)
check("Ch12 step speeds",          float(inv['step_speeds']), 12.0, tol=0.01)
check("Ch12 S non-zero cells",     float(inv['S_nonzero']),   49.0, tol=0.01)
check("Ch12 GOV non-zero cells",   float(inv['GOV_nonzero']), 35.0, tol=0.01)
check("Ch12 chosen by hand",       float(inv['chosen']),     118.0, tol=0.01)
check("Ch12 total cells",          float(inv['cells']),      226.0, tol=0.01)
print("\n  -> 118 chosen by hand out of 226 cells. Quoted in the preface, both plans and")
print("     PARAMETERS.md; it has drifted twice, so it is asserted here in one place.")


depreciation delta0 = 0.06 per week  ->  unattended half-life 11.6 weeks
backward complementarity psi = 0.2
ordering exponent p_gate     = 1.5
top step speeds a: min 0.15 (Step 9), max 0.30 (Step 1)
  OK  Ch12 unattended half-life, weeks                 11.552  (book: 11.6)
  OK  Ch12 delta0                                       0.060  (book: 0.06)
  OK  Ch12 psi                                          0.200  (book: 0.2)
  OK  Ch12 p_gate                                       1.500  (book: 1.5)
  OK  Ch12 slowest step speed                           0.150  (book: 0.15)
  OK  Ch12 fastest step speed                           0.300  (book: 0.3)

eight group resources: ['admission', 'identify', 'proof', 'confidential', 'counsel', 'recipient', 'continuity', 'pressure']
  OK  Ch12 number of group resources                    8.000  (book: 8.0)

derived group-dependence, beta = row sum of S, normalised by the largest
   1 admit         1.00
   2 believe       0.71
   3 decide        0.29
  

---
## 11. Index-pairing, and the robustness of the two matrices
### Chapter 16 (undrafted), and `research/PARAMETERS.md` section 2

The book's third headline claim is that the natural assumption pairing Step *i* with
Tradition *i* is wrong on all twelve counts. Part Four is built on it and Chapter 16 is
entirely about it.

`PARAMETERS.md` cited an 85.5 per cent robustness figure for this that could not be
reproduced from anything in the book's own repository. The code turned out to live in
`paper/anonymity-as-an-aggregation-condition.ipynb` and nowhere else. It is ported here so
the claim stands on the book's own files.

The coupling between steps and traditions is B = S G', where S records what each step
consumes and G what each tradition governs. The principal supplier of a step is the
tradition with the largest entry in that step's row. Robustness is tested by multiplying
every entry of both matrices by a uniform factor in [0.7, 1.3], 2,000 draws.

In [ ]:
# --- Ch 16: the derived step-to-tradition coupling, and its robustness
B = m.S @ m.GOV.T
TR = [f"T{j+1}" for j in range(12)]
STEPS = ["1 admit","2 believe","3 decide","4 inventory","5 tell someone","6 willing",
         "7 ask","8 list harms","9 amends","10 daily","11 connect","12 carry it"]
print("Principal supplying tradition for each step, unperturbed\n")
print(f"{'step':<16}{'principal':>10}{'index-pairing says':>20}")
hold = []
for i, s in enumerate(STEPS):
    j = int(np.argmax(B[i]))
    if j == i: hold.append(i+1)
    print(f"{s:<16}{TR[j]:>10}{TR[i]:>20}")
print(f"\n  steps where index-pairing holds: {hold}")
check("steps where index-pairing holds", float(len(hold)), 0.0, tol=0.01)
assert int(np.argmax(B[4])) == 11, "Step 5's principal supplier is no longer T12"
assert int(np.argmax(B[11])) == 4, "Step 12's principal supplier is no longer T5"
print("  -> Step 5 is supplied mainly by T12 (anonymity, via confidentiality) and Step 12")
print("     by T5 (single purpose). The two most quotable pairings are inverted.\n")

load = B.sum(0)
print("Load per tradition (column sums of B), largest first")
for j in np.argsort(-load)[:6]:
    print(f"   {TR[j]:<5}{load[j]:>7.2f}")
check("T1 is the most load-bearing tradition", float(np.argmax(load)), 0.0, tol=0.01)
check("T1 load", float(load.max()), 6.52, tol=0.01)

# --- robustness: 2,000 draws, every entry of both matrices scaled by U[0.7, 1.3]
rng = np.random.default_rng(3)
N = 2000
t1 = dw = s12 = s5 = 0
for _ in range(N):
    Bp = (m.S*rng.uniform(.7, 1.3, m.S.shape)) @ (m.GOV*rng.uniform(.7, 1.3, m.GOV.shape)).T
    if int(np.argmax(Bp.sum(0))) == 0: t1 += 1
    if all(int(np.argmax(Bp[i])) != i for i in range(12)): dw += 1
    if int(np.argmax(Bp[11])) == 4: s12 += 1
    if int(np.argmax(Bp[4])) == 11: s5 += 1
print(f"\nRobustness over {N} draws at plus or minus 30 per cent on every entry\n")
for lab, c, want in [("T1 unity is the most load-bearing", t1, 100.0),
                     ("index-pairing wrong for ALL twelve steps", dw, 85.5),
                     ("Step 5's principal supplier is T12", s5, 99.5),
                     ("Step 12's principal supplier is T5", s12, 67.3)]:
    got = 100*c/N
    print(f"   {lab:<42}{got:>6.1f}%")
    check(lab, got, want, tol=0.1)

protective = [j for j in range(12) if m.GOV[j].sum() == 0]
print(f"\n   two-tier split (traditions governing no consumed resource): "
      f"{[TR[j] for j in protective]}")
check("protective traditions", float(len(protective)), 5.0, tol=0.01)
print("   This survives 100 per cent of draws TRIVIALLY: the perturbation is")
print("   multiplicative, so a zero row stays a zero row. It is a statement about the")
print("   derivation, not a robustness result, and should not be quoted as one.")
print("\n   The 67.3 per cent on Step 12 is the weak entry. T3 is a close competitor,")
print("   since both T5 and T3 govern the recipient resource. Chapter 16 should report")
print("   the inversion of Step 5 (99.5%) confidently and Step 12 (67.3%) with care.")

# --- degradation curve, and the structural test the survival claims already pass
print("\n\nDegradation curve. Multiplicative jitter at four levels, then STRUCTURAL")
print("randomisation which replaces every non-zero entry and keeps only the sparsity.\n")
def coupling_test(fn, seed, N=2000):
    rg = np.random.default_rng(seed); t1 = dw = s5 = s12 = 0
    for _ in range(N):
        Sp, Gp = fn(rg); Bp = Sp @ Gp.T
        if int(np.argmax(Bp.sum(0))) == 0: t1 += 1
        if all(int(np.argmax(Bp[i])) != i for i in range(12)): dw += 1
        if int(np.argmax(Bp[4]))  == 11: s5 += 1
        if int(np.argmax(Bp[11])) == 4:  s12 += 1
    return [100*x/N for x in (t1, dw, s5, s12)]

print(f"{'design':<28}{'T1 top':>9}{'all 12':>9}{'S5->T12':>10}{'S12->T5':>10}")
CURVE = {}
for lvl, want in [(0.15, [100.0, 98.8, 100.0, 89.7]), (0.30, [100.0, 85.5, 99.5, 67.3]),
                  (0.50, [98.0, 72.5, 88.7, 52.9]),  (0.75, [86.8, 65.2, 71.5, 43.8])]:
    f = lambda rg, l=lvl: (m.S*rg.uniform(1-l, 1+l, m.S.shape), m.GOV*rg.uniform(1-l, 1+l, m.GOV.shape))
    r = CURVE[lvl] = coupling_test(f, 3)
    print(f"{'jitter +/-'+str(int(lvl*100))+'%':<28}" + "".join(f"{v:>9.1f}%" for v in r))
    for got, w, lab in zip(r, want, ['T1', 'all12', 'S5', 'S12']):
        check(f"Ch16 {lab} at jitter {int(lvl*100)}%", got, w, tol=0.15)

fstruct = lambda rg: (np.where(m.S > 0, rg.uniform(.05, 1., m.S.shape), 0.),
                      np.where(m.GOV > 0, rg.uniform(.05, 1., m.GOV.shape), 0.))
ST = coupling_test(fstruct, 23)
print(f"{'structural (sparsity only)':<28}" + "".join(f"{v:>9.1f}%" for v in ST))
for got, w, lab in zip(ST, [75.4, 40.6, 17.5, 27.6], ['T1', 'all12', 'S5', 'S12']):
    check(f"Ch16 {lab}, structural", got, w, tol=0.15)

print("\n  -> THIS CHANGES WHAT PART FOUR MAY CLAIM. The coupling results are NOT")
print("     structural. They degrade smoothly with disagreement about the magnitudes and")
print("     are essentially gone once only the sparsity pattern is kept: index-pairing")
print("     fails on all twelve in 40.6 per cent of structurally randomised draws, which")
print("     is less often than it holds, and the Step 5 inversion survives in 17.5.")
print("     The SURVIVAL claims pass the same structural test at 100 per cent (section 7).")
print("     So 'the model rests on structure, not magnitudes' is true of Parts One and")
print("     Three and FALSE of Part Four. The two must not be defended the same way.")
print("     Defensible statement: index-pairing fails robustly to modest disagreement")
print("     about the magnitudes, and a reader who thinks the matrices are arbitrary")
print("     should not be persuaded by it.")


Principal supplying tradition for each step, unperturbed

step             principal  index-pairing says
1 admit                 T3                  T1
2 believe              T11                  T2
3 decide                T2                  T3
4 inventory             T2                  T4
5 tell someone         T12                  T5
6 willing               T1                  T6
7 ask                   T1                  T7
8 list harms            T2                  T8
9 amends                T2                  T9
10 daily                T1                 T10
11 connect              T1                 T11
12 carry it             T5                 T12

  steps where index-pairing holds: []
  OK  steps where index-pairing holds                   0.000  (book: 0.0)
  -> Step 5 is supplied mainly by T12 (anonymity, via confidentiality) and Step 12
     by T5 (single purpose). The two most quotable pairings are inverted.

Load per tradition (column sums of B), largest first
   T1 

---
### 11b. How index-pairing fails, step by step, and how much of it is arithmetic
#### Chapter 16

Section 11 establishes that no Step's principal supplier is its own index-mate. This cell
establishes three further things the chapter needs and did not have:

1. **The per-step table**, exact, with each Step's principal supplier, its runner-up, its
   own index-mate and where that index-mate ranks. Step 1 is the only near miss.
2. **The trivial-count decomposition.** Five Traditions govern nothing any Step consumes,
   so Steps 4, 6, 7, 9 and 10 have index-mate entries of exactly zero and index-pairing
   cannot hold for them under any sparsity-preserving perturbation. The check below shows
   that the proportion of draws in which index-pairing fails on all twelve is identical, to
   the last draw, to the proportion in which it fails on the seven that could have gone
   either way. **The headline "all twelve" is seven findings and one structural fact.**
3. **Wilson intervals** on every Monte Carlo proportion Chapter 16 prints, at n = 2,000.

The exact figures carry no sampling error and are asserted as exact. The proportions are
Monte Carlo and are asserted with their intervals.

In [ ]:
# --- Ch 16: the per-step coupling table, the trivial-count decomposition, and intervals
# Added when Chapter 16 was drafted. Section 11 above establishes that index-pairing fails
# on all twelve; this cell establishes HOW it fails, step by step, and how much of the
# headline "all twelve" is arithmetic rather than evidence.
B = m.S @ m.GOV.T
TR = [f"T{j+1}" for j in range(12)]
STEPS = ["1 admit","2 believe","3 decide","4 inventory","5 tell someone","6 willing",
         "7 ask","8 list harms","9 amends","10 daily","11 connect","12 carry it"]

print("Per-step coupling, EXACT (matrix product of two fixed matrices, no sampling error)\n")
print(f"{'step':<16}{'1st':>5}{'val':>7}{'2nd':>6}{'val':>7}{'own T':>7}{'val':>7}{'rank':>6}")
TABLE = {}
for i, s in enumerate(STEPS):
    row = B[i]; o = np.argsort(-row)
    rank = int(np.where(o == i)[0][0]) + 1
    TABLE[i] = (TR[o[0]], row[o[0]], TR[o[1]], row[o[1]], row[i], rank)
    print(f"{s:<16}{TR[o[0]]:>5}{row[o[0]]:>7.2f}{TR[o[1]]:>6}{row[o[1]]:>7.2f}"
          f"{TR[i]:>7}{row[i]:>7.2f}{rank:>6}")

for i, (p1, v1, p2, v2, own, rk) in TABLE.items():
    check(f"Ch16 step {i+1} principal value", float(v1), round(float(v1), 2), tol=0.005)
    check(f"Ch16 step {i+1} runner-up value", float(v2), round(float(v2), 2), tol=0.005)
    check(f"Ch16 step {i+1} index-mate rank", float(rk), float(rk), tol=0.01)

# The chapter quotes these explicitly.
for lab, got, want in [("step 1 principal", float(B[0].max()), 1.22),
                       ("step 1 own T1", float(B[0][0]), 0.99),
                       ("step 2 principal", float(B[1].max()), 0.82),
                       ("step 2 runner-up T5", float(B[1][4]), 0.72),
                       ("step 5 principal", float(B[4].max()), 1.08),
                       ("step 5 own T5", float(B[4][4]), 0.12),
                       ("step 12 principal", float(B[11].max()), 1.25),
                       ("step 12 runner-up T3", float(B[11][2]), 1.10),
                       ("step 12 own T12", float(B[11][11]), 0.12),
                       ("step 3 own T3", float(B[2][2]), 0.01),
                       ("step 8 own T8", float(B[7][7]), 0.07),
                       ("step 11 own T11", float(B[10][10]), 0.11),
                       ("T8 load", float(B.sum(0)[7]), 1.11),
                       ("step 4 top-two margin", float(np.sort(B[3])[-1]-np.sort(B[3])[-2]), 0.00),
                       ("step 6 top-two margin", float(np.sort(B[5])[-1]-np.sort(B[5])[-2]), 0.01),
                       ("step 3 top-two margin", float(np.sort(B[2])[-1]-np.sort(B[2])[-2]), 0.04),
                       ("step 7 top-two margin", float(np.sort(B[6])[-1]-np.sort(B[6])[-2]), 0.04),
                       ("step 12 top-two margin", float(np.sort(B[11])[-1]-np.sort(B[11])[-2]), 0.15),
                       ("step 5 top-two margin", float(np.sort(B[4])[-1]-np.sort(B[4])[-2]), 0.43),
                       ("step 1 top-two margin", float(np.sort(B[0])[-1]-np.sort(B[0])[-2]), 0.23),
                       ("T5 governs recipient", float(m.GOV[4][5]), 0.9),
                       ("T3 governs recipient", float(m.GOV[2][5]), 0.8)]:
    check(f"Ch16 {lab}", got, want, tol=0.006)

# --- how much of "all twelve" is arithmetic
TRIV = [i for i in range(12) if m.GOV[i].sum() == 0]
NONT = [i for i in range(12) if i not in TRIV]
print(f"\nIndex-mate governs nothing, so the count is trivial: steps {[i+1 for i in TRIV]}")
print(f"Counts that could have gone either way:            steps {[i+1 for i in NONT]}")
check("Ch16 trivial counts", float(len(TRIV)), 5.0, tol=0.01)
check("Ch16 non-trivial counts", float(len(NONT)), 7.0, tol=0.01)

def both(fn, seed, N=2000):
    rg = np.random.default_rng(seed); a12 = a7 = 0
    for _ in range(N):
        Sp, Gp = fn(rg); Bp = Sp @ Gp.T
        if all(int(np.argmax(Bp[i])) != i for i in range(12)): a12 += 1
        if all(int(np.argmax(Bp[i])) != i for i in NONT):      a7 += 1
    return 100*a12/N, 100*a7/N

print("\nThe five trivial counts contribute nothing: 'all twelve' equals 'all seven'\n")
print(f"{'design':<28}{'all 12':>9}{'all 7':>9}")
for lvl, want in [(0.15, 98.75), (0.30, 85.50), (0.50, 72.45), (0.75, 65.15)]:
    f = lambda rg, l=lvl: (m.S*rg.uniform(1-l, 1+l, m.S.shape),
                           m.GOV*rg.uniform(1-l, 1+l, m.GOV.shape))
    a12, a7 = both(f, 3)
    print(f"{'jitter +/-'+str(int(lvl*100))+'%':<28}{a12:>8.2f}%{a7:>8.2f}%")
    check(f"Ch16 all12 == all7 at {int(lvl*100)}%", a12 - a7, 0.0, tol=1e-9)
    check(f"Ch16 all7 at jitter {int(lvl*100)}%", a7, want, tol=0.15)
fs = lambda rg: (np.where(m.S > 0, rg.uniform(.05, 1., m.S.shape), 0.),
                 np.where(m.GOV > 0, rg.uniform(.05, 1., m.GOV.shape), 0.))
a12, a7 = both(fs, 23)
print(f"{'structural (sparsity only)':<28}{a12:>8.2f}%{a7:>8.2f}%")
check("Ch16 all12 == all7 structural", a12 - a7, 0.0, tol=1e-9)
check("Ch16 all7 structural", a7, 40.60, tol=0.15)

# --- Wilson 95% intervals on every Monte Carlo proportion Chapter 16 prints
print("\nWilson 95 per cent intervals, n = 2,000 draws\n")
CH16CI = [("jitter 15 T1", 100.00), ("jitter 15 all", 98.75), ("jitter 15 S5", 100.00), ("jitter 15 S12", 89.70),
          ("jitter 30 T1", 100.00), ("jitter 30 all", 85.50), ("jitter 30 S5", 99.50), ("jitter 30 S12", 67.35),
          ("jitter 50 T1", 98.00), ("jitter 50 all", 72.45), ("jitter 50 S5", 88.65), ("jitter 50 S12", 52.90),
          ("jitter 75 T1", 86.85), ("jitter 75 all", 65.15), ("jitter 75 S5", 71.50), ("jitter 75 S12", 43.75),
          ("struct T1", 75.40), ("struct all", 40.60), ("struct S5", 17.50), ("struct S12", 27.65)]
CI16 = {}
for lab, pct in CH16CI:
    lo, hi = wilson(int(round(pct/100*2000)), 2000)
    CI16[lab] = (100*lo, 100*hi)
    print(f"   {lab:<16}{pct:>7.2f}   [{100*lo:>5.1f}, {100*hi:>5.1f}]")
for lab, wlo, whi in [("jitter 15 all", 98.2, 99.2), ("jitter 15 S12", 88.3, 91.0),
                      ("jitter 30 all", 83.9, 87.0), ("jitter 30 S5", 99.1, 99.7),
                      ("jitter 30 S12", 65.3, 69.4), ("jitter 50 T1", 97.3, 98.5),
                      ("jitter 50 all", 70.5, 74.4), ("jitter 50 S5", 87.2, 90.0),
                      ("jitter 50 S12", 50.7, 55.1), ("jitter 75 T1", 85.3, 88.3),
                      ("jitter 75 all", 63.0, 67.2), ("jitter 75 S5", 69.5, 73.4),
                      ("jitter 75 S12", 41.6, 45.9), ("struct T1", 73.5, 77.2),
                      ("struct all", 38.5, 42.8), ("struct S5", 15.9, 19.2),
                      ("struct S12", 25.7, 29.7), ("jitter 15 T1", 99.8, 100.0),
                      ("jitter 30 T1", 99.8, 100.0), ("jitter 15 S5", 99.8, 100.0)]:
    check(f"Ch16 CI lo {lab}", CI16[lab][0], wlo, tol=0.05)
    check(f"Ch16 CI hi {lab}", CI16[lab][1], whi, tol=0.05)
print("\n  -> Chapter 16 quotes the exact table as exact and every proportion with its")
print("     interval and its sample size. The five trivial counts are named as trivial.")


Per-step coupling, EXACT (matrix product of two fixed matrices, no sampling error)

step              1st    val   2nd    val  own T    val  rank
1 admit            T3   1.22    T1   0.99     T1   0.99     2
2 believe         T11   0.82    T5   0.72     T2   0.13     7
3 decide           T2   0.31    T1   0.27     T3   0.01     7
4 inventory        T2   0.35    T1   0.35     T4   0.00     8
5 tell someone    T12   1.08    T1   0.65     T5   0.12     5
6 willing          T1   0.25    T2   0.24     T6   0.00     9
7 ask              T1   0.17    T2   0.13     T7   0.00    10
8 list harms       T2   0.43    T1   0.29     T8   0.07     4
9 amends           T2   0.90    T1   0.48     T9   0.00    11
10 daily           T1   0.94    T2   0.43    T10   0.00    12
11 connect         T1   0.47    T2   0.21    T11   0.11     4
12 carry it        T5   1.25    T3   1.10    T12   0.12     6
  OK  Ch16 step 1 principal value                       1.220  (book: 1.22)
  OK  Ch16 step 1 runner-up value 

---
### 11c. The threshold test: the only Part Four design that can move a structural zero
#### Chapter 18

Every other perturbation in Part Four multiplies the matrix entries, so a zero row stays a
zero row. That makes two things untestable by those designs: the two-tier split itself, and
five of Chapter 16's twelve counts against index-pairing. The 100 per cent figure once
recorded for the split was vacuous, and this cell replaces it with a design that could have
come out the other way.

The question is inverted. Rather than jittering the entries that exist, ask what strength a
Tradition would need across every resource its own Step consumes before index-pairing held
at that Step. It is one division per Step, exact, with no sampling error:

> c\*_i = (largest entry in row i other than the index-mate's) / (row sum of S for Step i)

Compare c\* against the entries actually written. The governance matrix has 35 non-zero
cells of 96, mean 0.374 and median 0.300. **Every one of the twelve thresholds exceeds both**,
and at the mean live strength no index-mate wins.

**One assertion here failed on its first run and corrected the chapter.** Chapter 18's main
text said the five protective thresholds sit inside the enabling range. They do not: the
protective range extends below the enabling range at both ends, and the protective mean
threshold is 0.501 against 0.531 for the enabling seven. The protective Steps are marginally
closer to index-pairing holding, which is the direction an objector would predict. The
chapter now says so.

In [ ]:
# --- Ch 18: the threshold test, the only design in Part Four that can move a structural zero
# Every other perturbation in this part multiplies the matrix entries, so a zero row stays a
# zero row and no draw can ever show a protective Tradition supplying anything. That makes the
# multiplicative "100 per cent" figure for the two-tier split vacuous, and it also makes five
# of Chapter 16's twelve counts untestable by that route. This cell asks the question the other
# way: how strongly would a Tradition have to govern its own Step's needs before index-pairing
# held? It is one division per Step and carries no sampling error.
B = m.S @ m.GOV.T
PROT = [j for j in range(12) if m.GOV[j].sum() == 0]
ENAB = [j for j in range(12) if j not in PROT]
check("Ch18 protective traditions", float(len(PROT)), 5.0, tol=0.01)
check("Ch18 protective are T4,T6,T7,T9,T10", float(sum(PROT)), float(3+5+6+8+9), tol=0.01)

NZ = m.GOV[m.GOV > 0]
print(f"Governance matrix: {NZ.size} non-zero of {m.GOV.size} cells; "
      f"mean {NZ.mean():.3f}, median {np.median(NZ):.3f}, min {NZ.min():.2f}, max {NZ.max():.2f}\n")
check("Ch18 non-zero governance cells", float(NZ.size), 35.0, tol=0.01)
check("Ch18 governance cells total", float(m.GOV.size), 96.0, tol=0.01)
check("Ch18 mean live entry", float(NZ.mean()), 0.374, tol=0.001)
check("Ch18 median live entry", float(np.median(NZ)), 0.300, tol=0.001)
check("Ch18 min live entry", float(NZ.min()), 0.10, tol=0.001)
check("Ch18 max live entry", float(NZ.max()), 1.00, tol=0.001)

print("Threshold c* = (what the index-mate must beat) / (Step's total consumption)\n")
print(f"{'step':<6}{'beats':>8}{'row sum S':>11}{'c*':>8}{'c*/mean':>9}   tier")
CSTAR = np.zeros(12)
for i in range(12):
    beat = np.sort(B[i])[-1] if int(np.argmax(B[i])) != i else np.sort(B[i])[-2]
    w = m.S[i].sum()
    CSTAR[i] = beat / w
    print(f"{i+1:<6}{beat:>8.2f}{w:>11.2f}{CSTAR[i]:>8.3f}{CSTAR[i]/NZ.mean():>9.2f}   "
          f"{'protective' if i in PROT else 'enabling'}")
    check(f"Ch18 c* step {i+1}", float(CSTAR[i]), round(float(CSTAR[i]), 3), tol=0.0006)

for lab, i, want in [("step 1", 0, 0.508), ("step 2", 1, 0.482), ("step 3", 2, 0.443),
                     ("step 4", 3, 0.438), ("step 5", 4, 0.635), ("step 6", 5, 0.417),
                     ("step 7", 6, 0.425), ("step 8", 7, 0.538), ("step 9", 8, 0.600),
                     ("step 10", 9, 0.627), ("step 11", 10, 0.588), ("step 12", 11, 0.521)]:
    check(f"Ch18 threshold {lab}", float(CSTAR[i]), want, tol=0.001)

print(f"\nprotective c* range {CSTAR[PROT].min():.3f} to {CSTAR[PROT].max():.3f}")
print(f"enabling   c* range {CSTAR[ENAB].min():.3f} to {CSTAR[ENAB].max():.3f}")
print(f"mean c* across all twelve: {CSTAR.mean():.3f}")
check("Ch18 protective c* min", float(CSTAR[PROT].min()), 0.417, tol=0.001)
check("Ch18 protective c* max", float(CSTAR[PROT].max()), 0.627, tol=0.001)
check("Ch18 enabling c* min", float(CSTAR[ENAB].min()), 0.443, tol=0.001)
check("Ch18 enabling c* max", float(CSTAR[ENAB].max()), 0.635, tol=0.001)
check("Ch18 mean c* over twelve", float(CSTAR.mean()), 0.518, tol=0.001)
# NOT containment. The protective range extends BELOW the enabling range at both ends
# (0.417 vs 0.443 at the bottom, 0.627 vs 0.635 at the top), so the protective Steps are
# marginally CLOSER to index-pairing holding than the enabling ones, not equidistant. An
# earlier draft of Chapter 18 said "inside", this assertion failed, and the chapter was
# corrected. Recorded here so the correction is visible rather than tidied away.
_overlap = bool(CSTAR[PROT].min() < CSTAR[ENAB].min() and CSTAR[PROT].max() < CSTAR[ENAB].max())
print(f"protective mean c* {CSTAR[PROT].mean():.3f} vs enabling mean c* {CSTAR[ENAB].mean():.3f}")
print(f"protective range extends below the enabling range at both ends: {_overlap}")
check("Ch18 protective mean c*", float(CSTAR[PROT].mean()), 0.501, tol=0.001)
check("Ch18 enabling mean c*", float(CSTAR[ENAB].mean()), 0.531, tol=0.001)
check("Ch18 protective range extends lower", float(_overlap), 1.0, tol=0.01)
check("Ch18 every c* above the mean live entry", float(bool((CSTAR > NZ.mean()).all())), 1.0, tol=0.01)
check("Ch18 every c* above the median live entry", float(bool((CSTAR > np.median(NZ)).all())), 1.0, tol=0.01)

print("\nGive every Tradition the MEAN live strength on every resource its own Step consumes:")
_wins = 0; _margins = []
for i in range(12):
    got = NZ.mean() * m.S[i].sum(); beat = np.sort(B[i])[-1] if int(np.argmax(B[i])) != i else np.sort(B[i])[-2]
    _margins.append(beat - got)
    if got > beat: _wins += 1
print(f"   index-mates that would then win: {_wins} of 12")
print(f"   smallest losing margin {min(_margins):.2f} (step {int(np.argmin(_margins))+1}), "
      f"largest {max(_margins):.2f} (step {int(np.argmax(_margins))+1})")
check("Ch18 index-mates winning at mean strength", float(_wins), 0.0, tol=0.01)
check("Ch18 smallest losing margin at mean strength", float(min(_margins)), 0.02, tol=0.005)
check("Ch18 largest losing margin at mean strength", float(max(_margins)), 0.44, tol=0.005)
print("\n  -> This is the only design in Part Four that could have overturned the five")
print("     protective counts, and it does not. Chapter 16's 'five counts are arithmetic'")
print("     is true of the multiplicative designs and is repaired here.")


  OK  Ch18 protective traditions                        5.000  (book: 5.0)
  OK  Ch18 protective are T4,T6,T7,T9,T10              31.000  (book: 31.0)
Governance matrix: 35 non-zero of 96 cells; mean 0.374, median 0.300, min 0.10, max 1.00

  OK  Ch18 non-zero governance cells                   35.000  (book: 35.0)
  OK  Ch18 governance cells total                      96.000  (book: 96.0)
  OK  Ch18 mean live entry                              0.374  (book: 0.374)
  OK  Ch18 median live entry                            0.300  (book: 0.3)
  OK  Ch18 min live entry                               0.100  (book: 0.1)
  OK  Ch18 max live entry                               1.000  (book: 1.0)
Threshold c* = (what the index-mate must beat) / (Step's total consumption)

step     beats  row sum S      c*  c*/mean   tier
1         1.22       2.40   0.508     1.36   enabling
  OK  Ch18 c* step 1                                    0.508  (book: 0.508)
2         0.82       1.70   0.482     1.29   en

---
### 11d. The column view, and the Kurtz conflation test
#### Chapter 17

Section 11 reads the coupling row-wise. This reads it column-wise: what each Tradition is
carrying across all twelve Steps at once. All exact.

Two things here are not in section 11. The first is a concentration measure, because unity's
primacy turns out to rest on breadth rather than strength and the chapter has to say how much
breadth. **Unity is the only Tradition governing all eight resources, and it is in the top two
for ten of the twelve Steps, but it is not the most diffuse by HHI**: singleness of purpose
scores 0.197 against unity's 0.204.

The second is the answer to an objection the plan raised before the chapter was written.
Kurtz's note 16 to his Chapter Five records that in some later AA literature the concept
conveyed by *single-purposed* was obfuscated by substituting *unity*. If this chapter absorbed
that substitution, its finding is an artefact. The test transfers unity's governance of one
resource to singleness of purpose, wholesale, and asks who leads. **Six of the eight cannot
flip it. Only continuity and pressure can.**

**One assertion failed on its first run and corrected the chapter.** The main text said that
stripped of continuity and pressure unity falls below the open door. It does not: 2.77 against
2.69, so third rather than fourth.

In [ ]:
# --- Ch 17: the column view of the coupling, and the Kurtz conflation test
# Chapter 16 reads B row-wise (which Tradition serves each Step). This reads it column-wise
# (what each Tradition is carrying). All exact; the only Monte Carlo figures are the
# robustness proportions, which are computed in section 11 above.
B = m.S @ m.GOV.T
CS = m.S.sum(0)                    # total demand per resource across all twelve Steps
LOAD = B.sum(0)
RESN = m.RES
print("Total demand per resource:", {RESN[r]: round(float(CS[r]), 2) for r in range(8)}, "\n")

print(f"{'T':<5}{'load':>7}{'#res':>6}{'top resource':>15}{'share':>8}{'HHI':>7}{'1st':>5}{'2nd':>5}")
STATS = {}
for j in np.argsort(-LOAD):
    contrib = np.array([CS[r]*m.GOV[j, r] for r in range(8)])
    first = sum(1 for i in range(12) if int(np.argmax(B[i])) == j)
    second = sum(1 for i in range(12) if int(np.argsort(-B[i])[1]) == j)
    if LOAD[j] == 0:
        STATS[j] = (0.0, 0, None, 0.0, 0.0, first, second)
        print(f"T{j+1:<4}{0.0:>7.2f}{0:>6}{'-':>15}{'-':>8}{'-':>7}{first:>5}{second:>5}")
        continue
    sh = contrib/contrib.sum(); hhi = float((sh**2).sum())
    STATS[j] = (float(LOAD[j]), int((contrib > 0).sum()), RESN[int(np.argmax(contrib))],
                float(sh.max()), hhi, first, second)
    print(f"T{j+1:<4}{LOAD[j]:>7.2f}{int((contrib>0).sum()):>6}{RESN[int(np.argmax(contrib))]:>15}"
          f"{sh.max():>8.2f}{hhi:>7.3f}{first:>5}{second:>5}")

for j, want in [(0, 6.52), (1, 3.89), (4, 3.88), (2, 2.69), (10, 2.69), (11, 2.62), (7, 1.11)]:
    check(f"Ch17 load T{j+1}", float(LOAD[j]), want, tol=0.006)
for j in [3, 5, 6, 8, 9]:
    check(f"Ch17 load T{j+1} is zero", float(LOAD[j]), 0.0, tol=1e-12)
check("Ch17 T1 governs all eight resources", float((m.GOV[0] > 0).sum()), 8.0, tol=0.01)
check("Ch17 T5 governs six resources", float((m.GOV[4] > 0).sum()), 6.0, tol=0.01)
check("Ch17 T1 HHI", STATS[0][4], 0.204, tol=0.001)
check("Ch17 T5 HHI", STATS[4][4], 0.197, tol=0.001)
check("Ch17 T2 HHI", STATS[1][4], 0.520, tol=0.001)
check("Ch17 T2 counsel share", STATS[1][3], 0.69, tol=0.005)
check("Ch17 T12 confidential share", STATS[11][3], 0.57, tol=0.005)
check("Ch17 T11 proof share", STATS[10][3], 0.49, tol=0.005)
check("Ch17 T1 principal count", float(STATS[0][5]), 4.0, tol=0.01)
check("Ch17 T1 runner-up count", float(STATS[0][6]), 6.0, tol=0.01)
check("Ch17 T2 principal count", float(STATS[1][5]), 4.0, tol=0.01)
check("Ch17 T2 runner-up count", float(STATS[1][6]), 4.0, tol=0.01)
_top2 = sum(1 for i in range(12) if 0 in list(np.argsort(-B[i])[:2]))
check("Ch17 T1 in top two for N steps", float(_top2), 10.0, tol=0.01)

print("\nT1's load by resource:")
_c1 = np.array([CS[r]*m.GOV[0, r] for r in range(8)])
for r in np.argsort(-_c1):
    print(f"   {RESN[r]:<14}{_c1[r]:>6.2f}")
for r, want in [(6, 1.89), (7, 1.86), (1, 0.85), (4, 0.60), (2, 0.57), (3, 0.45), (0, 0.20), (5, 0.10)]:
    check(f"Ch17 T1 from {RESN[r]}", float(_c1[r]), want, tol=0.006)
_pair = float(_c1[6] + _c1[7])
print(f"\ncontinuity + pressure = {_pair:.2f} of {LOAD[0]:.2f} = {100*_pair/LOAD[0]:.1f} per cent")
check("Ch17 continuity+pressure share of T1", 100*_pair/float(LOAD[0]), 57.5, tol=0.1)
check("Ch17 T1 without continuity and pressure", float(LOAD[0]) - _pair, 2.77, tol=0.006)
_rest = float(LOAD[0]) - _pair
_rank = 1 + sum(1 for j in range(12) if j != 0 and LOAD[j] > _rest)
print(f"stripped of both, T1 would rank {_rank} of twelve at {_rest:.2f}")
# An earlier draft of Chapter 17 said unity falls BELOW the open door when stripped of
# continuity and pressure. It does not: 2.77 is above the open door's 2.69, so the rank
# is third, not fourth. This assertion failed and the chapter was corrected.
check("Ch17 T1 rank stripped of both", float(_rank), 3.0, tol=0.01)

# --- the Kurtz conflation test: hand T1's governance of one resource to T5, wholesale
print("\nReassignment test. T5 takes T1's coefficient for one resource; T1 keeps nothing.\n")
print(f"{'resource moved':<16}{'T1':>7}{'T5':>7}   leader")
FLIP = []
for r in range(8):
    G2 = m.GOV.copy(); G2[4, r] = max(G2[4, r], G2[0, r]); G2[0, r] = 0.0
    L2 = (m.S @ G2.T).sum(0)
    lead = int(np.argmax(L2))
    if lead != 0: FLIP.append(RESN[r])
    print(f"{RESN[r]:<16}{L2[0]:>7.2f}{L2[4]:>7.2f}   T{lead+1}")
    check(f"Ch17 reassign {RESN[r]}: T1", float(L2[0]), round(float(L2[0]), 2), tol=0.006)
    check(f"Ch17 reassign {RESN[r]}: T5", float(L2[4]), round(float(L2[4]), 2), tol=0.006)
print(f"\nresources whose transfer flips the lead: {FLIP}")
check("Ch17 flips the lead, count", float(len(FLIP)), 2.0, tol=0.01)
check("Ch17 flip is continuity and pressure",
      float(set(FLIP) == {'continuity', 'pressure'}), 1.0, tol=0.01)
for r, w1, w5 in [(0, 6.32, 3.98), (1, 5.67, 4.05), (2, 5.95, 3.88), (3, 6.07, 4.33),
                  (4, 5.92, 4.48), (5, 6.42, 3.88), (6, 4.63, 5.14), (7, 4.66, 5.12)]:
    G2 = m.GOV.copy(); G2[4, r] = max(G2[4, r], G2[0, r]); G2[0, r] = 0.0
    L2 = (m.S @ G2.T).sum(0)
    check(f"Ch17 reassign {RESN[r]} T1 value", float(L2[0]), w1, tol=0.006)
    check(f"Ch17 reassign {RESN[r]} T5 value", float(L2[4]), w5, tol=0.006)
_marg = []
for r in range(8):
    G2 = m.GOV.copy(); G2[4, r] = max(G2[4, r], G2[0, r]); G2[0, r] = 0.0
    L2 = (m.S @ G2.T).sum(0)
    if int(np.argmax(L2)) == 0: _marg.append(float(L2[0] - L2[4]))
print(f"margins where unity still leads: {min(_marg):.2f} to {max(_marg):.2f}")
check("Ch17 smallest surviving margin", min(_marg), 1.44, tol=0.006)
check("Ch17 largest surviving margin", max(_marg), 2.54, tol=0.006)
# HHI reference point: a Tradition drawing equally on all eight resources
_uniform_hhi = float((np.full(8, 1/8)**2).sum())
print(f"HHI of a Tradition drawing equally on all eight resources: {_uniform_hhi:.3f}")
check("Ch17 uniform-eight HHI reference", _uniform_hhi, 0.125, tol=0.0005)

print("\n  -> Kurtz's charge that later AA literature substituted 'unity' for 'single-purposed'")
print("     can only reach this chapter's finding through the continuity or pressure rows.")
print("     Six of the eight resources cannot flip it even when transferred wholesale.")


Total demand per resource: {'admission': 1.0, 'identify': 1.7, 'proof': 1.9, 'confidential': 1.5, 'counsel': 3.0, 'recipient': 1.0, 'continuity': 2.1, 'pressure': 3.1} 

T       load  #res   top resource   share    HHI  1st  2nd
T1      6.52     8     continuity    0.29  0.204    4    6
T2      3.89     4        counsel    0.69  0.520    4    4
T5      3.88     6          proof    0.24  0.197    1    1
T3      2.69     4      admission    0.37  0.297    1    1
T11     2.69     4          proof    0.49  0.335    1    0
T12     2.62     5   confidential    0.57  0.387    1    0
T8      1.11     4   confidential    0.41  0.290    0    0
T4      0.00     0              -       -      -    0    0
T6      0.00     0              -       -      -    0    0
T7      0.00     0              -       -      -    0    0
T9      0.00     0              -       -      -    0    0
T10     0.00     0              -       -      -    0    0
  OK  Ch17 load T1                                      6.520  

---
## 12. Morris elementary-effects screen
### `appendix/APPENDIX.md` A5.5

Every earlier design is either a global jitter, which confounds all interactions together,
or a one-at-a-time star around the nominal point, which cannot see them at all. Morris
(1991), in the improved sampling of Campolongo, Cariboni and Saltelli (2007), is the
standard screen at this many factors. It reports, per factor:

**mu\*** the mean absolute elementary effect, that is, how much the factor moves the
output; **mu**, the signed mean, giving direction and consistency; and **sigma**, the
spread of the effect across the parameter space. High sigma means the factor's effect
depends on where the others are, which is interaction or non-linearity; Morris cannot
distinguish the two and nothing here claims to.

Design: 10 trajectories, 4 levels, delta = 2/3, factors mapped to plus or minus 25 per cent
of nominal to match the OAT sweep, 1,190 model evaluations, 5 seeds each under common
random numbers. Traditions held at **0.85, not 1.0**, because at full adherence the
governance matrix cancels exactly and a third of the design would be wasted.

In [ ]:
# --- Morris screen: read the raw evaluations and recompute the elementary effects
import importlib.util as _il, json as _json, os as _os
_R = '../research' if _os.path.isdir('../research') else 'research'
_MS = '../model/morris_screen.py' if _os.path.exists('../model/morris_screen.py') else 'morris_screen.py'
_sp = _il.spec_from_file_location('ms', _MS); ms = _il.module_from_spec(_sp); _sp.loader.exec_module(ms)
MO = _json.load(open(f'{_R}/morris.json')); META = MO['meta']; IDS = META['ids']
K, RT = META['k'], META['r']
check("Morris factors screened", float(K), 118.0, tol=0.5)
check("Morris trajectories",     float(RT), 10.0, tol=0.5)
check("Morris evaluations",      float(len(MO)-1), 1190.0, tol=0.5)

TR = ms.trajectories(K, RT)
EE = {0: np.full((RT, K), np.nan), 1: np.full((RT, K), np.nan)}
for t, (pts, _o, _s) in enumerate(TR):
    for s in range(len(pts)-1):
        i = int(np.argmax(np.abs(pts[s+1]-pts[s]))); dx = pts[s+1][i]-pts[s][i]
        if abs(dx) < 1e-12: continue
        y0, y1 = MO[f'{t}_{s}'], MO[f'{t}_{s+1}']
        for o in (0, 1): EE[o][t, i] = (y1[o]-y0[o])/dx

for o, name in [(0, 'final membership'), (1, 'mean practice')]:
    E = EE[o]
    mus = np.nanmean(np.abs(E), axis=0); mu = np.nanmean(E, axis=0)
    sg  = np.nanstd(E, axis=0, ddof=1); se = sg/np.sqrt(RT)
    gov = np.array([mus[i] for i, p in enumerate(IDS) if p.startswith('GOV:')])
    floor = np.percentile(gov, 95)
    print(f"\n=== {name} ===")
    print(f"resolution limit (95th percentile of the near-inert governance factors): {floor:.3f}")
    print(f"{'factor':<17}{'mu*':>9}{'SE':>7}{'mu':>9}{'sigma':>8}{'s/mu*':>7}  reading")
    for i in np.argsort(-mus)[:8]:
        rd = 'largely additive' if sg[i]/mus[i] < 1 else 'interaction or non-linearity'
        print(f"{IDS[i]:<17}{mus[i]:>9.3f}{se[i]:>7.3f}{mu[i]:>9.3f}{sg[i]:>8.3f}"
              f"{sg[i]/mus[i]:>7.2f}  {rd}")
    tot = mus.sum()
    print(f"\n{'kind':<8}{'n':>4}{'max mu*':>11}{'median':>11}{'share of total':>16}")
    for kk in ('scalar', 'a', 'S', 'GOV'):
        v = np.array([mus[i] for i, p in enumerate(IDS) if p.startswith(kk+':')])
        print(f"{kk:<8}{len(v):>4}{v.max():>11.3f}{np.median(v):>11.3f}{100*v.sum()/tot:>15.1f}%")
    if o == 0:
        check("Morris: p_gate is the top factor for membership", float(np.argmax(mus) == IDS.index('scalar:p_gate')), 1.0, tol=0.01)
        check("Morris: mu* for p_gate",  mus[IDS.index('scalar:p_gate')], 42.24, tol=0.05)
        check("Morris: mu* for delta0",  mus[IDS.index('scalar:delta0')], 36.15, tol=0.05)
        check("Morris: governance share of total mu*",
              100*np.array([mus[i] for i, p in enumerate(IDS) if p.startswith('GOV:')]).sum()/tot, 3.6, tol=0.1)
        top5 = set(IDS[i] for i in np.argsort(-mus)[:5])
        assert top5 == {'scalar:p_gate','scalar:delta0','scalar:churn','scalar:het_sd','scalar:drop_k'}, top5
        assert all(sg[i]/mus[i] < 1 for i in np.argsort(-mus)[:5]), "a top-five factor is now interaction-dominated"
    else:
        check("Morris: governance share of total mu*, practice",
              100*np.array([mus[i] for i, p in enumerate(IDS) if p.startswith('GOV:')]).sum()/tot, 4.1, tol=0.1)

print("\n--- what the screen establishes ---")
print("1. The same five factors dominate as the one-at-a-time sweep found, and for")
print("   membership all five have sigma/mu* below 1. Their effects are largely additive,")
print("   so the OAT ranking was not an artefact of looking at one point.")
print("2. The governance matrix is 35 of 118 factors and carries under 4 per cent of the")
print("   total effect even at PARTIAL adherence, where it is not cancelled. This is a")
print("   stronger statement than the identity in appendix A5.3, which covers full")
print("   adherence only.")
print("3. Interactions are present but secondary: every factor with sigma/mu* above 1 sits")
print("   outside the top five on membership. No claim in the book rests on one.")
print("4. Morris screens; it does not decompose variance. Separating interaction from")
print("   non-linearity for the top factors needs Sobol indices, which are not run here.")


  OK  Morris factors screened                         118.000  (book: 118.0)
  OK  Morris trajectories                              10.000  (book: 10.0)
  OK  Morris evaluations                             1190.000  (book: 1190.0)

=== final membership ===
resolution limit (95th percentile of the near-inert governance factors): 1.413
factor                 mu*     SE       mu   sigma  s/mu*  reading
scalar:p_gate       42.240 10.696  -42.000  33.824   0.80  largely additive
scalar:delta0       36.150  6.315  -36.150  19.969   0.55  largely additive
scalar:churn        20.580  5.586  -20.580  17.664   0.86  largely additive
scalar:het_sd       18.420  5.279   18.420  16.694   0.91  largely additive
scalar:drop_k       16.410  3.873   16.410  12.248   0.75  largely additive
a:3                 10.530  4.032    6.270  12.752   1.21  interaction or non-linearity
a:7                  9.870  5.183    9.690  16.390   1.66  interaction or non-linearity
scalar:omega         9.600  3.863    9.54

---
## 13. Service as a load-bearing wall
### Chapter 15

The twelfth step is the only consumer of the **recipient** resource, and the recipient
resource is the only one of the eight not produced by the members themselves. Three
configurations at 400 seeds test whether service is terminal or structural.

In [ ]:
# --- Ch 15: the recipient resource and what removing service costs
P = m.DEFAULTS
sat = lambda c, k: c/(c + k)
print(f"Recipient resource: sat(newcomers per experienced member, k_recip={P['k_recip']})\n")
print(f"{'ratio':>7}{'supply':>9}")
for r in (0.25, 0.5, 1, 2, 4, 10):
    print(f"{r:>7.2f}{sat(r, P['k_recip']):>9.3f}")
check("Ch15 k_recip", float(P['k_recip']), 2.0, tol=1e-9)
check("Ch15 half-supply ratio", float(P['k_recip']), 2.0, tol=1e-9)

consumers = [i+1 for i in range(12) if m.S[i, 5] > 0]
print(f"\nsteps consuming the recipient resource: {consumers}")
check("Ch15 recipient consumers", float(len(consumers)), 1.0, tol=0.01)
assert consumers == [12], "the recipient resource is no longer exclusive to Step 12"
check("Ch15 Step 12 recipient share of its bundle", float(m.Snorm[11, 5]), 0.417, tol=0.002)
check("Ch15 Step 12 group-dependence", float(m.BETA[11]), 1.00, tol=0.005)
govs = {f"T{j+1}": round(float(m.GOVW[j, 5]), 3) for j in range(12) if m.GOV[j, 5] > 0}
print(f"traditions governing it, normalised: {govs}")
check("Ch15 T5 governs recipient", float(m.GOVW[4, 5]), 0.375, tol=0.002)
check("Ch15 T3 governs recipient", float(m.GOVW[2, 5]), 0.333, tol=0.002)

# --- the three configurations, read from disk (ch15_service.py, 400 seeds each)
SV = list(_json.load(open(f'{_R}/ch15_service.json')).values())
def cfg(c):
    r = [v for v in SV if v['cfg'] == c]; n = len(r); k = sum(v['alive'] for v in r)
    N = np.array([v['N'] for v in r], float); ok = [v for v in r if v['alive']]
    g = lambda key: np.array([v[key] for v in ok], float)
    return dict(n=n, surv=k/n, ci=wilson(k, n), N=N.mean(), Nse=N.std(ddof=1)/np.sqrt(n),
                pr=g('practice').mean(), prse=g('practice').std(ddof=1)/np.sqrt(len(ok)),
                s1=g('s1').mean(), s9=g('s9').mean(), s12=g('s12').mean(), mt=g('maint').mean())
C = {c: cfg(c) for c in ('base', 'no12', 'norecip')}
print(f"\n{'configuration':<28}{'surv':>7}{'N':>8}{'+/-':>7}{'practice':>10}{'Step 1':>9}"
      f"{'Step 9':>9}{'Step 12':>9}{'maint':>9}")
for c, lab in [('base', 'baseline'), ('no12', 'twelfth step disabled'),
               ('norecip', 'recipient dependence removed')]:
    v = C[c]
    print(f"{lab:<28}{v['surv']:>7.3f}{v['N']:>8.1f}{1.96*v['Nse']:>7.1f}{v['pr']:>10.3f}"
          f"{v['s1']:>9.3f}{v['s9']:>9.3f}{v['s12']:>9.3f}{v['mt']:>9.4f}")
for c, wn, wp, w9, wmt in [('base', 41.7, 0.320, 0.134, 0.1158),
                           ('no12', 14.2, 0.269, 0.097, 0.0597),
                           ('norecip', 45.4, 0.327, 0.140, 0.1264)]:
    check(f"Ch15 {c} mean N",      C[c]['N'],  wn,  tol=0.25)
    check(f"Ch15 {c} practice",    C[c]['pr'], wp,  tol=0.004)
    check(f"Ch15 {c} Step 9",      C[c]['s9'], w9,  tol=0.004)
    check(f"Ch15 {c} maintenance", C[c]['mt'], wmt, tol=0.004)
check("Ch15 survival with no twelfth step", C['no12']['surv'], 1.000, tol=0.006)

dN = 100*(C['no12']['N']/C['base']['N'] - 1); d9 = 100*(C['no12']['s9']/C['base']['s9'] - 1)
dR = 100*(C['norecip']['N']/C['base']['N'] - 1)
print(f"\n  twelfth step disabled: N {dN:+.1f}%, Step 9 {d9:+.1f}%, "
      f"maintenance {100*(C['no12']['mt']/C['base']['mt']-1):+.1f}%")
print(f"  recipient dependence removed: N {dR:+.1f}%, "
      f"Step 12 {100*(C['norecip']['s12']/C['base']['s12']-1):+.1f}%")
check("Ch15 membership cost of losing service, per cent", dN, -65.9, tol=0.6)
check("Ch15 Step 9 cost of losing service, per cent",     d9, -27.6, tol=0.8)
check("Ch15 membership gain from relaxing the recipient constraint", dR, 8.7, tol=0.6)

se = np.sqrt(C['base']['Nse']**2 + C['no12']['Nse']**2)
print(f"\n  baseline minus disabled: {C['base']['N']-C['no12']['N']:.1f} members, "
      f"combined SE {se:.2f}, z = {(C['base']['N']-C['no12']['N'])/se:.1f}")
se2 = np.sqrt(C['base']['Nse']**2 + C['norecip']['Nse']**2)
print(f"  relaxed minus baseline:  {C['norecip']['N']-C['base']['N']:.1f} members, "
      f"combined SE {se2:.2f}, z = {(C['norecip']['N']-C['base']['N'])/se2:.1f}")
print("\n  -> Step 9 has NO direct dependence on service and still falls 28 per cent.")
print("     That is what load-bearing means: removing the twelfth step damages things")
print("     it is not connected to, by way of maintenance capacity.")
print("  -> and the group does not die. 400 of 400 survive without any service practice,")
print("     at a third of the size. That shape is stable and should not be called failure.")
print("  -> relaxing the recipient constraint HELPS, so in the ordinary configuration the")
print("     shortage of people to help is binding on established members.")


Recipient resource: sat(newcomers per experienced member, k_recip=2.0)

  ratio   supply
   0.25    0.111
   0.50    0.200
   1.00    0.333
   2.00    0.500
   4.00    0.667
  10.00    0.833
  OK  Ch15 k_recip                                      2.000  (book: 2.0)
  OK  Ch15 half-supply ratio                            2.000  (book: 2.0)

steps consuming the recipient resource: [12]
  OK  Ch15 recipient consumers                          1.000  (book: 1.0)
  OK  Ch15 Step 12 recipient share of its bundle        0.417  (book: 0.417)
  OK  Ch15 Step 12 group-dependence                     1.000  (book: 1.0)
traditions governing it, normalised: {'T1': 0.042, 'T3': 0.333, 'T5': 0.375, 'T11': 0.25}
  OK  Ch15 T5 governs recipient                         0.375  (book: 0.375)
  OK  Ch15 T3 governs recipient                         0.333  (book: 0.333)

configuration                  surv       N    +/-  practice   Step 1   Step 9  Step 12    maint
baseline                      0.995    41.7 

---
## 6. Regression result

In [ ]:
if FAILURES:
    print(f"{len(FAILURES)} MISMATCH(ES) between this notebook and the book:\n")
    for f in FAILURES: print("  ", f)
    print("\nEither the model changed or a chapter needs correcting.")
else:
    print("All checked figures match the values printed in the book.")

All checked figures match the values printed in the book.


---
## 12b. Sobol variance decomposition on the eight Morris leaders
### Appendix A5.7; no chapter depends on it

The Morris screen in section 12 ranks factors and reports a spread, sigma, that mixes two
different things: interaction with other factors, and curvature in the factor itself. It
cannot separate them and the appendix says so. Sobol can, because the gap between a
factor's total-order and first-order index is interaction and nothing else.

1,408 evaluations, N = 128 rows, Saltelli (2010) for first order and Jansen (1999) for
total order, bootstrap intervals over the rows, common random numbers, traditions at 0.85.
Ranges are plus or minus 25 per cent, matching the Morris hypercube and the one-at-a-time
star, so all three designs describe the same neighbourhood.

**Two results and one failure.** The total-order indices are stable and answer the question
Morris could not: delta0 and het_sd act on membership largely through interaction, a:8's
high Morris sigma was non-linearity rather than interaction, and Spearman's rho between
Morris mu\* and Sobol S_T is 0.833, so the Morris ranking is not an artefact of its
estimator. **The first-order indices are not estimable at this sample size** and the cell
demonstrates it three ways rather than asserting it: one index exceeds its own total for
membership and five do for practice, three are negative, the practice first-order indices
sum to 1.263, and splitting the sample in half moves the first-order indices by 0.092 on
average against 0.042 for total order. Resolving p_gate's first-order index to plus or minus
0.10 would need N = 1,379, about 13,800 evaluations, roughly ten times what was run.

Only the total-order indices are quoted in the appendix. The first-order ones are reported
as unresolved, which is what they are.

In [ ]:
# --- Appendix A5.7: Sobol variance decomposition on the eight Morris leaders
# Read from research/sobol.json, produced by model/sobol_indices.py. 1,408 model
# evaluations: A, B, a noise replicate of A, and eight AB_i matrices, at N = 128 rows.
# Nothing in the book's argument depends on this section; it exists because appendix A5.5
# named it as the last quantitative gap and A7 threat 5 named it as unaddressed.
_S = _json.load(open(f'{_R}/sobol.json'))
FAC = _S['meta']['factors']; NB = _S['meta']['n_base']; KF = len(FAC)
print(f"Sobol on {KF} factors, N = {NB}, cost {_S['meta']['cost']} evaluations, "
      f"+/-{int(100*_S['meta']['spread'])}% ranges, traditions at {_S['meta']['T_ref']}\n")
check("A5.7 factor count", float(KF), 8.0, tol=0.01)
check("A5.7 base sample rows", float(NB), 128.0, tol=0.01)
check("A5.7 evaluation cost", float(_S['meta']['cost']), 1408.0, tol=0.01)

NF = _S['result']['noise_floor_ST']
print(f"Monte Carlo noise floor on S_T: {NF[0]:.4f} membership, {NF[1]:.4f} practice")
check("A5.7 noise floor, membership", NF[0], 0.023, tol=0.001)
check("A5.7 noise floor, practice",   NF[1], 0.034, tol=0.001)

for _oc in ['membership', 'mean_practice']:
    d = _S['result'][_oc]
    print(f"\n{_oc}\n{'factor':<9}{'S1':>8}{'95% CI':>17}{'ST':>8}{'95% CI':>17}{'ST-S1':>8}")
    for i, f in enumerate(FAC):
        print(f"{f:<9}{d['S1'][i]:>8.3f}  [{d['S1_lo'][i]:>6.3f},{d['S1_hi'][i]:>6.3f}]"
              f"{d['ST'][i]:>8.3f}  [{d['ST_lo'][i]:>6.3f},{d['ST_hi'][i]:>6.3f}]"
              f"{d['ST'][i]-d['S1'][i]:>8.3f}")
    print(f"  sum S1 = {sum(d['S1']):.3f}   sum ST = {sum(d['ST']):.3f}")

_M = _S['result']['membership']
for lab, i, want in [("p_gate", 0, 0.456), ("delta0", 1, 0.258), ("churn", 2, 0.121),
                     ("het_sd", 3, 0.178), ("drop_k", 4, 0.134), ("a4", 5, 0.032),
                     ("a8", 6, 0.058), ("omega", 7, 0.063)]:
    check(f"A5.7 ST membership {lab}", _M['ST'][i], want, tol=0.001)
check("A5.7 sum ST membership", sum(_M['ST']), 1.300, tol=0.002)
check("A5.7 sum S1 membership", sum(_M['S1']), 0.615, tol=0.002)
check("A5.7 sum S1 practice", sum(_S['result']['mean_practice']['S1']), 1.263, tol=0.002)

# --- the three diagnostics that show first order is NOT resolved at this sample size
_bad = [FAC[i] for i in range(KF) if _M['S1'][i] > _M['ST'][i]]
_neg = [FAC[i] for i in range(KF) if _M['S1'][i] < 0]
_badp = [FAC[i] for i in range(KF) if _S['result']['mean_practice']['S1'][i]
         > _S['result']['mean_practice']['ST'][i]]
print(f"\nImpossible values, which can only be estimator noise:")
print(f"   membership, S1 > ST: {_bad}")
print(f"   membership, S1 < 0:  {_neg}")
print(f"   practice,   S1 > ST: {_badp}")
check("A5.7 membership S1>ST count", float(len(_bad)), 1.0, tol=0.01)
check("A5.7 membership S1<0 count", float(len(_neg)), 3.0, tol=0.01)
check("A5.7 practice S1>ST count", float(len(_badp)), 5.0, tol=0.01)

# half-sample stability: the direct demonstration
_fA = np.array([_S[f'A_{j}'] for j in range(NB)]); _fB = np.array([_S[f'B_{j}'] for j in range(NB)])
def _idx(rows, oc):
    a = _fA[rows, oc]; b = _fB[rows, oc]
    v = np.var(np.concatenate([a, b]), ddof=1)
    ab = np.array([[_S[f'AB{i}_{j}'][oc] for i in range(KF)] for j in rows])
    return (np.mean(b[:, None]*(ab - a[:, None]), axis=0)/v,
            np.mean((a[:, None] - ab)**2, axis=0)/(2*v))
_h1 = _idx(np.arange(0, NB//2), 0); _h2 = _idx(np.arange(NB//2, NB), 0)
_dS1 = float(np.mean(np.abs(_h1[0] - _h2[0]))); _dST = float(np.mean(np.abs(_h1[1] - _h2[1])))
print(f"\nHalf-sample instability, membership: mean |difference| S1 {_dS1:.3f}, ST {_dST:.3f}")
check("A5.7 half-sample mean diff S1", _dS1, 0.092, tol=0.001)
check("A5.7 half-sample mean diff ST", _dST, 0.042, tol=0.001)
_hw = (np.array(_M['S1_hi']) - np.array(_M['S1_lo']))/2
_need = int(round(NB*(_hw[0]/0.10)**2))
print(f"N needed for p_gate's S1 half-width of 0.10: {_need}, costing {_need*(KF+2)} evaluations")
check("A5.7 N needed for S1 half-width 0.10", float(_need), 1379.0, tol=1.0)

# --- does Sobol confirm the Morris ranking? Spearman on mu* against S_T.
MU = [42.24, 36.15, 20.58, 18.42, 16.41, 10.53, 9.87, 9.60]      # appendix A5.5, same order
def _rank(v):
    o = np.argsort(-np.asarray(v)); r = np.empty(len(v)); r[o] = np.arange(1, len(v)+1); return r
_d = _rank(MU) - _rank(_M['ST']); _rho = 1 - 6*float(np.sum(_d**2))/(KF*(KF**2-1))
print(f"\nSpearman rho between Morris mu* and Sobol S_T: {_rho:.3f}")
check("A5.7 Spearman Morris vs Sobol", _rho, 0.833, tol=0.001)
print("  -> two independent designs, same top two, rho 0.833. The Morris ranking is not")
print("     an artefact of the elementary-effect estimator.\n")
print("  -> WHAT SOBOL SETTLES: delta0 and het_sd have S_T far above zero while their S1")
print("     intervals cover zero, so they act on membership largely THROUGH interaction.")
print("     a:8 has S_T - S1 of 0.005, so its high Morris sigma was non-linearity, not")
print("     interaction. a:4 and omega show interaction but both sit near the noise floor.")
print("  -> WHAT IT DOES NOT SETTLE: any first-order index. See the diagnostics above.")


Sobol on 8 factors, N = 128, cost 1408 evaluations, +/-25% ranges, traditions at 0.85

  OK  A5.7 factor count                                 8.000  (book: 8.0)
  OK  A5.7 base sample rows                           128.000  (book: 128.0)
  OK  A5.7 evaluation cost                           1408.000  (book: 1408.0)
Monte Carlo noise floor on S_T: 0.0231 membership, 0.0338 practice
  OK  A5.7 noise floor, membership                      0.023  (book: 0.023)
  OK  A5.7 noise floor, practice                        0.034  (book: 0.034)

membership
factor         S1           95% CI      ST           95% CI   ST-S1
p_gate      0.483  [ 0.172, 0.828]   0.456  [ 0.331, 0.599]  -0.027
delta0      0.010  [-0.207, 0.224]   0.258  [ 0.164, 0.367]   0.248
churn       0.108  [-0.029, 0.270]   0.121  [ 0.076, 0.179]   0.014
het_sd     -0.044  [-0.210, 0.117]   0.178  [ 0.106, 0.261]   0.222
drop_k      0.078  [-0.071, 0.236]   0.134  [ 0.088, 0.193]   0.056
a:4        -0.052  [-0.135, 0.024]   0.032

---
## 14. Part Five: how a group dies
### Chapters 19 to 22

4,800 runs at 400 seeds per condition, thirty-year horizon, from `model/part5_runs.py`,
cached in `research/part5.json`. Three things the endpoint table in section 1 cannot supply.

**(a) The Tradition 3 sweep, for Chapter 19.** A group cannot refuse admission, so the open
door acts on retention. Closing it entirely costs 14.2 members and 5.5 points of survival, and
**raises** measured quality by 0.046 while **raising** the newcomer share from 0.122 to 0.201.
A closed group is smaller, looks better, and is more of a revolving door. Note that survival at
T3 = 0.75 and T3 = 1.00 is not distinguishable on the Wilson intervals and must not be read as
a peak at 0.75.

**(b) Trajectories, for Chapters 20 and 21.** The three failure modes have distinct
fingerprints, and the important one is that they are distinguishable *in a way an insider could
not see*. An unreferred group's survivors look healthy at every horizon: at year 10 they are at
quality 0.348 against a healthy 0.338, which is **better**, while 9.3 per cent of such groups
are already dead. By year 30 only 36 per cent survive and those survivors are still at 26.6
members. The decline is entirely in the deaths, not in the survivors. An invisible group, by
contrast, degrades where anyone can see it: quality 0.310 at year 10 against 0.338.

**(c) The composition experiment, for Chapter 22.** Same founding practice distributed three
ways gives a spread of 1.8 members against a 95 per cent half-width of 1.6. A null result, and
the model was nearly incapable of producing anything else, which the chapter says.

Every figure below carries its sample size and, where it is a proportion, a Wilson interval.
Quality is conditional on survival throughout and the conditioning is reported beside it.

In [ ]:
# --- Part Five: the Tradition 3 sweep, three fingerprints of decline, and composition
# All from research/part5.json, produced by model/part5_runs.py. 4,800 model runs at 400
# seeds per condition, thirty-year horizon. Nothing here is carried by hand.
_P5 = _json.load(open(f'{_R}/part5.json')); _MT = _P5['meta']
def _g(kind, arg): return [_P5[f'{kind}|{arg}|{s}'] for s in range(_MT['nseed'])]
check("P5 seeds per condition", float(_MT['nseed']), 400.0, tol=0.01)
check("P5 total runs", float(_MT['jobs']), 4800.0, tol=0.01)

# ---------- (a) Chapter 19: what a closed door costs
print("Tradition 3 sweep, 400 seeds, everything else at full adherence\n")
print(f"{'T3':>6}{'survival':>10}{'95% CI':>18}{'mean N':>9}{'+/-':>7}{'quality':>9}{'+/-':>8}{'core':>7}{'newcomer':>10}")
T3ROW = {}
for lv in _MT['t3_levels']:
    v = _g('t3', lv)
    N = np.array([x['N'] for x in v], float); k = sum(x['alive'] for x in v)
    q = np.array([x['est_mean'] for x in v if x['alive']], float)
    ne = np.array([x['n_est'] for x in v if x['alive']], float)
    nf = np.array([x['newcomer_frac'] for x in v if x['alive']], float)
    lo, hi = wilson(k, 400)
    T3ROW[lv] = (k/400, lo, hi, N.mean(), 1.96*N.std(ddof=1)/20, q.mean(),
                 1.96*q.std(ddof=1)/np.sqrt(len(q)), ne.mean(), nf.mean())
    print(f"{lv:>6.2f}{k/400:>10.3f}   [{lo:.3f},{hi:.3f}]{N.mean():>9.1f}{1.96*N.std(ddof=1)/20:>7.1f}"
          f"{q.mean():>9.4f}{1.96*q.std(ddof=1)/np.sqrt(len(q)):>8.4f}{ne.mean():>7.1f}{nf.mean():>10.3f}")
for lv, sv, nn, qq, nc in [(0.0, 0.940, 27.5, 0.4003, 0.201), (0.25, 0.978, 33.7, 0.3878, 0.165),
                           (0.50, 0.990, 36.6, 0.3700, 0.148), (0.75, 1.000, 38.9, 0.3621, 0.132),
                           (1.00, 0.995, 41.7, 0.3539, 0.122)]:
    check(f"Ch19 T3={lv} survival", T3ROW[lv][0], sv, tol=0.003)
    check(f"Ch19 T3={lv} mean N", T3ROW[lv][3], nn, tol=0.06)
    check(f"Ch19 T3={lv} quality", T3ROW[lv][5], qq, tol=0.0006)
    check(f"Ch19 T3={lv} newcomer share", T3ROW[lv][8], nc, tol=0.0006)
print(f"\nClosing the door entirely costs {T3ROW[1.0][3]-T3ROW[0.0][3]:.1f} members and "
      f"{100*(T3ROW[1.0][0]-T3ROW[0.0][0]):.1f} points of survival,")
print(f"and RAISES measured quality by {T3ROW[0.0][5]-T3ROW[1.0][5]:.4f} while raising the "
      f"newcomer share by {T3ROW[0.0][8]-T3ROW[1.0][8]:.3f}.")
check("Ch19 members lost by closing the door", T3ROW[1.0][3]-T3ROW[0.0][3], 14.2, tol=0.1)
check("Ch19 quality rise from closing the door", T3ROW[0.0][5]-T3ROW[1.0][5], 0.0464, tol=0.001)
check("Ch19 newcomer share rise from closing the door", T3ROW[0.0][8]-T3ROW[1.0][8], 0.079, tol=0.001)
check("Ch19 newcomer share when fully closed, per cent", 100*T3ROW[0.0][8], 20.1, tol=0.06)
check("Ch19 survival when fully closed, per cent", 100*T3ROW[0.0][0], 94.0, tol=0.06)
check("Ch19 survival when fully open, per cent", 100*T3ROW[1.0][0], 99.5, tol=0.06)
# the 0.75 and 1.00 survival rows are NOT distinguishable and must not be read as a peak
_ov = bool(T3ROW[0.75][1] <= T3ROW[1.0][0] <= T3ROW[0.75][2])
print(f"survival at T3=0.75 and T3=1.00 overlap on the Wilson interval: {_ov} "
      f"(do not read a peak at 0.75)")
check("Ch19 0.75 and 1.00 survival indistinguishable", float(_ov), 1.0, tol=0.01)

# ---------- (b) Chapters 20 and 21: the trajectories
LMAX = 31; YRS = [0, 1, 2, 5, 10, 15, 20, 25, 30]
FP = {}
print("\n\nThirty-year trajectories, 400 seeds. 'alive' is more than five members.")
print("N (all) counts a dead group as zero; N (alive) and quality condition on survival.\n")
for sc in _MT['traj']:
    v = _g('traj', sc)
    N = np.zeros((400, LMAX)); Q = np.full((400, LMAX), np.nan); A = np.zeros((400, LMAX), bool)
    for i, x in enumerate(v):
        n = x['yrN'][:LMAX]; q = x['yrQ'][:LMAX]
        N[i, :len(n)] = n; Q[i, :len(q)] = q; A[i, :len(n)] = np.array(n) > 5
    FP[sc] = dict(alive=[float(A[:, y].mean()) for y in YRS],
                  Nall=[float(N[:, y].mean()) for y in YRS],
                  Nal=[float(N[A[:, y], y].mean()) if A[:, y].any() else 0.0 for y in YRS],
                  Q=[float(np.nanmean(Q[A[:, y], y])) if A[:, y].any() else 0.0 for y in YRS])
    print(f"--- {sc} ---")
    print("  year        " + "".join(f"{y:>8}" for y in YRS))
    for lab, key in [("alive frac ", 'alive'), ("N (all)    ", 'Nall'),
                     ("N (alive)  ", 'Nal'), ("quality    ", 'Q')]:
        fmt = "{:>8.3f}" if key in ('alive', 'Q') else "{:>8.1f}"
        print(f"  {lab} " + "".join(fmt.format(x) for x in FP[sc][key]))
    print()

_I = {y: i for i, y in enumerate(YRS)}
for sc, y, key, want in [
    ('full', 30, 'Nall', 41.7), ('full', 30, 'Q', 0.320), ('full', 5, 'Nall', 47.6),
    ('attraction', 30, 'Nall', 13.5), ('attraction', 30, 'Q', 0.263),
    ('attraction', 5, 'Q', 0.359), ('attraction', 10, 'Q', 0.310),
    ('referral', 5, 'alive', 0.995), ('referral', 5, 'Nal', 33.5), ('referral', 5, 'Q', 0.353),
    ('referral', 10, 'alive', 0.907), ('referral', 10, 'Nal', 29.4), ('referral', 10, 'Q', 0.348),
    ('referral', 20, 'alive', 0.603), ('referral', 20, 'Nal', 25.8), ('referral', 20, 'Q', 0.346),
    ('referral', 30, 'alive', 0.360), ('referral', 30, 'Nall', 9.9), ('referral', 30, 'Nal', 26.6),
    ('gatekeeping', 30, 'alive', 0.940), ('gatekeeping', 30, 'Nal', 29.1),
    ('gatekeeping', 10, 'Q', 0.357), ('gatekeeping', 30, 'Q', 0.335)]:
    check(f"Ch20/21 {sc} yr{y} {key}", FP[sc][key][_I[y]], want, tol=0.06 if key in ('Nall','Nal') else 0.0015)

print("THE THREE FINGERPRINTS, at year 10 and at year 30\n")
print(f"{'failure mode':<26}{'alive y10':>10}{'N|alive y10':>13}{'Q y10':>8}{'alive y30':>11}{'Q y30':>8}")
for sc, lab in [('full', 'nothing wrong'), ('attraction', 'invisible (no attraction)'),
                ('referral', 'unreferred'), ('gatekeeping', 'unwelcoming')]:
    f = FP[sc]
    print(f"{lab:<26}{f['alive'][_I[10]]:>10.3f}{f['Nal'][_I[10]]:>13.1f}{f['Q'][_I[10]]:>8.3f}"
          f"{f['alive'][_I[30]]:>11.3f}{f['Q'][_I[30]]:>8.3f}")
_gap = FP['referral']['Q'][_I[10]] - FP['full']['Q'][_I[10]]
print(f"\nAt year 10 an unreferred group's SURVIVORS are at quality {FP['referral']['Q'][_I[10]]:.3f} "
      f"against a healthy {FP['full']['Q'][_I[10]]:.3f}, a difference of {_gap:+.3f},")
print(f"while 9.3 per cent of them are already dead. Nothing in the surviving groups shows it.")
check("Ch21 quality gap at year 10, unreferred vs healthy", _gap, 0.010, tol=0.0015)
_att = FP['attraction']['Q'][_I[10]] - FP['full']['Q'][_I[10]]
check("Ch21 quality gap at year 10, invisible vs healthy", _att, -0.028, tol=0.0015)
print(f"An invisible group's survivors are at {_att:+.3f}, which IS visible.")


# --- Monte Carlo error on every trajectory figure Chapters 20 and 21 print
print("\n\nMonte Carlo error at years 10 and 30: Wilson on survival, 1.96*SE on the rest\n")
print(f"{'condition':<14}{'yr':>4}{'alive':>8}{'95% interval':>18}{'N|alive':>9}{'+/-':>6}{'quality':>9}{'+/-':>8}")
CI5 = {}
for sc in _MT['traj']:
    v = _g('traj', sc)
    N = np.zeros((400, LMAX)); Q = np.full((400, LMAX), np.nan); A = np.zeros((400, LMAX), bool)
    for i, x in enumerate(v):
        n = x['yrN'][:LMAX]; q = x['yrQ'][:LMAX]
        N[i, :len(n)] = n; Q[i, :len(q)] = q; A[i, :len(n)] = np.array(n) > 5
    for y in (10, 30):
        k = int(A[:, y].sum()); lo, hi = wilson(k, 400)
        na = N[A[:, y], y]; qa = Q[A[:, y], y]
        CI5[(sc, y)] = (k/400, lo, hi, na.mean(), 1.96*na.std(ddof=1)/np.sqrt(len(na)),
                        float(np.nanmean(qa)), 1.96*float(np.nanstd(qa, ddof=1))/np.sqrt(len(qa)),
                        1.96*N[:, y].std(ddof=1)/20)
        c = CI5[(sc, y)]
        print(f"{sc:<14}{y:>4}{c[0]:>8.3f}   [{c[1]:.3f},{c[2]:.3f}]{c[3]:>9.1f}{c[4]:>6.1f}{c[5]:>9.3f}{c[6]:>8.4f}")
for sc, y, lo, hi, nh, qh in [
        ('full', 10, 0.990, 1.000, 1.2, 0.0045), ('full', 30, 0.982, 0.999, 1.5, 0.0060),
        ('attraction', 10, 0.986, 1.000, 0.4, 0.0088), ('attraction', 30, 0.986, 1.000, 0.3, 0.0080),
        ('referral', 10, 0.875, 0.932, 1.6, 0.0062), ('referral', 30, 0.314, 0.408, 3.0, 0.0099),
        ('gatekeeping', 10, 0.978, 0.997, 1.4, 0.0059), ('gatekeeping', 30, 0.912, 0.959, 1.6, 0.0084)]:
    c = CI5[(sc, y)]
    check(f"Ch20 {sc} y{y} survival CI lo", c[1], lo, tol=0.0006)
    check(f"Ch20 {sc} y{y} survival CI hi", c[2], hi, tol=0.0006)
    check(f"Ch20 {sc} y{y} N|alive halfwidth", c[4], nh, tol=0.06)
    check(f"Ch20 {sc} y{y} quality halfwidth", c[6], qh, tol=0.0006)
for sc, want in [('full', 1.5), ('attraction', 0.4), ('referral', 1.6), ('gatekeeping', 1.7)]:
    check(f"Ch20 {sc} unconditional N halfwidth y30", CI5[(sc, 30)][7], want, tol=0.06)
_y5 = {}
for sc in _MT['traj']:
    v = _g('traj', sc); N = np.zeros((400, LMAX))
    for i, x in enumerate(v):
        n = x['yrN'][:LMAX]; N[i, :len(n)] = n
    _y5[sc] = 1.96*N[:, 5].std(ddof=1)/20
for sc, want in [('full', 1.0), ('attraction', 0.4), ('referral', 1.2), ('gatekeeping', 1.0)]:
    check(f"Ch20 {sc} unconditional N halfwidth y5", _y5[sc], want, tol=0.06)
_r10 = CI5[('referral', 10)]; _r30 = CI5[('referral', 30)]
check("Ch21 unreferred y10 unconditional halfwidth", _r10[7], 1.7, tol=0.06)
# Chapter 21 quotes the survival fractions as percentages; assert them in that form too.
for _y, _w in [(5, 99.5), (10, 90.7), (15, 73.8), (20, 60.3), (25, 44.0), (30, 36.0)]:
    _vv = _g('traj', 'referral')
    _AA = np.zeros((400, LMAX), bool)
    for _i, _x in enumerate(_vv):
        _n = _x['yrN'][:LMAX]; _AA[_i, :len(_n)] = np.array(_n) > 5
    check(f"Ch21 unreferred survival y{_y}, per cent", 100*float(_AA[:, _y].mean()), _w, tol=0.06)
_ref = _g('traj', 'referral')
_NN = np.zeros((400, LMAX)); _QQ = np.full((400, LMAX), np.nan); _AA = np.zeros((400, LMAX), bool)
for _i, _x in enumerate(_ref):
    _n = _x['yrN'][:LMAX]; _q = _x['yrQ'][:LMAX]
    _NN[_i, :len(_n)] = _n; _QQ[_i, :len(_q)] = _q; _AA[_i, :len(_n)] = np.array(_n) > 5
print("\nUnreferred, survivor series by year (Chapter 21's central table)")
for _y in (2, 5, 10, 15, 20, 25, 30):
    _na = _NN[_AA[:, _y], _y]; _qa = _QQ[_AA[:, _y], _y]
    print(f"   y{_y:<3} alive {_AA[:,_y].mean():.3f}  all {_NN[:,_y].mean():>5.1f}  "
          f"survivors {_na.mean():>5.1f} (n={len(_na)})  Q {np.nanmean(_qa):.3f}")
for _y, _wall, _wsurv, _wq in [(2, 35.8, 35.8, 0.359), (5, 33.4, 33.5, 0.353),
                               (10, 27.0, 29.4, 0.348), (15, 21.2, 28.1, 0.346),
                               (20, 16.0, 25.8, 0.346), (25, 12.4, 27.2, 0.349),
                               (30, 9.9, 26.6, 0.335)]:
    _na = _NN[_AA[:, _y], _y]
    check(f"Ch21 unreferred y{_y} all runs", float(_NN[:, _y].mean()), _wall, tol=0.06)
    check(f"Ch21 unreferred y{_y} survivors", float(_na.mean()), _wsurv, tol=0.06)
    check(f"Ch21 unreferred y{_y} quality", float(np.nanmean(_QQ[_AA[:, _y], _y])), _wq, tol=0.0015)
    check(f"Ch21 unreferred y{_y} survivor count", float(len(_na)), float(int(round(_AA[:,_y].mean()*400))), tol=0.5)
_full = _g('traj', 'full')
_FN = np.zeros((400, LMAX)); _FQ = np.full((400, LMAX), np.nan); _FA = np.zeros((400, LMAX), bool)
for _i, _x in enumerate(_full):
    _n = _x['yrN'][:LMAX]; _q = _x['yrQ'][:LMAX]
    _FN[_i, :len(_n)] = _n; _FQ[_i, :len(_q)] = _q; _FA[_i, :len(_n)] = np.array(_n) > 5
for _y, _w in [(10, 0.010), (20, 0.022), (30, 0.015)]:
    _gapy = float(np.nanmean(_QQ[_AA[:, _y], _y]) - np.nanmean(_FQ[_FA[:, _y], _y]))
    check(f"Ch21 quality gap over healthy at y{_y}", _gapy, _w, tol=0.0015)
check("Ch21 survivor count at y10", float(int(round(_AA[:,10].mean()*400))), 363.0, tol=0.5)
check("Ch21 survivor count at y30", float(int(round(_AA[:,30].mean()*400))), 144.0, tol=0.5)

# ---------- (c) Chapter 22: does composition matter?
print("\n\nComposition experiment: same total founding practice, distributed three ways\n")
print(f"{'condition':<16}{'survival':>10}{'mean N':>9}{'95% halfwidth':>15}{'quality':>9}{'core':>7}")
COMP = {}
for c in _MT['composition']:
    v = _g('comp', c)
    N = np.array([x['N'] for x in v], float); k = sum(x['alive'] for x in v)
    q = np.array([x['est_mean'] for x in v if x['alive']], float)
    ne = np.array([x['n_est'] for x in v if x['alive']], float)
    COMP[c] = (k/400, N.mean(), 1.96*N.std(ddof=1)/20, q.mean(), ne.mean())
    print(f"{c:<16}{k/400:>10.3f}{N.mean():>9.1f}{1.96*N.std(ddof=1)/20:>15.1f}{q.mean():>9.4f}{ne.mean():>7.1f}")
for c, sv, nn, qq in [('even', 0.995, 41.7, 0.3539), ('concentrated', 1.000, 41.3, 0.3534),
                      ('split', 0.998, 39.9, 0.3514)]:
    check(f"Ch22 {c} survival", COMP[c][0], sv, tol=0.003)
    check(f"Ch22 {c} mean N", COMP[c][1], nn, tol=0.06)
    check(f"Ch22 {c} quality", COMP[c][3], qq, tol=0.0006)
_spread = max(COMP[c][1] for c in COMP) - min(COMP[c][1] for c in COMP)
_hw = max(COMP[c][2] for c in COMP)
print(f"\nspread across conditions {_spread:.1f} members against a 95 per cent half-width of {_hw:.1f}")
check("Ch22 spread across composition conditions", _spread, 1.8, tol=0.06)
check("Ch22 largest interval half-width", _hw, 1.6, tol=0.06)
# The seeding vectors themselves, so the chapter's design description is checkable.
_n0 = 25; _tot = 0.55*_n0
_conc_low = (_tot - 5.0)/20; _split_low = (_tot - 12*0.9)/13
print(f"founding total practice {_tot:.2f}; concentrated low tier {_conc_low:.4f}; "
      f"split low tier {_split_low:.4f}")
check("Ch22 founding total practice", _tot, 13.75, tol=0.001)
check("Ch22 concentrated low tier", _conc_low, 0.4375, tol=0.0001)
# Chapter 22 first printed 0.2769 for this. (13.75 - 12*0.9)/13 = 0.2269. The
# assertion caught the transcription error before the chapter was saved.
check("Ch22 split low tier", _split_low, 0.2269, tol=0.0001)
check("Ch22 even survival, half-width", COMP['even'][2], 1.5, tol=0.06)
check("Ch22 concentrated survival, half-width", COMP['concentrated'][2], 1.5, tol=0.06)
check("Ch22 split survival, half-width", COMP['split'][2], 1.6, tol=0.06)
for _c, _w in [('even', 37.2), ('concentrated', 36.7), ('split', 35.4)]:
    check(f"Ch22 {_c} core", COMP[_c][4], _w, tol=0.06)
print("  -> A NULL RESULT, and the model was nearly incapable of producing anything else:")
print("     resources are computed from group aggregates, so composition can only act through")
print("     the Hill gate's non-linearity and through member heterogeneity. Chapter 22 says so.")


  OK  P5 seeds per condition                          400.000  (book: 400.0)
  OK  P5 total runs                                  4800.000  (book: 4800.0)
Tradition 3 sweep, 400 seeds, everything else at full adherence

    T3  survival            95% CI   mean N    +/-  quality     +/-   core  newcomer
  0.00     0.940   [0.912,0.959]     27.5    1.7   0.4003  0.0083   24.1     0.201
  0.25     0.978   [0.958,0.988]     33.7    1.7   0.3878  0.0072   29.3     0.165
  0.50     0.990   [0.975,0.996]     36.6    1.7   0.3700  0.0066   32.0     0.148
  0.75     1.000   [0.990,1.000]     38.9    1.6   0.3621  0.0065   34.2     0.132
  1.00     0.995   [0.982,0.999]     41.7    1.5   0.3539  0.0061   37.2     0.122
  OK  Ch19 T3=0.0 survival                              0.940  (book: 0.94)
  OK  Ch19 T3=0.0 mean N                               27.495  (book: 27.5)
  OK  Ch19 T3=0.0 quality                               0.400  (book: 0.4003)
  OK  Ch19 T3=0.0 newcomer share                  

---
## 15. Part Two sensitivity: the mapping between the theorem and the Traditions
### Chapters 8, 9, 10 and 11

**Part Two contains the book's central claim and carried no sensitivity analysis of any kind
until 2 August 2026.** Its four chapters have no robustness language anywhere and the appendix
did not mention the mapping. The theorem needs no sensitivity, being a theorem. What needed it
was everything between the theorem and the Traditions. `model/part2_influence.py`, cached in
`research/part2_influence.json`. All deterministic linear algebra except the random control.

**Three results, and the second is a correction.**

**(a) The dose-response Chapter 8 never gave.** That chapter illustrates concentration with two
matrices, flat and one member holding 0.35. The full sweep shows the damage starting far
earlier: a member holding 0.05 of every row floors the consensus error at 0.0465 against a flat
0.0252 at N = 1000, a factor of 1.84, and the factor grows without limit in N.

**(b) At fixed magnitude, two of Golub and Jackson's three obstructions do not obstruct.** A
clique of three giving a tenth of its attention outward holds 0.029 of the influence at
N = 1000 and falling. A member receiving twenty times what he gives holds 0.020 and falling.
Both vanish, so both satisfy the condition. Let the same practices scale with the group and
both become obstructions, with max influence flat in N. **So the three obstructions are not
three failure modes at fixed magnitude; they are three ways of describing sequences in which
somebody's share fails to vanish, and whether a practice obstructs depends on how it scales
with the group rather than on its level.** Chapter 8's mapping is stated at fixed magnitude
and is corrected. This is Chapter 10's rotation finding generalised to the whole of Part Two.

**(c) Chapter 11's touring-speaker structure, which that chapter flags as argued rather than
simulated, has an exact closed form.** With speakers giving a fraction `back` of their
attention to the general membership, the speakers' total influence is `share / (share + back)`,
verified to 1e-12, **independent of the size of the movement and of the number of speakers**.
With back = 0 they hold everything trivially, because they are then a closed communicating
class and the theorem does not apply; that case is reported as vacuous and not as a finding.

In [ ]:
# --- Part Two sensitivity: the mapping between the theorem and the Traditions
# Part Two contains the book's central claim and, until 2 August 2026, carried no sensitivity
# analysis of any kind. The theorem needs none. What needed it is everything between the
# theorem and the Traditions. From research/part2_influence.json, model/part2_influence.py.
# All deterministic linear algebra except the random control.
_P2 = _json.load(open(f'{_R}/part2_influence.json'))
_NSs = ['10', '25', '50', '100', '250', '500', '1000']

print("1. IDENTITY CHECK: flat error against the closed form sqrt(2/pi)/sqrt(N)\n")
for _r in _P2['identity']:
    check(f"P2 flat error N={_r['n']}", _r['computed'], _r['closed_form'], tol=1e-9)
print()

print("2. DOMINANCE DOSE-RESPONSE. Chapter 8 gave alpha = 0.35 only.\n")
print("   alpha " + "".join(f"{n:>9}" for n in _NSs) + "   <- max influence")
for _r in _P2['dominance']:
    print(f"   {_r['alpha']:<6}" + "".join(f"{_r[n][0]:>9.4f}" for n in _NSs))
print("   alpha " + "".join(f"{n:>9}" for n in _NSs) + "   <- consensus error")
for _r in _P2['dominance']:
    print(f"   {_r['alpha']:<6}" + "".join(f"{_r[n][1]:>9.4f}" for n in _NSs))
_D = {r['alpha']: r for r in _P2['dominance']}
for _a, _w in [(0.0, 0.0252), (0.02, 0.0294), (0.05, 0.0465), (0.10, 0.0830),
               (0.20, 0.1608), (0.35, 0.2797), (0.50, 0.3991), (0.75, 0.5984)]:
    check(f"P2 error at alpha={_a}, N=1000", _D[_a]['1000'][1], _w, tol=0.0006)
_R2P = np.sqrt(2/np.pi)
print("\n   Error converges to alpha*sqrt(2/pi). Ratio of computed to limit at N=1000:")
for _a in (0.05, 0.10, 0.20, 0.35):
    print(f"      alpha={_a:<5} {_D[_a]['1000'][1]/(_a*_R2P):.4f}")
check("P2 alpha=0.35 ratio to limit at N=1000", _D[0.35]['1000'][1]/(0.35*_R2P), 1.0017, tol=0.0006)
check("P2 alpha=0.05 ratio to limit at N=1000", _D[0.05]['1000'][1]/(0.05*_R2P), 1.1668, tol=0.0006)
print("   -> Even alpha = 0.05 floors the error at 0.0465 against a flat 0.0252 at N=1000,")
print("      a factor of 1.84, and the factor grows without limit in N. Chapter 8's single")
print("      illustration at 0.35 understates how little concentration is needed.")
check("P2 damage factor at alpha=0.05, N=1000", _D[0.05]['1000'][1]/_D[0.0]['1000'][1], 1.844, tol=0.002)

print("\n\n3. THE THREE OBSTRUCTIONS AT FIXED MAGNITUDE. Do they actually obstruct?\n")
print("   clique, total influence held by the clique")
for _r in _P2['clique']:
    print(f"      k={_r['k']:<3} inward={_r['inward']:<5}" +
          "".join(f"{_r[n][2]:>9.4f}" for n in _NSs if n in _r))
print("   imbalance, total influence held by the favoured members")
for _r in _P2['imbalance']:
    print(f"      k={_r['k']:<3} ratio={_r['ratio']:<5}" +
          "".join(f"{_r[n][2]:>9.4f}" for n in _NSs if n in _r))
_C = {(r['k'], r['inward']): r for r in _P2['clique']}
_I = {(r['k'], r['ratio']): r for r in _P2['imbalance']}
check("P2 clique k=3 inward=0.9 at N=1000", _C[(3, 0.9)]['1000'][2], 0.0291, tol=0.0006)
check("P2 clique k=3 inward=0.99 at N=1000", _C[(3, 0.99)]['1000'][2], 0.2308, tol=0.0006)
check("P2 imbalance k=1 ratio=20 at N=1000", _I[(1, 20)]['1000'][2], 0.0196, tol=0.0006)
check("P2 imbalance k=5 ratio=20 at N=1000", _I[(5, 20)]['1000'][2], 0.0913, tol=0.0006)
print("\n   -> AT FIXED MAGNITUDE, NEITHER IS AN OBSTRUCTION. A clique of three giving a tenth")
print("      of its attention outward holds 0.0291 of the influence at N=1000 and falling. A")
print("      member receiving twenty times what he gives holds 0.0196 and falling. Both")
print("      VANISH, so both satisfy the Golub-Jackson condition.")

print("\n4. THE SAME TWO PRACTICES, SCALED WITH THE GROUP\n")
for _r in _P2['scaling']:
    print(f"   {_r['kind']:<20}" + "".join(f"{_r[n][0]:>9.4f}" for n in _NSs) + "  max influence")
    print(f"   {'   error':<20}" + "".join(f"{_r[n][1]:>9.4f}" for n in _NSs))
_S = {r['kind']: r for r in _P2['scaling']}
for _n in _NSs:
    check(f"P2 clique_scaling max influence N={_n}", _S['clique_scaling'][_n][0], 0.1667, tol=0.0006)
check("P2 imbalance_scaling max influence N=10", _S['imbalance_scaling']['10'][0], 0.1818, tol=0.0006)
check("P2 imbalance_scaling max influence N=1000", _S['imbalance_scaling']['1000'][0], 0.1668, tol=0.0006)
print("\n   -> SCALED, BOTH ARE OBSTRUCTIONS. Max influence is flat in N rather than falling.")
print("      THE CORRECTION THIS PRODUCES: the three obstructions are not three failure modes")
print("      at fixed magnitude. They are three ways of describing sequences in which somebody's")
print("      share fails to vanish, and whether a practice obstructs depends on HOW IT SCALES")
print("      WITH THE GROUP, not on its level. That is exactly Chapter 10's rotation finding")
print("      generalised: a fixed pool in a growing group is an obstruction for the same reason.")

print("\n\n5. CHAPTER 11'S TOURING-SPEAKER STRUCTURE, WHICH THAT CHAPTER FLAGS AS UNSIMULATED\n")
print("   speakers' total influence; 'back' is attention flowing from speakers to members")
print("   k   share back " + "".join(f"{n:>9}" for n in _NSs))
for _r in _P2['speakers']:
    print(f"   {_r['speakers']:<4}{_r['share']:<6}{_r['back']:<5}" +
          "".join(f"{_r[n][2]:>9.4f}" for n in _NSs))
_SP = {(r['speakers'], r['share'], r['back']): r for r in _P2['speakers']}
check("P2 speakers, back=0, N=1000", _SP[(5, 0.3, 0.0)]['1000'][2], 1.0, tol=1e-9)
print("\n   With back = 0 the speakers are a closed communicating class, the chain is not")
print("   strongly connected, the theorem does not apply, and they hold ALL the influence at")
print("   every n and every share. That is vacuous and is not reported as a finding.")
print("\n   With back > 0 the structure has an exact closed form, verified to 1e-12:")
print("\n      speakers' total influence = share / (share + back)\n")
print("   independent of n AND of the number of speakers.")
_worst = 0.0
for _r in _P2['speakers']:
    if _r['back'] == 0: continue
    _pred = _r['share'] / (_r['share'] + _r['back'])
    for _n in _NSs:
        _worst = max(_worst, abs(_r[_n][2] - _pred))
    check(f"P2 speakers k={_r['speakers']} share={_r['share']} back={_r['back']}",
          _r['1000'][2], _pred, tol=1e-6)
print(f"   largest deviation from the closed form across all rows and all n: {_worst:.2e}")
check("P2 speakers closed form, largest deviation", float(_worst < 1e-9), 1.0, tol=0.01)
print("\n   -> THIS IS CHAPTER 11'S CLAIM MADE EXACT. The touring-speaker structure gives the")
print("      speakers a share of influence that does not vanish as the movement grows, and the")
print("      share depends only on the ratio of attention flowing out to attention flowing back.")
print("      At share 0.3 and back 0.2, five speakers hold 0.600 of a movement of any size.")

print("\n\n6. RANDOM CONTROL: is max influence a fair proxy for the l2 norm the theorem uses?\n")
print(f"   {'N':>5}{'conc':>7}{'max':>9}{'err':>9}{'corr':>8}")
for _r in _P2['random_control']:
    print(f"   {_r['n']:>5}{_r['conc']:>7}{_r['max_mean']:>9.4f}{_r['err_mean']:>9.4f}{_r['corr_max_err']:>8.3f}")
_rc = {(r['n'], r['conc']): r for r in _P2['random_control']}
check("P2 control N=25 conc=0.05 corr", _rc[(25, 0.05)]['corr_max_err'], 0.970, tol=0.002)
check("P2 control N=250 conc=50 corr", _rc[(250, 50.0)]['corr_max_err'], 0.351, tol=0.002)
print("\n   -> The correlation between max influence and consensus error is strong where")
print("      concentration is high and weak where it is not. The book uses max influence as a")
print("      proxy for concentration throughout and this says the proxy is good in the regime")
print("      the book cares about and poor in the regime where nothing is wrong anyway.")


1. IDENTITY CHECK: flat error against the closed form sqrt(2/pi)/sqrt(N)

  OK  P2 flat error N=10                                0.252  (book: 0.252313252202016)
  OK  P2 flat error N=25                                0.160  (book: 0.15957691216057307)
  OK  P2 flat error N=50                                0.113  (book: 0.11283791670955126)
  OK  P2 flat error N=100                               0.080  (book: 0.07978845608028654)
  OK  P2 flat error N=250                               0.050  (book: 0.050462650440403205)
  OK  P2 flat error N=500                               0.036  (book: 0.035682482323055424)
  OK  P2 flat error N=1000                              0.025  (book: 0.025231325220201602)

2. DOMINANCE DOSE-RESPONSE. Chapter 8 gave alpha = 0.35 only.

   alpha        10       25       50      100      250      500     1000   <- max influence
   0.0      0.1000   0.0400   0.0200   0.0100   0.0040   0.0020   0.0010
   0.02     0.1089   0.0408   0.0200   0.0200   0.0200   0.

---
## 16. Structural sensitivity: perturbing the model's choices, not its numbers
### Appendix A9; and it splits the book's strongest simulation claim

**Appendix A5.6 opened by saying that structural choices were not perturbed.** Every design in
A4 and A5 varies the 118 hand-chosen numbers. None varied the shape. That distinction matters
because the book's surviving claims are orderings, and an ordering can be robust to every number
in a model and still be an artefact of its architecture.

Four variants, each one structural change with everything else at nominal, run against the same
five scenarios at 400 seeds. 10,000 runs. `model/structural_variants.py`.

**The finding, and it is a correction.** "Losing referrals is worse than losing attraction" is
described in the preface, in Chapters 1 and 4, in `README.md`, in `CLAUDE.md` and in
`research/PARAMETERS.md` as the most robust thing the simulation says, surviving all 236 targeted
perturbations and all global jitter. **Nowhere is "worse" defined.** Against architecture the two
readings come apart:

- **On survival it holds in all five variants**, and by a wide margin: 0.360 against 0.998 in the
  base model, 0.490 against 0.825 with a flat capacity gate, 0.635 against 1.000 with capacity
  from all members, 0.573 against 0.998 without saturation.
- **On mean membership it reverses in three of the four variants.** With a flat gate the
  referral-starved group ends at 15.6 members and the attraction-starved one at 8.4. The core-size
  ordering reverses in the same three.

So the claim is architecture-robust when it means *more likely to die* and architecture-dependent
when it means *ends up smaller*. Everything above has been amended to say survival.

**One variant is degenerate and is reported rather than hidden.** Reading Tradition 3 as governing
arrival makes adherence a multiplier on inflow, so zero adherence is a group that admits nobody,
and it dies in 400 runs of 400. That is a shut group, not a cliquish one. The variant supports
Chapter 19's retention reading rather than testing it.

In [ ]:
# --- Structural sensitivity: perturbing the model's CHOICES rather than its numbers
# Appendix A5.6 opened by saying structural choices were never perturbed. They are now.
# model/structural_variants.py, cached in research/structural.json. 10,000 runs: five
# variants x five scenarios x 400 seeds.
_SV = _json.load(open(f'{_R}/structural.json')); _SM = _SV['meta']
def _sg(v, s): return [_SV[f'{v}|{s}|{i}'] for i in range(_SM['nseed'])]
check("A9 variants", float(len(_SM['variants'])), 5.0, tol=0.01)
check("A9 total runs", float(_SM['jobs']), 10000.0, tol=0.01)

print("Five variants x five scenarios x 400 seeds. Survival with a Wilson interval;")
print("mean N counts a dead group as zero and carries a 95 per cent half-width.\n")
print(f"{'variant':<15}{'scenario':<13}{'surv':>7}{'95% interval':>18}{'meanN':>8}{'+/-':>6}{'quality':>9}")
SVR = {}
for _v in _SM['variants']:
    for _s in _SM['scenarios']:
        _rr = _sg(_v, _s); _N = np.array([x['N'] for x in _rr], float)
        _k = sum(x['alive'] for x in _rr)
        _q = np.array([x['est_mean'] for x in _rr if x['alive']], float)
        _c = np.array([x['n_est'] for x in _rr], float)
        _lo, _hi = wilson(_k, 400)
        SVR[(_v, _s)] = (_k/400, _lo, _hi, _N.mean(), 1.96*_N.std(ddof=1)/20,
                         float(_q.mean()) if len(_q) else 0.0, float(_c.mean()))
        print(f"{_v:<15}{_s:<13}{_k/400:>7.3f}   [{_lo:.3f},{_hi:.3f}]{_N.mean():>8.1f}"
              f"{1.96*_N.std(ddof=1)/20:>6.1f}{(_q.mean() if len(_q) else 0):>9.4f}")
    print()

for _v, _s, _sv, _n in [
    ('base', 'full', 0.995, 41.7), ('base', 'attraction', 0.998, 13.5),
    ('base', 'referral', 0.360, 9.9), ('base', 'gatekeeping', 0.940, 27.5),
    ('gate_flat', 'full', 0.998, 46.8), ('gate_flat', 'attraction', 0.825, 8.4),
    ('gate_flat', 'referral', 0.490, 15.6), ('gate_flat', 'gatekeeping', 0.630, 14.6),
    ('t3_admission', 'gatekeeping', 0.000, 0.0),
    ('capacity_all', 'full', 1.000, 51.5), ('capacity_all', 'attraction', 1.000, 14.7),
    ('capacity_all', 'referral', 0.635, 20.3),
    ('no_saturation', 'full', 1.000, 51.2), ('no_saturation', 'attraction', 0.998, 14.0),
    ('no_saturation', 'referral', 0.573, 21.7)]:
    check(f"A9 {_v} {_s} survival", SVR[(_v, _s)][0], _sv, tol=0.003)
    check(f"A9 {_v} {_s} mean N", SVR[(_v, _s)][3], _n, tol=0.06)

print("THE BOOK'S TWO SURVIVING SIMULATION CLAIMS, TESTED AGAINST ARCHITECTURE\n")
print(f"{'variant':<15}{'full persists':>14}{'referral worse ON SURVIVAL':>28}{'ON SIZE':>10}{'ON CORE':>10}")
_persist = _surv_ok = _size_ok = _core_ok = 0
for _v in _SM['variants']:
    _p = SVR[(_v, 'full')][0] >= 0.90
    _so = SVR[(_v, 'referral')][0] < SVR[(_v, 'attraction')][0]
    _no = SVR[(_v, 'referral')][3] < SVR[(_v, 'attraction')][3]
    _co = SVR[(_v, 'referral')][6] < SVR[(_v, 'attraction')][6]
    _persist += _p; _surv_ok += _so; _size_ok += _no; _core_ok += _co
    print(f"{_v:<15}{str(_p):>14}{str(_so):>28}{str(_no):>10}{str(_co):>10}")
print(f"\n   holds in: persistence {_persist}/5, survival ordering {_surv_ok}/5, "
      f"size ordering {_size_ok}/5, core ordering {_core_ok}/5")
check("A9 full persists in all variants", float(_persist), 5.0, tol=0.01)
check("A9 survival ordering holds in all variants", float(_surv_ok), 5.0, tol=0.01)
check("A9 size ordering holds in", float(_size_ok), 2.0, tol=0.01)
check("A9 core ordering holds in", float(_core_ok), 2.0, tol=0.01)
print("""
   -> THE CLAIM SPLITS, AND THE BOOK HAS NOT BEEN SAYING WHICH HALF IT MEANS.
      'Losing referrals is worse than losing attraction' survives all 236 targeted parameter
      perturbations and all global jitter. Against ARCHITECTURE it survives only when 'worse'
      means MORE LIKELY TO DIE. Read as SMALLER it reverses under three of the four structural
      variants: with a flat capacity gate the referral-starved group ends at 15.6 members
      against the attraction-starved group's 8.4, and similarly under capacity_all and
      no_saturation. The survival gap is large and one-directional in every variant, from
      0.360 against 0.998 in the base model to 0.490 against 0.825 under gate_flat.
      The book now states the claim on survival and says the size ordering is architecture-
      dependent.""")

# --- the degenerate variant, reported rather than hidden
print("""
   THE TRADITION 3 VARIANT IS DEGENERATE AND THAT IS ITSELF INFORMATIVE.
   Reading Tradition 3 as governing ARRIVAL makes adherence a multiplier on inflow, so
   T3 = 0 means a group that admits nobody at all, and it dies in 400 runs out of 400. That
   is not a cliquish group; it is a shut one. Under the RETENTION reading of Chapter 19 the
   same T3 = 0 gives 94.0 per cent survival at 27.5 members. The variant therefore supports
   the modelling choice rather than testing it, and the three non-gatekeeping scenarios are
   identical to base by construction because T3 = 1 there.""")
check("A9 t3_admission gatekeeping survival", SVR[('t3_admission', 'gatekeeping')][0], 0.0, tol=1e-9)
check("A9 t3_admission full identical to base", SVR[('t3_admission', 'full')][3],
      SVR[('base', 'full')][3], tol=1e-9)


  OK  A9 variants                                       5.000  (book: 5.0)
  OK  A9 total runs                                  10000.000  (book: 10000.0)
Five variants x five scenarios x 400 seeds. Survival with a Wilson interval;
mean N counts a dead group as zero and carries a 95 per cent half-width.

variant        scenario        surv      95% interval   meanN   +/-  quality
base           full           0.995   [0.982,0.999]    41.7   1.5   0.3539
base           attraction     0.998   [0.986,1.000]    13.5   0.4   0.3116
base           referral       0.360   [0.314,0.408]     9.9   1.6   0.3718
base           both           0.000   [0.000,0.010]     0.0   0.0   0.0000
base           gatekeeping    0.940   [0.912,0.959]    27.5   1.7   0.4003

gate_flat      full           0.998   [0.986,1.000]    46.8   1.4   0.4217
gate_flat      attraction     0.825   [0.785,0.859]     8.4   0.4   0.3955
gate_flat      referral       0.490   [0.441,0.539]    15.6   1.9   0.4469
gate_flat      b

---
## 17. The fourth structural choice: must the resource list have eight entries?
### Appendix A9.5

A5.6 named four structural choices that had never been perturbed. A9 perturbed three. This is
the fourth, and it had been left on the ground that changing the number of resources changes
both matrices' column count and is therefore a different model. For the coupling that is not
so: B = S G' is exact algebra on two matrices whose columns are resources, and dropping or
merging a column is well defined on both at once.

64 variants: 8 single deletions, 28 pairwise merges, 28 double deletions. All deterministic.

**Unity's primacy is nearly insensitive to the resource list** — 8/8, 28/28 and 27/28. The one
failure drops continuity and pressure together, which Chapter 17's reassignment test had
already identified as the only two resources able to move it.

**Index-pairing is more sensitive and the sensitivity is in one place.** Across all 64 variants
only two of the twelve Steps ever regain their index-mate: Step 1 in fifteen and Step 2 in one.
Five of the seven non-trivial counts never break. Chapter 16 already calls Step 1 the near miss
and the only one; this measures that caveat.

**The two-tier split is unchanged in all 64 and that is vacuous**, for the same reason the
multiplicative designs in A5.4 cannot reach it: the protective rows are zero across every
resource, so no deletion or merge can make them non-zero.

**What no design here does.** Splitting a resource or inventing a ninth needs fresh judgement
about what the new column contains. The list can be shown to be no *finer* than it needs to be;
it cannot be shown to be *fine enough*.

In [ ]:
# --- The fourth structural choice: must the resource list have eight entries?
# Appendix A5.6 named four structural choices never perturbed. A9 did three. This is the
# fourth, left because changing the resource count changes both matrices' column count and
# looked like a different model. For the coupling it is not: B = S G' is exact algebra on two
# matrices whose columns are resources, and dropping or merging a column is well defined on
# both at once. model/resource_list_test.py, research/resource_list.json.
_RL = _json.load(open(f'{_R}/resource_list.json'))
print("Resources:", ", ".join(_RL['resources']), "\n")
_b = _RL['base']
print(f"base: index-pairing fails on all twelve = {_b['all12']}; T1 top = {_b['t1']}; "
      f"T1 load {_b['load_t1']:.2f}; margin over second {_b['margin']:.2f}\n")
check("RL base all twelve fail", float(_b['all12']), 1.0, tol=0.01)
check("RL base T1 top", float(_b['t1']), 1.0, tol=0.01)
check("RL base T1 load", _b['load_t1'], 6.52, tol=0.006)

print(f"{'design':<16}{'n':>4}{'all 12':>9}{'T1 top':>9}{'S5->T12':>9}{'S12->T5':>9}{'prot':>7}")
for _k, _lab in [('leave_one_out', 'drop one'), ('merge', 'merge a pair'), ('drop_two', 'drop two')]:
    _s = _RL[_k + '_summary']
    print(f"{_lab:<16}{_s['n']:>4}{_s['all12']:>7}/{_s['n']:<2}{_s['t1']:>7}/{_s['n']:<2}"
          f"{_s['s5']:>7}/{_s['n']:<2}{_s['s12']:>7}/{_s['n']:<2}"
          f"{_s['protective_unchanged']:>5}/{_s['n']:<2}")
_S1 = {k: _RL[k + '_summary'] for k in ('leave_one_out', 'merge', 'drop_two')}
for _k, _n, _a, _t, _p in [('leave_one_out', 8, 7, 8, 8), ('merge', 28, 20, 28, 28),
                           ('drop_two', 28, 21, 27, 28)]:
    check(f"RL {_k} variants", float(_S1[_k]['n']), float(_n), tol=0.01)
    check(f"RL {_k} all-twelve holds in", float(_S1[_k]['all12']), float(_a), tol=0.01)
    check(f"RL {_k} T1 top holds in", float(_S1[_k]['t1']), float(_t), tol=0.01)
    check(f"RL {_k} protective unchanged in", float(_S1[_k]['protective_unchanged']), float(_p), tol=0.01)

print("\nSteps that EVER regain their index-mate, across all 64 variants:")
_all = {}
for _k in ('leave_one_out', 'merge', 'drop_two'):
    for _st, _c in _S1[_k]['steps_regaining'].items():
        _all[int(_st)] = _all.get(int(_st), 0) + _c
for _st in sorted(_all): print(f"   Step {_st}: {_all[_st]} of 64 variants")
check("RL Step 1 regains in N variants", float(_all.get(1, 0)), 15.0, tol=0.01)
check("RL Step 2 regains in N variants", float(_all.get(2, 0)), 1.0, tol=0.01)
check("RL number of Steps ever regaining", float(len(_all)), 2.0, tol=0.01)
_nontriv = {1, 2, 3, 5, 8, 11, 12}
check("RL non-trivial Steps never regaining", float(len(_nontriv - set(_all))), 5.0, tol=0.01)
print(f"""
  -> WHAT THIS SETTLES. Unity's primacy is almost entirely insensitive to the resource list:
     it holds in 8 of 8 single deletions, 28 of 28 merges and 27 of 28 double deletions. The
     one failure is dropping continuity AND pressure together, which Chapter 17's reassignment
     test already identified as the only two resources that can move it.

     Index-pairing is more sensitive, and the sensitivity is concentrated in exactly one place.
     Across all 64 variants only TWO of the twelve Steps ever regain their index-mate: Step 1
     in 15 of them and Step 2 in one. Five of the seven non-trivial counts never break under
     any deletion or merge. Chapter 16 already calls Step 1 "the near miss and the only one"
     and says it would not want to rest on it; this is that caveat measured.

  -> WHAT THIS CANNOT TOUCH. The two-tier split is unchanged in all 64 variants and that is
     VACUOUS, not a result: the protective rows of G are zero across every resource, so no
     deletion or merge can make them non-zero. Same vacuity as A5.4. The threshold test in
     A5.4b remains the only design in the project that can reach the split.

  -> AND WHAT NO DESIGN HERE DOES. Splitting a resource in two, or inventing a ninth, needs
     fresh judgement about what the new columns contain and cannot be done by rearranging the
     existing ones. The resource list can be shown to be no FINER than it needs to be. It
     cannot be shown to be fine enough.""")


Resources: admission, identify, proof, confidential, counsel, recipient, continuity, pressure 

base: index-pairing fails on all twelve = True; T1 top = True; T1 load 6.52; margin over second 2.63

  OK  RL base all twelve fail                           1.000  (book: 1.0)
  OK  RL base T1 top                                    1.000  (book: 1.0)
  OK  RL base T1 load                                   6.520  (book: 6.52)
design             n   all 12   T1 top  S5->T12  S12->T5   prot
drop one           8      7/8       8/8       7/8       7/8     8/8 
merge a pair      28     20/28     28/28     26/28     23/28   28/28
drop two          28     21/28     27/28     21/28     19/28   28/28
  OK  RL leave_one_out variants                         8.000  (book: 8.0)
  OK  RL leave_one_out all-twelve holds in              7.000  (book: 7.0)
  OK  RL leave_one_out T1 top holds in                  8.000  (book: 8.0)
  OK  RL leave_one_out protective unchanged in          8.000  (book: 8.0)
  OK 

---
## 18. Sparsity perturbation: how much may a second reader differ?
### Appendix A5.4e

Every perturbation design in A5.4 varies the **magnitudes** of the two matrices and holds the
**sparsity pattern** fixed. A second reader disagrees about the pattern. This flips k cells of
the governance matrix, confined to the seven enabling rows so the two-tier split is held fixed,
and asks how often each Part Four claim survives.

| Cells flipped | Index-pairing fails on all twelve | T1 has the largest load |
|---|---|---|
| 1 | 96.3% | 100.0% |
| 2 | 92.7% | 99.7% |
| 4 | 86.2% | 96.5% |
| 8 | 75.0% | 83.7% |
| 16 | 53.0% | 50.8% |

**Part Four tolerates a reader who differs on about four of the fifty-six enabling cells and
does not tolerate one who differs on sixteen.** That is the quantity the elicitation form is
designed to measure, and until forms come back this is a bound rather than a measurement.

**Why only a bound.** A random flip is not a plausible reader: somebody who thinks self-support
governs continuity changes that cell for a reason and their other cells correlate with it.
Random flips are harsher, respecting no reason, and gentler, not concentrating on the cells that
matter. And they hold the split fixed; a reader who fills a protective row is making the more
serious disagreement, for which the elicitation is the only instrument.

In [ ]:
# --- Sparsity perturbation: how much would a second reader have to differ before Part Four moves?
# Every design in A5.4 varies the MAGNITUDES and holds the sparsity pattern fixed. A second
# reader disagrees about the PATTERN. This flips k cells of the governance matrix, confined to
# the seven enabling rows so the two-tier split is held fixed, and asks how often each Part Four
# claim survives. model/elicitation_compare.py --sparsity.
import importlib.util as _il, sys as _sys
_spec = _il.spec_from_file_location('ec', 'elicitation_compare.py')
_ec = _il.module_from_spec(_spec); _sys.modules['ec'] = _ec; _spec.loader.exec_module(_ec)
_PROT = _ec.PROTECTIVE
_live = [(j, r) for j in range(12) for r in range(8) if j not in _PROT]
check("SP enabling cells available to flip", float(len(_live)), 56.0, tol=0.01)
_rng = np.random.default_rng(20260802)
_ND = 2000
print(f"{'flips':>6}{'all 12':>10}{'T1 top':>10}{'S5->T12':>10}{'S12->T5':>10}")
_SP = {}
for _k in (1, 2, 3, 4, 6, 8, 12, 16):
    _c = dict(all12=0, t1=0, s5=0, s12=0)
    for _ in range(_ND):
        _G = m.GOV.copy()
        for _i in _rng.choice(len(_live), _k, replace=False):
            _j, _r = _live[_i]
            _G[_j, _r] = 0.0 if _G[_j, _r] > 0 else 0.5
        _q = _ec.consequences(m.S, _G)
        _c['all12'] += _q['all12']; _c['t1'] += _q['t1_top']
        _c['s5'] += _q['s5_to_t12']; _c['s12'] += _q['s12_to_t5']
    _SP[_k] = {a: 100*b/_ND for a, b in _c.items()}
    print(f"{_k:>6}{_SP[_k]['all12']:>9.1f}%{_SP[_k]['t1']:>9.1f}%"
          f"{_SP[_k]['s5']:>9.1f}%{_SP[_k]['s12']:>9.1f}%")
for _k, _a, _t in [(1, 96.3, 100.0), (2, 92.7, 99.7), (4, 86.2, 96.5),
                   (8, 75.0, 83.7), (16, 53.0, 50.8)]:
    check(f"SP all-twelve at {_k} flips", _SP[_k]['all12'], _a, tol=0.06)
    check(f"SP T1-top at {_k} flips", _SP[_k]['t1'], _t, tol=0.06)
print("""
  -> A CONCRETE ANSWER TO A QUESTION THE BOOK COULD NOT PREVIOUSLY ANSWER. A second reader who
     differs from me on FOUR of the fifty-six enabling cells, about seven per cent of them,
     leaves index-pairing failing on all twelve in 86 per cent of draws and unity leading in 97.
     A reader who differs on SIXTEEN, about a third, leaves both at a coin flip.

     So Part Four tolerates a reader who differs on a handful of cells and does not tolerate one
     who differs on a third of them. That is the quantity the elicitation in
     research/GOVERNANCE-MATRIX-ELICITATION.md is designed to measure, and until three forms
     come back this sweep is a bound rather than a measurement.

  -> WHY IT IS ONLY A BOUND. A random flip is not a plausible reader. Somebody who thinks
     self-support governs continuity changes that cell for a reason, and their other cells are
     correlated with the reason. Random flips are harsher than that, because they respect no
     reason, and gentler, because they will not concentrate on the cells that matter. This says
     how much disagreement counted in cells; it says nothing about which cells a reader picks.

  -> AND IT HOLDS THE SPLIT FIXED. Flips are confined to the seven enabling rows. A reader who
     fills a protective row is making the more serious disagreement, and the only instrument for
     that is the elicitation itself.""")


  OK  SP enabling cells available to flip              56.000  (book: 56.0)
 flips    all 12    T1 top   S5->T12   S12->T5
     1     96.3%    100.0%     98.7%     94.8%
     2     92.7%     99.7%     96.3%     90.5%
     3     89.2%     97.8%     93.9%     85.8%
     4     86.2%     96.5%     93.3%     80.3%
     6     81.0%     90.0%     89.0%     78.2%
     8     75.0%     83.7%     85.4%     72.0%
    12     64.1%     67.3%     77.8%     62.5%
    16     53.0%     50.8%     70.8%     53.0%
  OK  SP all-twelve at 1 flips                         96.350  (book: 96.3)
  OK  SP T1-top at 1 flips                            100.000  (book: 100.0)
  OK  SP all-twelve at 2 flips                         92.650  (book: 92.7)
  OK  SP T1-top at 2 flips                             99.650  (book: 99.7)
  OK  SP all-twelve at 4 flips                         86.250  (book: 86.2)
  OK  SP T1-top at 4 flips                             96.550  (book: 96.5)
  OK  SP all-twelve at 8 flips              